# Python lab: Asset Returns — Empirical Stylized Facts

ใช้ Python standard library กด Run All ตามลำดับ ข้อมูลราคาปิด S&P 500 จริง 5,031 วันฝังอยู่ในไฟล์ ใช้คำนวณผลตอบแทน 5,030 ค่า ส่วนตัวอย่าง mixture, intraday และ known-sigma standardization เป็นข้อมูลสมมติ ภาพประกอบฝังไว้แล้วและรันได้โดยไม่ดาวน์โหลดข้อมูลเพิ่ม

# Asset Returns — Empirical Stylized Facts

ผลตอบแทนวันนี้ช่วยบอกความผันผวนวันพรุ่งนี้ได้แค่ไหน?

> “การเปลี่ยนแปลงขนาดใหญ่มักตามด้วยการเปลี่ยนแปลงขนาดใหญ่ ไม่ว่าจะขึ้นหรือลง ส่วนการเปลี่ยนแปลงขนาดเล็กมักตามด้วยการเปลี่ยนแปลงขนาดเล็ก”
>
> <span lang="en">“…large changes tend to be followed by large changes—of either sign—and small changes tend to be followed by small changes…”</span>
>
> — **Benoit Mandelbrot** · [*The Variation of Certain Speculative Prices* (1963), น. 418](https://oftp.cyrax.hu/doc/mandelbrot.pdf#page=26) · ข้อความบางส่วน แปลไทยเพื่อประกอบบทเรียน

ในบท [VaR และ Expected Shortfall](../value-at-risk-expected-shortfall.html) เราเริ่มคำนวณจากการแจกแจงที่กำหนดให้ การใช้ Normal กับ volatility คงที่ทำให้คำนวณสะดวก แต่ในข้อมูลตลาด วันที่ราคาแกว่งแรงมักเกิดติดกัน ค่า volatility ค่าเดียวจึงอาจอธิบายทั้งช่วงสงบและช่วงผันผวนได้ไม่ดี

**Stylized facts** คือรูปแบบเชิงสถิติที่พบซ้ำในข้อมูลหลายตลาดและหลายช่วงเวลา เช่น หางของการแจกแจงที่หนากว่า Normal หรือความสัมพันธ์ของขนาดผลตอบแทนระหว่างวัน เราใช้ข้อสังเกตเหล่านี้ตรวจและเลือกแบบจำลอง โดยต้องดูด้วยว่าพบในสินทรัพย์ ช่วงเวลา และความถี่ใด

เนื้อหาอ้างอิงงานของ Stephen Taylor เรื่อง *Asset Price Dynamics, Volatility, and Prediction* เริ่มจากผลตอบแทนรายวัน แล้วดูว่าราคาระหว่างวันให้ข้อมูลอะไรเพิ่ม กราฟและตัวทดลอง volatility clustering ใช้ราคาปิดดัชนี S&P 500 จริงช่วง 1999–2018 ส่วนตัวอย่าง mixture และข้อมูลระหว่างวันใช้ข้อมูลสมมติเพื่อแยกผลของแต่ละสมมติฐาน

อ่านพื้นฐานเพิ่มเติมได้ที่ [Prices and Returns](../prices-and-returns.html) และ [Stochastic Processes](../stochastic-processes.html) ส่วน [ตารางเทียบหัวข้อ](#coverage-map) ระบุที่อยู่ของเนื้อหาครบบท 2–4 ตามสารบัญ

In [1]:
"""Standard-library numerical examples; all observations are simulated."""
import math
import statistics


def uniform(seed):
    state = seed & 0xffffffff
    while True:
        state = (1664525*state + 1013904223) & 0xffffffff
        yield (state+.5)/4294967296


def normal_generator(seed):
    u = uniform(seed)
    while True:
        yield math.sqrt(-2*math.log(next(u)))*math.cos(2*math.pi*next(u))


def return_pair(previous, current, dividend=0):
    assert previous > 0 and current+dividend > 0
    simple = (current+dividend)/previous-1
    return simple, math.log1p(simple)


def moments(values):
    average = statistics.mean(values)
    centered = [x-average for x in values]
    m2 = statistics.mean(x*x for x in centered)
    m4 = statistics.mean(x**4 for x in centered)
    return {'mean': average, 'sd': statistics.stdev(values), 'variance': m2,
            'kurtosis': m4/m2**2 if m2 > 0 else None}


def acf(values, max_lag=20):
    assert 0 <= max_lag < len(values)
    average = statistics.mean(values)
    x = [v-average for v in values]
    denominator = sum(v*v for v in x)
    if denominator == 0:
        return [None]*(max_lag+1)
    return [sum(x[i]*x[i-lag] for i in range(lag,len(x)))/denominator for lag in range(max_lag+1)]


def portmanteau(values, lags=20):
    rho = acf(values,lags)
    if rho[0] is None:
        return None, None
    n = len(values)
    return n*sum(r*r for r in rho[1:]), n*(n+2)*sum(rho[k]**2/(n-k) for k in range(1,lags+1))


def shuffle(values, seed=731):
    out = list(values)
    random = uniform(seed)
    for i in range(len(out)-1,0,-1):
        j = math.floor(next(random)*(i+1))
        out[i],out[j] = out[j],out[i]
    return out


def clustered_returns(seed=2524):
    random = normal_generator(seed)
    return [next(random)*(.005 if (i//50)%2 == 0 else .025) for i in range(600)]


def variance_mixture(p=.2, ratio=5):
    assert 0 <= p <= 1 and ratio >= 1
    variance = 1-p+p*ratio**2
    sd = math.sqrt(variance)
    low, high = 1/sd, ratio/sd
    normal = statistics.NormalDist()
    return {'variance': variance, 'sd': sd, 'low': low, 'high': high,
            'kurtosis': 3*((1-p)+p*ratio**4)/variance**2,
            'density': lambda x: (1-p)*normal.pdf(x/low)/low+p*normal.pdf(x/high)/high,
            'tail': lambda threshold: (1-p)*math.erfc(abs(threshold)/low/math.sqrt(2))+p*math.erfc(abs(threshold)/high/math.sqrt(2))}


def realized_variance(log_prices, stride=1):
    assert isinstance(stride,int) and stride > 0 and (len(log_prices)-1)%stride == 0
    returns = [log_prices[i]-log_prices[i-stride] for i in range(stride,len(log_prices),stride)]
    variance = sum(r*r for r in returns)
    return {'variance':variance, 'volatility':math.sqrt(variance), 'returns':returns, 'count':len(returns)}


def intraday_sample(noise_bps=3, seed=81):
    assert 0 <= noise_bps <= 10
    random, signs = normal_generator(seed), uniform(seed+1000)
    latent = [0]
    for _ in range(390):
        latent.append(latent[-1]+.01/math.sqrt(390)*next(random))
    eta = noise_bps/10000
    observed = [p+eta*(-1 if next(signs)<.5 else 1) for p in latent]
    return {'latent':latent, 'observed':observed, 'eta':eta, 'integrated_variance':.01**2}


def intraday_profile(news=True):
    raw = [1+3*math.exp(-i/6)+2*math.exp(-(77-i)/7)+(5*math.exp(-.5*((i-30)/1.2)**2) if news else 0) for i in range(78)]
    total = sum(raw)
    return [x/total for x in raw]

"""Independent stdlib calculations for the prices / stochastic process lessons."""
import math
import statistics


PRICE_A=[100,102,101,103,102,104]
PRICE_B=[100,98,103,99,106,104]

def price_returns(prices):
    assert len(prices)>1 and all(p>0 and math.isfinite(p) for p in prices)
    return [{'simple':p/prices[i]-1,'log':math.log(p/prices[i])} for i,p in enumerate(prices[1:])]

def summary_stats(values):
    stats=moments(values)
    m3=statistics.mean((x-stats['mean'])**3 for x in values)
    return dict(stats,skewness=m3/stats['variance']**1.5 if stats['variance']>0 else None)

def arma11(phi,theta,innovation_variance=1,max_lag=20):
    assert abs(phi)<1 and innovation_variance>0
    numerator=1+theta*theta+2*phi*theta
    first=(phi+theta)*(1+phi*theta)/numerator
    return {'variance':innovation_variance*numerator/(1-phi*phi),'acf':[1]+[first*phi**(k-1) for k in range(1,max_lag+1)]}

def simulate_arma(phi,theta,count=600,seed=303):
    arma11(phi,theta)
    random=normal_generator(seed);out=[];previous=previous_shock=0
    for i in range(count+1000):
        shock=next(random);value=phi*previous+shock+theta*previous_shock
        if i>=1000:out.append(value)
        previous,previous_shock=value,shock
    return out

def fractional_weights(d,count=20):
    weights=[1]
    for j in range(1,count):weights.append(weights[-1]*(j-1-d)/j)
    return weights

def arfima_acf(d,max_lag=20):
    assert -.5<d<.5
    rho=[1]
    for k in range(1,max_lag+1):rho.append(rho[-1]*(k-1+d)/(k-d))
    return rho

def calendar_acf(strength=1,noise_sd=.01,max_lag=15):
    means=[v*strength for v in [-.004,.001,.001,.001,.001]]
    average=statistics.mean(means);a=[v-average for v in means]
    between=statistics.mean(x*x for x in a);variance=between+noise_sd**2
    return {'means':means,'between':between,'variance':variance,'acf':[1]+[statistics.mean(a[d]*a[(d-k)%5] for d in range(5))/variance for k in range(1,max_lag+1)]}

def squared_linear_correlation(psi,c4=0,innovation_variance=1,lag=1):
    gamma0=innovation_variance*sum(x*x for x in psi)
    gamma=innovation_variance*sum(psi[j]*psi[j+lag] for j in range(max(0,len(psi)-lag)))
    variance=2*gamma0**2+c4*sum(x**4 for x in psi)
    assert variance>0
    return (2*gamma**2+c4*sum(psi[j]**2*psi[j+lag]**2 for j in range(max(0,len(psi)-lag))))/variance

def standardized_t_pdf(x,nu=5):
    assert nu>2
    coefficient=math.exp(math.lgamma((nu+1)/2)-math.lgamma(nu/2))/math.sqrt(math.pi*(nu-2))
    return coefficient*(1+x*x/(nu-2))**(-(nu+1)/2)


def close(a,b,tol=1e-10):
    assert math.isclose(a,b,rel_tol=tol,abs_tol=tol),(a,b)


import json
import hashlib
sp500_snapshot_text = '{\n  "schema_version": 1,\n  "name": "S&P 500",\n  "symbol": "^GSPC",\n  "series_type": "Daily closing price index; excludes dividends",\n  "source": "Yahoo Finance via the arch 8.0.0 bundled example",\n  "source_url": "https://github.com/bashtage/arch/blob/v8.0.0/arch/data/sp500/sp500.csv.gz",\n  "source_download_url": "https://raw.githubusercontent.com/bashtage/arch/v8.0.0/arch/data/sp500/sp500.csv.gz",\n  "source_sha256": "1e028cbb9c400cc018c816ccc439b33c919387e726c3ed5ca2c05c82746059de",\n  "retrieved_on": "2026-09-19",\n  "price_start": "1999-01-04",\n  "price_end": "2018-12-31",\n  "return_start": "1999-01-05",\n  "return_end": "2018-12-31",\n  "price_count": 5031,\n  "return_count": 5030,\n  "return_method": "ln(Close[t] / Close[t-1]); decimal units; consecutive observed trading dates",\n  "processing": "All source rows retained. No filling, winsorizing, rounding or resampling. Date normalized to ISO 8601. First price is a baseline, not a zero return.",\n  "prices": [\n    ["1999-01-04", 1228.099976],\n    ["1999-01-05", 1244.780029],\n    ["1999-01-06", 1272.339966],\n    ["1999-01-07", 1269.72998],\n    ["1999-01-08", 1275.089966],\n    ["1999-01-11", 1263.880005],\n    ["1999-01-12", 1239.51001],\n    ["1999-01-13", 1234.400024],\n    ["1999-01-14", 1212.189941],\n    ["1999-01-15", 1243.26001],\n    ["1999-01-19", 1252.0],\n    ["1999-01-20", 1256.619995],\n    ["1999-01-21", 1235.160034],\n    ["1999-01-22", 1225.189941],\n    ["1999-01-25", 1233.97998],\n    ["1999-01-26", 1252.310059],\n    ["1999-01-27", 1243.170044],\n    ["1999-01-28", 1265.369995],\n    ["1999-01-29", 1279.640015],\n    ["1999-02-01", 1273.0],\n    ["1999-02-02", 1261.98999],\n    ["1999-02-03", 1272.069946],\n    ["1999-02-04", 1248.48999],\n    ["1999-02-05", 1239.400024],\n    ["1999-02-08", 1243.77002],\n    ["1999-02-09", 1216.140015],\n    ["1999-02-10", 1223.550049],\n    ["1999-02-11", 1254.040039],\n    ["1999-02-12", 1230.130005],\n    ["1999-02-16", 1241.869995],\n    ["1999-02-17", 1224.030029],\n    ["1999-02-18", 1237.280029],\n    ["1999-02-19", 1239.219971],\n    ["1999-02-22", 1272.140015],\n    ["1999-02-23", 1271.180054],\n    ["1999-02-24", 1253.410034],\n    ["1999-02-25", 1245.02002],\n    ["1999-02-26", 1238.329956],\n    ["1999-03-01", 1236.160034],\n    ["1999-03-02", 1225.5],\n    ["1999-03-03", 1227.699951],\n    ["1999-03-04", 1246.640015],\n    ["1999-03-05", 1275.469971],\n    ["1999-03-08", 1282.72998],\n    ["1999-03-09", 1279.839966],\n    ["1999-03-10", 1286.839966],\n    ["1999-03-11", 1297.680054],\n    ["1999-03-12", 1294.589966],\n    ["1999-03-15", 1307.26001],\n    ["1999-03-16", 1306.380005],\n    ["1999-03-17", 1297.819946],\n    ["1999-03-18", 1316.550049],\n    ["1999-03-19", 1299.290039],\n    ["1999-03-22", 1297.01001],\n    ["1999-03-23", 1262.140015],\n    ["1999-03-24", 1268.589966],\n    ["1999-03-25", 1289.98999],\n    ["1999-03-26", 1282.800049],\n    ["1999-03-29", 1310.170044],\n    ["1999-03-30", 1300.75],\n    ["1999-03-31", 1286.369995],\n    ["1999-04-01", 1293.719971],\n    ["1999-04-05", 1321.119995],\n    ["1999-04-06", 1317.890015],\n    ["1999-04-07", 1326.890015],\n    ["1999-04-08", 1343.97998],\n    ["1999-04-09", 1348.349976],\n    ["1999-04-12", 1358.630005],\n    ["1999-04-13", 1349.819946],\n    ["1999-04-14", 1328.439941],\n    ["1999-04-15", 1322.849976],\n    ["1999-04-16", 1319.0],\n    ["1999-04-19", 1289.47998],\n    ["1999-04-20", 1306.170044],\n    ["1999-04-21", 1336.119995],\n    ["1999-04-22", 1358.819946],\n    ["1999-04-23", 1356.849976],\n    ["1999-04-26", 1360.040039],\n    ["1999-04-27", 1362.800049],\n    ["1999-04-28", 1350.910034],\n    ["1999-04-29", 1342.829956],\n    ["1999-04-30", 1335.180054],\n    ["1999-05-03", 1354.630005],\n    ["1999-05-04", 1332.0],\n    ["1999-05-05", 1347.310059],\n    ["1999-05-06", 1332.050049],\n    ["1999-05-07", 1345.0],\n    ["1999-05-10", 1340.300049],\n    ["1999-05-11", 1355.609985],\n    ["1999-05-12", 1364.0],\n    ["1999-05-13", 1367.560059],\n    ["1999-05-14", 1337.800049],\n    ["1999-05-17", 1339.48999],\n    ["1999-05-18", 1333.319946],\n    ["1999-05-19", 1344.22998],\n    ["1999-05-20", 1338.829956],\n    ["1999-05-21", 1330.290039],\n    ["1999-05-24", 1306.650024],\n    ["1999-05-25", 1284.400024],\n    ["1999-05-26", 1304.76001],\n    ["1999-05-27", 1281.410034],\n    ["1999-05-28", 1301.839966],\n    ["1999-06-01", 1294.26001],\n    ["1999-06-02", 1294.810059],\n    ["1999-06-03", 1299.540039],\n    ["1999-06-04", 1327.75],\n    ["1999-06-07", 1334.52002],\n    ["1999-06-08", 1317.329956],\n    ["1999-06-09", 1318.640015],\n    ["1999-06-10", 1302.819946],\n    ["1999-06-11", 1293.640015],\n    ["1999-06-14", 1294.0],\n    ["1999-06-15", 1301.160034],\n    ["1999-06-16", 1330.410034],\n    ["1999-06-17", 1339.900024],\n    ["1999-06-18", 1342.839966],\n    ["1999-06-21", 1349.0],\n    ["1999-06-22", 1335.880005],\n    ["1999-06-23", 1333.060059],\n    ["1999-06-24", 1315.780029],\n    ["1999-06-25", 1315.310059],\n    ["1999-06-28", 1331.349976],\n    ["1999-06-29", 1351.449951],\n    ["1999-06-30", 1372.709961],\n    ["1999-07-01", 1380.959961],\n    ["1999-07-02", 1391.219971],\n    ["1999-07-06", 1388.119995],\n    ["1999-07-07", 1395.859985],\n    ["1999-07-08", 1394.420044],\n    ["1999-07-09", 1403.280029],\n    ["1999-07-12", 1399.099976],\n    ["1999-07-13", 1393.560059],\n    ["1999-07-14", 1398.170044],\n    ["1999-07-15", 1409.619995],\n    ["1999-07-16", 1418.780029],\n    ["1999-07-19", 1407.650024],\n    ["1999-07-20", 1377.099976],\n    ["1999-07-21", 1379.290039],\n    ["1999-07-22", 1360.969971],\n    ["1999-07-23", 1356.939941],\n    ["1999-07-26", 1347.76001],\n    ["1999-07-27", 1362.839966],\n    ["1999-07-28", 1365.400024],\n    ["1999-07-29", 1341.030029],\n    ["1999-07-30", 1328.719971],\n    ["1999-08-02", 1328.050049],\n    ["1999-08-03", 1322.180054],\n    ["1999-08-04", 1305.329956],\n    ["1999-08-05", 1313.709961],\n    ["1999-08-06", 1300.290039],\n    ["1999-08-09", 1297.800049],\n    ["1999-08-10", 1281.430054],\n    ["1999-08-11", 1301.930054],\n    ["1999-08-12", 1298.160034],\n    ["1999-08-13", 1327.680054],\n    ["1999-08-16", 1330.77002],\n    ["1999-08-17", 1344.160034],\n    ["1999-08-18", 1332.839966],\n    ["1999-08-19", 1323.589966],\n    ["1999-08-20", 1336.609985],\n    ["1999-08-23", 1360.219971],\n    ["1999-08-24", 1363.5],\n    ["1999-08-25", 1381.790039],\n    ["1999-08-26", 1362.01001],\n    ["1999-08-27", 1348.27002],\n    ["1999-08-30", 1324.02002],\n    ["1999-08-31", 1320.410034],\n    ["1999-09-01", 1331.069946],\n    ["1999-09-02", 1319.109985],\n    ["1999-09-03", 1357.23999],\n    ["1999-09-07", 1350.449951],\n    ["1999-09-08", 1344.150024],\n    ["1999-09-09", 1347.660034],\n    ["1999-09-10", 1351.660034],\n    ["1999-09-13", 1344.130005],\n    ["1999-09-14", 1336.290039],\n    ["1999-09-15", 1317.969971],\n    ["1999-09-16", 1318.47998],\n    ["1999-09-17", 1335.420044],\n    ["1999-09-20", 1335.530029],\n    ["1999-09-21", 1307.579956],\n    ["1999-09-22", 1310.51001],\n    ["1999-09-23", 1280.410034],\n    ["1999-09-24", 1277.359985],\n    ["1999-09-27", 1283.310059],\n    ["1999-09-28", 1282.199951],\n    ["1999-09-29", 1268.369995],\n    ["1999-09-30", 1282.709961],\n    ["1999-10-01", 1282.810059],\n    ["1999-10-04", 1304.599976],\n    ["1999-10-05", 1301.349976],\n    ["1999-10-06", 1325.400024],\n    ["1999-10-07", 1317.640015],\n    ["1999-10-08", 1336.02002],\n    ["1999-10-11", 1335.209961],\n    ["1999-10-12", 1313.040039],\n    ["1999-10-13", 1285.550049],\n    ["1999-10-14", 1283.420044],\n    ["1999-10-15", 1247.410034],\n    ["1999-10-18", 1254.130005],\n    ["1999-10-19", 1261.319946],\n    ["1999-10-20", 1289.430054],\n    ["1999-10-21", 1283.609985],\n    ["1999-10-22", 1301.650024],\n    ["1999-10-25", 1293.630005],\n    ["1999-10-26", 1281.910034],\n    ["1999-10-27", 1296.709961],\n    ["1999-10-28", 1342.439941],\n    ["1999-10-29", 1362.930054],\n    ["1999-11-01", 1354.119995],\n    ["1999-11-02", 1347.73999],\n    ["1999-11-03", 1354.930054],\n    ["1999-11-04", 1362.640015],\n    ["1999-11-05", 1370.22998],\n    ["1999-11-08", 1377.01001],\n    ["1999-11-09", 1365.280029],\n    ["1999-11-10", 1373.459961],\n    ["1999-11-11", 1381.459961],\n    ["1999-11-12", 1396.060059],\n    ["1999-11-15", 1394.390015],\n    ["1999-11-16", 1420.069946],\n    ["1999-11-17", 1410.709961],\n    ["1999-11-18", 1424.939941],\n    ["1999-11-19", 1422.0],\n    ["1999-11-22", 1420.939941],\n    ["1999-11-23", 1404.640015],\n    ["1999-11-24", 1417.079956],\n    ["1999-11-26", 1416.619995],\n    ["1999-11-29", 1407.829956],\n    ["1999-11-30", 1388.910034],\n    ["1999-12-01", 1397.719971],\n    ["1999-12-02", 1409.040039],\n    ["1999-12-03", 1433.300049],\n    ["1999-12-06", 1423.329956],\n    ["1999-12-07", 1409.170044],\n    ["1999-12-08", 1403.880005],\n    ["1999-12-09", 1408.109985],\n    ["1999-12-10", 1417.040039],\n    ["1999-12-13", 1415.219971],\n    ["1999-12-14", 1403.170044],\n    ["1999-12-15", 1413.329956],\n    ["1999-12-16", 1418.780029],\n    ["1999-12-17", 1421.030029],\n    ["1999-12-20", 1418.089966],\n    ["1999-12-21", 1433.430054],\n    ["1999-12-22", 1436.130005],\n    ["1999-12-23", 1458.339966],\n    ["1999-12-27", 1457.099976],\n    ["1999-12-28", 1457.660034],\n    ["1999-12-29", 1463.459961],\n    ["1999-12-30", 1464.469971],\n    ["1999-12-31", 1469.25],\n    ["2000-01-03", 1455.219971],\n    ["2000-01-04", 1399.420044],\n    ["2000-01-05", 1402.109985],\n    ["2000-01-06", 1403.449951],\n    ["2000-01-07", 1441.469971],\n    ["2000-01-10", 1457.599976],\n    ["2000-01-11", 1438.560059],\n    ["2000-01-12", 1432.25],\n    ["2000-01-13", 1449.680054],\n    ["2000-01-14", 1465.150024],\n    ["2000-01-18", 1455.140015],\n    ["2000-01-19", 1455.900024],\n    ["2000-01-20", 1445.569946],\n    ["2000-01-21", 1441.359985],\n    ["2000-01-24", 1401.530029],\n    ["2000-01-25", 1410.030029],\n    ["2000-01-26", 1404.089966],\n    ["2000-01-27", 1398.560059],\n    ["2000-01-28", 1360.160034],\n    ["2000-01-31", 1394.459961],\n    ["2000-02-01", 1409.280029],\n    ["2000-02-02", 1409.119995],\n    ["2000-02-03", 1424.969971],\n    ["2000-02-04", 1424.369995],\n    ["2000-02-07", 1424.23999],\n    ["2000-02-08", 1441.719971],\n    ["2000-02-09", 1411.709961],\n    ["2000-02-10", 1416.829956],\n    ["2000-02-11", 1387.119995],\n    ["2000-02-14", 1389.939941],\n    ["2000-02-15", 1402.050049],\n    ["2000-02-16", 1387.670044],\n    ["2000-02-17", 1388.26001],\n    ["2000-02-18", 1346.089966],\n    ["2000-02-22", 1352.170044],\n    ["2000-02-23", 1360.689941],\n    ["2000-02-24", 1353.430054],\n    ["2000-02-25", 1333.359985],\n    ["2000-02-28", 1348.050049],\n    ["2000-02-29", 1366.420044],\n    ["2000-03-01", 1379.189941],\n    ["2000-03-02", 1381.76001],\n    ["2000-03-03", 1409.170044],\n    ["2000-03-06", 1391.280029],\n    ["2000-03-07", 1355.619995],\n    ["2000-03-08", 1366.699951],\n    ["2000-03-09", 1401.689941],\n    ["2000-03-10", 1395.069946],\n    ["2000-03-13", 1383.619995],\n    ["2000-03-14", 1359.150024],\n    ["2000-03-15", 1392.140015],\n    ["2000-03-16", 1458.469971],\n    ["2000-03-17", 1464.469971],\n    ["2000-03-20", 1456.630005],\n    ["2000-03-21", 1493.869995],\n    ["2000-03-22", 1500.640015],\n    ["2000-03-23", 1527.349976],\n    ["2000-03-24", 1527.459961],\n    ["2000-03-27", 1523.859985],\n    ["2000-03-28", 1507.72998],\n    ["2000-03-29", 1508.52002],\n    ["2000-03-30", 1487.920044],\n    ["2000-03-31", 1498.579956],\n    ["2000-04-03", 1505.969971],\n    ["2000-04-04", 1494.72998],\n    ["2000-04-05", 1487.369995],\n    ["2000-04-06", 1501.339966],\n    ["2000-04-07", 1516.349976],\n    ["2000-04-10", 1504.459961],\n    ["2000-04-11", 1500.589966],\n    ["2000-04-12", 1467.170044],\n    ["2000-04-13", 1440.51001],\n    ["2000-04-14", 1356.560059],\n    ["2000-04-17", 1401.439941],\n    ["2000-04-18", 1441.609985],\n    ["2000-04-19", 1427.469971],\n    ["2000-04-20", 1434.540039],\n    ["2000-04-24", 1429.859985],\n    ["2000-04-25", 1477.439941],\n    ["2000-04-26", 1460.98999],\n    ["2000-04-27", 1464.920044],\n    ["2000-04-28", 1452.430054],\n    ["2000-05-01", 1468.25],\n    ["2000-05-02", 1446.290039],\n    ["2000-05-03", 1415.099976],\n    ["2000-05-04", 1409.569946],\n    ["2000-05-05", 1432.630005],\n    ["2000-05-08", 1424.170044],\n    ["2000-05-09", 1412.140015],\n    ["2000-05-10", 1383.050049],\n    ["2000-05-11", 1407.810059],\n    ["2000-05-12", 1420.959961],\n    ["2000-05-15", 1452.359985],\n    ["2000-05-16", 1466.040039],\n    ["2000-05-17", 1447.800049],\n    ["2000-05-18", 1437.209961],\n    ["2000-05-19", 1406.949951],\n    ["2000-05-22", 1400.719971],\n    ["2000-05-23", 1373.859985],\n    ["2000-05-24", 1399.050049],\n    ["2000-05-25", 1381.52002],\n    ["2000-05-26", 1378.02002],\n    ["2000-05-30", 1422.449951],\n    ["2000-05-31", 1420.599976],\n    ["2000-06-01", 1448.810059],\n    ["2000-06-02", 1477.26001],\n    ["2000-06-05", 1467.630005],\n    ["2000-06-06", 1457.839966],\n    ["2000-06-07", 1471.359985],\n    ["2000-06-08", 1461.670044],\n    ["2000-06-09", 1456.949951],\n    ["2000-06-12", 1446.0],\n    ["2000-06-13", 1469.439941],\n    ["2000-06-14", 1470.540039],\n    ["2000-06-15", 1478.72998],\n    ["2000-06-16", 1464.459961],\n    ["2000-06-19", 1486.0],\n    ["2000-06-20", 1475.949951],\n    ["2000-06-21", 1479.130005],\n    ["2000-06-22", 1452.180054],\n    ["2000-06-23", 1441.47998],\n    ["2000-06-26", 1455.310059],\n    ["2000-06-27", 1450.550049],\n    ["2000-06-28", 1454.819946],\n    ["2000-06-29", 1442.390015],\n    ["2000-06-30", 1454.599976],\n    ["2000-07-03", 1469.540039],\n    ["2000-07-05", 1446.22998],\n    ["2000-07-06", 1456.670044],\n    ["2000-07-07", 1478.900024],\n    ["2000-07-10", 1475.619995],\n    ["2000-07-11", 1480.880005],\n    ["2000-07-12", 1492.920044],\n    ["2000-07-13", 1495.839966],\n    ["2000-07-14", 1509.97998],\n    ["2000-07-17", 1510.48999],\n    ["2000-07-18", 1493.73999],\n    ["2000-07-19", 1481.959961],\n    ["2000-07-20", 1495.569946],\n    ["2000-07-21", 1480.189941],\n    ["2000-07-24", 1464.290039],\n    ["2000-07-25", 1474.469971],\n    ["2000-07-26", 1452.420044],\n    ["2000-07-27", 1449.619995],\n    ["2000-07-28", 1419.890015],\n    ["2000-07-31", 1430.829956],\n    ["2000-08-01", 1438.099976],\n    ["2000-08-02", 1438.699951],\n    ["2000-08-03", 1452.560059],\n    ["2000-08-04", 1462.930054],\n    ["2000-08-07", 1479.319946],\n    ["2000-08-08", 1482.800049],\n    ["2000-08-09", 1472.869995],\n    ["2000-08-10", 1460.25],\n    ["2000-08-11", 1471.839966],\n    ["2000-08-14", 1491.560059],\n    ["2000-08-15", 1484.430054],\n    ["2000-08-16", 1479.849976],\n    ["2000-08-17", 1496.069946],\n    ["2000-08-18", 1491.719971],\n    ["2000-08-21", 1499.47998],\n    ["2000-08-22", 1498.130005],\n    ["2000-08-23", 1505.969971],\n    ["2000-08-24", 1508.310059],\n    ["2000-08-25", 1506.449951],\n    ["2000-08-28", 1514.089966],\n    ["2000-08-29", 1509.839966],\n    ["2000-08-30", 1502.589966],\n    ["2000-08-31", 1517.680054],\n    ["2000-09-01", 1520.77002],\n    ["2000-09-05", 1507.079956],\n    ["2000-09-06", 1492.25],\n    ["2000-09-07", 1502.51001],\n    ["2000-09-08", 1494.5],\n    ["2000-09-11", 1489.26001],\n    ["2000-09-12", 1481.98999],\n    ["2000-09-13", 1484.910034],\n    ["2000-09-14", 1480.869995],\n    ["2000-09-15", 1465.810059],\n    ["2000-09-18", 1444.51001],\n    ["2000-09-19", 1459.900024],\n    ["2000-09-20", 1451.339966],\n    ["2000-09-21", 1449.050049],\n    ["2000-09-22", 1448.719971],\n    ["2000-09-25", 1439.030029],\n    ["2000-09-26", 1427.209961],\n    ["2000-09-27", 1426.569946],\n    ["2000-09-28", 1458.290039],\n    ["2000-09-29", 1436.51001],\n    ["2000-10-02", 1436.22998],\n    ["2000-10-03", 1426.459961],\n    ["2000-10-04", 1434.319946],\n    ["2000-10-05", 1436.280029],\n    ["2000-10-06", 1408.98999],\n    ["2000-10-09", 1402.030029],\n    ["2000-10-10", 1387.02002],\n    ["2000-10-11", 1364.589966],\n    ["2000-10-12", 1329.780029],\n    ["2000-10-13", 1374.170044],\n    ["2000-10-16", 1374.619995],\n    ["2000-10-17", 1349.969971],\n    ["2000-10-18", 1342.130005],\n    ["2000-10-19", 1388.76001],\n    ["2000-10-20", 1396.930054],\n    ["2000-10-23", 1395.780029],\n    ["2000-10-24", 1398.130005],\n    ["2000-10-25", 1364.900024],\n    ["2000-10-26", 1364.439941],\n    ["2000-10-27", 1379.579956],\n    ["2000-10-30", 1398.660034],\n    ["2000-10-31", 1429.400024],\n    ["2000-11-01", 1421.219971],\n    ["2000-11-02", 1428.319946],\n    ["2000-11-03", 1426.689941],\n    ["2000-11-06", 1432.189941],\n    ["2000-11-07", 1431.869995],\n    ["2000-11-08", 1409.280029],\n    ["2000-11-09", 1400.140015],\n    ["2000-11-10", 1365.97998],\n    ["2000-11-13", 1351.26001],\n    ["2000-11-14", 1382.949951],\n    ["2000-11-15", 1389.810059],\n    ["2000-11-16", 1372.319946],\n    ["2000-11-17", 1367.719971],\n    ["2000-11-20", 1342.619995],\n    ["2000-11-21", 1347.349976],\n    ["2000-11-22", 1322.359985],\n    ["2000-11-24", 1341.77002],\n    ["2000-11-27", 1348.969971],\n    ["2000-11-28", 1336.089966],\n    ["2000-11-29", 1341.930054],\n    ["2000-11-30", 1314.949951],\n    ["2000-12-01", 1315.22998],\n    ["2000-12-04", 1324.969971],\n    ["2000-12-05", 1376.540039],\n    ["2000-12-06", 1351.459961],\n    ["2000-12-07", 1343.550049],\n    ["2000-12-08", 1369.890015],\n    ["2000-12-11", 1380.199951],\n    ["2000-12-12", 1371.180054],\n    ["2000-12-13", 1359.98999],\n    ["2000-12-14", 1340.930054],\n    ["2000-12-15", 1312.150024],\n    ["2000-12-18", 1322.73999],\n    ["2000-12-19", 1305.599976],\n    ["2000-12-20", 1264.73999],\n    ["2000-12-21", 1274.859985],\n    ["2000-12-22", 1305.949951],\n    ["2000-12-26", 1315.189941],\n    ["2000-12-27", 1328.920044],\n    ["2000-12-28", 1334.219971],\n    ["2000-12-29", 1320.280029],\n    ["2001-01-02", 1283.27002],\n    ["2001-01-03", 1347.560059],\n    ["2001-01-04", 1333.339966],\n    ["2001-01-05", 1298.349976],\n    ["2001-01-08", 1295.859985],\n    ["2001-01-09", 1300.800049],\n    ["2001-01-10", 1313.27002],\n    ["2001-01-11", 1326.819946],\n    ["2001-01-12", 1318.550049],\n    ["2001-01-16", 1326.650024],\n    ["2001-01-17", 1329.469971],\n    ["2001-01-18", 1347.969971],\n    ["2001-01-19", 1342.540039],\n    ["2001-01-22", 1342.900024],\n    ["2001-01-23", 1360.400024],\n    ["2001-01-24", 1364.300049],\n    ["2001-01-25", 1357.51001],\n    ["2001-01-26", 1354.949951],\n    ["2001-01-29", 1364.170044],\n    ["2001-01-30", 1373.72998],\n    ["2001-01-31", 1366.01001],\n    ["2001-02-01", 1373.469971],\n    ["2001-02-02", 1349.469971],\n    ["2001-02-05", 1354.310059],\n    ["2001-02-06", 1352.26001],\n    ["2001-02-07", 1340.890015],\n    ["2001-02-08", 1332.530029],\n    ["2001-02-09", 1314.76001],\n    ["2001-02-12", 1330.310059],\n    ["2001-02-13", 1318.800049],\n    ["2001-02-14", 1315.920044],\n    ["2001-02-15", 1326.609985],\n    ["2001-02-16", 1301.530029],\n    ["2001-02-20", 1278.939941],\n    ["2001-02-21", 1255.27002],\n    ["2001-02-22", 1252.819946],\n    ["2001-02-23", 1245.859985],\n    ["2001-02-26", 1267.650024],\n    ["2001-02-27", 1257.939941],\n    ["2001-02-28", 1239.939941],\n    ["2001-03-01", 1241.22998],\n    ["2001-03-02", 1234.180054],\n    ["2001-03-05", 1241.410034],\n    ["2001-03-06", 1253.800049],\n    ["2001-03-07", 1261.890015],\n    ["2001-03-08", 1264.73999],\n    ["2001-03-09", 1233.420044],\n    ["2001-03-12", 1180.160034],\n    ["2001-03-13", 1197.660034],\n    ["2001-03-14", 1166.709961],\n    ["2001-03-15", 1173.560059],\n    ["2001-03-16", 1150.530029],\n    ["2001-03-19", 1170.810059],\n    ["2001-03-20", 1142.619995],\n    ["2001-03-21", 1122.140015],\n    ["2001-03-22", 1117.579956],\n    ["2001-03-23", 1139.829956],\n    ["2001-03-26", 1152.689941],\n    ["2001-03-27", 1182.170044],\n    ["2001-03-28", 1153.290039],\n    ["2001-03-29", 1147.949951],\n    ["2001-03-30", 1160.329956],\n    ["2001-04-02", 1145.869995],\n    ["2001-04-03", 1106.459961],\n    ["2001-04-04", 1103.25],\n    ["2001-04-05", 1151.439941],\n    ["2001-04-06", 1128.430054],\n    ["2001-04-09", 1137.589966],\n    ["2001-04-10", 1168.380005],\n    ["2001-04-11", 1165.890015],\n    ["2001-04-12", 1183.5],\n    ["2001-04-16", 1179.680054],\n    ["2001-04-17", 1191.810059],\n    ["2001-04-18", 1238.160034],\n    ["2001-04-19", 1253.689941],\n    ["2001-04-20", 1242.97998],\n    ["2001-04-23", 1224.359985],\n    ["2001-04-24", 1209.469971],\n    ["2001-04-25", 1228.75],\n    ["2001-04-26", 1234.52002],\n    ["2001-04-27", 1253.050049],\n    ["2001-04-30", 1249.459961],\n    ["2001-05-01", 1266.439941],\n    ["2001-05-02", 1267.430054],\n    ["2001-05-03", 1248.579956],\n    ["2001-05-04", 1266.609985],\n    ["2001-05-07", 1263.51001],\n    ["2001-05-08", 1261.199951],\n    ["2001-05-09", 1255.540039],\n    ["2001-05-10", 1255.180054],\n    ["2001-05-11", 1245.670044],\n    ["2001-05-14", 1248.920044],\n    ["2001-05-15", 1249.439941],\n    ["2001-05-16", 1284.98999],\n    ["2001-05-17", 1288.48999],\n    ["2001-05-18", 1291.959961],\n    ["2001-05-21", 1312.829956],\n    ["2001-05-22", 1309.380005],\n    ["2001-05-23", 1289.050049],\n    ["2001-05-24", 1293.170044],\n    ["2001-05-25", 1277.890015],\n    ["2001-05-29", 1267.930054],\n    ["2001-05-30", 1248.079956],\n    ["2001-05-31", 1255.819946],\n    ["2001-06-01", 1260.670044],\n    ["2001-06-04", 1267.109985],\n    ["2001-06-05", 1283.569946],\n    ["2001-06-06", 1270.030029],\n    ["2001-06-07", 1276.959961],\n    ["2001-06-08", 1264.959961],\n    ["2001-06-11", 1254.390015],\n    ["2001-06-12", 1255.849976],\n    ["2001-06-13", 1241.599976],\n    ["2001-06-14", 1219.869995],\n    ["2001-06-15", 1214.359985],\n    ["2001-06-18", 1208.430054],\n    ["2001-06-19", 1212.579956],\n    ["2001-06-20", 1223.140015],\n    ["2001-06-21", 1237.040039],\n    ["2001-06-22", 1225.349976],\n    ["2001-06-25", 1218.599976],\n    ["2001-06-26", 1216.76001],\n    ["2001-06-27", 1211.069946],\n    ["2001-06-28", 1226.199951],\n    ["2001-06-29", 1224.380005],\n    ["2001-07-02", 1236.719971],\n    ["2001-07-03", 1234.449951],\n    ["2001-07-05", 1219.23999],\n    ["2001-07-06", 1190.589966],\n    ["2001-07-09", 1198.780029],\n    ["2001-07-10", 1181.52002],\n    ["2001-07-11", 1180.180054],\n    ["2001-07-12", 1208.140015],\n    ["2001-07-13", 1215.680054],\n    ["2001-07-16", 1202.449951],\n    ["2001-07-17", 1214.439941],\n    ["2001-07-18", 1207.709961],\n    ["2001-07-19", 1215.02002],\n    ["2001-07-20", 1210.849976],\n    ["2001-07-23", 1191.030029],\n    ["2001-07-24", 1171.650024],\n    ["2001-07-25", 1190.48999],\n    ["2001-07-26", 1202.930054],\n    ["2001-07-27", 1205.819946],\n    ["2001-07-30", 1204.52002],\n    ["2001-07-31", 1211.22998],\n    ["2001-08-01", 1215.930054],\n    ["2001-08-02", 1220.75],\n    ["2001-08-03", 1214.349976],\n    ["2001-08-06", 1200.47998],\n    ["2001-08-07", 1204.400024],\n    ["2001-08-08", 1183.530029],\n    ["2001-08-09", 1183.430054],\n    ["2001-08-10", 1190.160034],\n    ["2001-08-13", 1191.290039],\n    ["2001-08-14", 1186.72998],\n    ["2001-08-15", 1178.02002],\n    ["2001-08-16", 1181.660034],\n    ["2001-08-17", 1161.969971],\n    ["2001-08-20", 1171.410034],\n    ["2001-08-21", 1157.26001],\n    ["2001-08-22", 1165.310059],\n    ["2001-08-23", 1162.089966],\n    ["2001-08-24", 1184.930054],\n    ["2001-08-27", 1179.209961],\n    ["2001-08-28", 1161.51001],\n    ["2001-08-29", 1148.560059],\n    ["2001-08-30", 1129.030029],\n    ["2001-08-31", 1133.579956],\n    ["2001-09-04", 1132.939941],\n    ["2001-09-05", 1131.73999],\n    ["2001-09-06", 1106.400024],\n    ["2001-09-07", 1085.780029],\n    ["2001-09-10", 1092.540039],\n    ["2001-09-17", 1038.77002],\n    ["2001-09-18", 1032.73999],\n    ["2001-09-19", 1016.099976],\n    ["2001-09-20", 984.539978],\n    ["2001-09-21", 965.799988],\n    ["2001-09-24", 1003.450012],\n    ["2001-09-25", 1012.27002],\n    ["2001-09-26", 1007.039978],\n    ["2001-09-27", 1018.609985],\n    ["2001-09-28", 1040.939941],\n    ["2001-10-01", 1038.550049],\n    ["2001-10-02", 1051.329956],\n    ["2001-10-03", 1072.280029],\n    ["2001-10-04", 1069.630005],\n    ["2001-10-05", 1071.380005],\n    ["2001-10-08", 1062.439941],\n    ["2001-10-09", 1056.75],\n    ["2001-10-10", 1080.98999],\n    ["2001-10-11", 1097.430054],\n    ["2001-10-12", 1091.650024],\n    ["2001-10-15", 1089.97998],\n    ["2001-10-16", 1097.540039],\n    ["2001-10-17", 1077.089966],\n    ["2001-10-18", 1068.609985],\n    ["2001-10-19", 1073.47998],\n    ["2001-10-22", 1089.900024],\n    ["2001-10-23", 1084.780029],\n    ["2001-10-24", 1085.199951],\n    ["2001-10-25", 1100.089966],\n    ["2001-10-26", 1104.609985],\n    ["2001-10-29", 1078.300049],\n    ["2001-10-30", 1059.790039],\n    ["2001-10-31", 1059.780029],\n    ["2001-11-01", 1084.099976],\n    ["2001-11-02", 1087.199951],\n    ["2001-11-05", 1102.839966],\n    ["2001-11-06", 1118.859985],\n    ["2001-11-07", 1115.800049],\n    ["2001-11-08", 1118.540039],\n    ["2001-11-09", 1120.310059],\n    ["2001-11-12", 1118.329956],\n    ["2001-11-13", 1139.089966],\n    ["2001-11-14", 1141.209961],\n    ["2001-11-15", 1142.23999],\n    ["2001-11-16", 1138.650024],\n    ["2001-11-19", 1151.060059],\n    ["2001-11-20", 1142.660034],\n    ["2001-11-21", 1137.030029],\n    ["2001-11-23", 1150.339966],\n    ["2001-11-26", 1157.420044],\n    ["2001-11-27", 1149.5],\n    ["2001-11-28", 1128.52002],\n    ["2001-11-29", 1140.199951],\n    ["2001-11-30", 1139.449951],\n    ["2001-12-03", 1129.900024],\n    ["2001-12-04", 1144.800049],\n    ["2001-12-05", 1170.349976],\n    ["2001-12-06", 1167.099976],\n    ["2001-12-07", 1158.310059],\n    ["2001-12-10", 1139.930054],\n    ["2001-12-11", 1136.76001],\n    ["2001-12-12", 1137.069946],\n    ["2001-12-13", 1119.380005],\n    ["2001-12-14", 1123.089966],\n    ["2001-12-17", 1134.359985],\n    ["2001-12-18", 1142.920044],\n    ["2001-12-19", 1149.560059],\n    ["2001-12-20", 1139.930054],\n    ["2001-12-21", 1144.890015],\n    ["2001-12-24", 1144.650024],\n    ["2001-12-26", 1149.369995],\n    ["2001-12-27", 1157.130005],\n    ["2001-12-28", 1161.02002],\n    ["2001-12-31", 1148.079956],\n    ["2002-01-02", 1154.670044],\n    ["2002-01-03", 1165.27002],\n    ["2002-01-04", 1172.51001],\n    ["2002-01-07", 1164.890015],\n    ["2002-01-08", 1160.709961],\n    ["2002-01-09", 1155.140015],\n    ["2002-01-10", 1156.550049],\n    ["2002-01-11", 1145.599976],\n    ["2002-01-14", 1138.410034],\n    ["2002-01-15", 1146.189941],\n    ["2002-01-16", 1127.569946],\n    ["2002-01-17", 1138.880005],\n    ["2002-01-18", 1127.579956],\n    ["2002-01-22", 1119.310059],\n    ["2002-01-23", 1128.180054],\n    ["2002-01-24", 1132.150024],\n    ["2002-01-25", 1133.280029],\n    ["2002-01-28", 1133.060059],\n    ["2002-01-29", 1100.640015],\n    ["2002-01-30", 1113.569946],\n    ["2002-01-31", 1130.199951],\n    ["2002-02-01", 1122.199951],\n    ["2002-02-04", 1094.439941],\n    ["2002-02-05", 1090.02002],\n    ["2002-02-06", 1083.51001],\n    ["2002-02-07", 1080.170044],\n    ["2002-02-08", 1096.219971],\n    ["2002-02-11", 1111.939941],\n    ["2002-02-12", 1107.5],\n    ["2002-02-13", 1118.51001],\n    ["2002-02-14", 1116.47998],\n    ["2002-02-15", 1104.180054],\n    ["2002-02-19", 1083.339966],\n    ["2002-02-20", 1097.97998],\n    ["2002-02-21", 1080.949951],\n    ["2002-02-22", 1089.839966],\n    ["2002-02-25", 1109.430054],\n    ["2002-02-26", 1109.380005],\n    ["2002-02-27", 1109.890015],\n    ["2002-02-28", 1106.72998],\n    ["2002-03-01", 1131.780029],\n    ["2002-03-04", 1153.839966],\n    ["2002-03-05", 1146.140015],\n    ["2002-03-06", 1162.77002],\n    ["2002-03-07", 1157.540039],\n    ["2002-03-08", 1164.310059],\n    ["2002-03-11", 1168.26001],\n    ["2002-03-12", 1165.579956],\n    ["2002-03-13", 1154.089966],\n    ["2002-03-14", 1153.040039],\n    ["2002-03-15", 1166.160034],\n    ["2002-03-18", 1165.550049],\n    ["2002-03-19", 1170.290039],\n    ["2002-03-20", 1151.849976],\n    ["2002-03-21", 1153.589966],\n    ["2002-03-22", 1148.699951],\n    ["2002-03-25", 1131.869995],\n    ["2002-03-26", 1138.48999],\n    ["2002-03-27", 1144.579956],\n    ["2002-03-28", 1147.390015],\n    ["2002-04-01", 1146.540039],\n    ["2002-04-02", 1136.76001],\n    ["2002-04-03", 1125.400024],\n    ["2002-04-04", 1126.339966],\n    ["2002-04-05", 1122.72998],\n    ["2002-04-08", 1125.290039],\n    ["2002-04-09", 1117.800049],\n    ["2002-04-10", 1130.469971],\n    ["2002-04-11", 1103.689941],\n    ["2002-04-12", 1111.01001],\n    ["2002-04-15", 1102.550049],\n    ["2002-04-16", 1128.369995],\n    ["2002-04-17", 1126.069946],\n    ["2002-04-18", 1124.469971],\n    ["2002-04-19", 1125.170044],\n    ["2002-04-22", 1107.829956],\n    ["2002-04-23", 1100.959961],\n    ["2002-04-24", 1093.140015],\n    ["2002-04-25", 1091.47998],\n    ["2002-04-26", 1076.319946],\n    ["2002-04-29", 1065.449951],\n    ["2002-04-30", 1076.920044],\n    ["2002-05-01", 1086.459961],\n    ["2002-05-02", 1084.560059],\n    ["2002-05-03", 1073.430054],\n    ["2002-05-06", 1052.670044],\n    ["2002-05-07", 1049.48999],\n    ["2002-05-08", 1088.849976],\n    ["2002-05-09", 1073.01001],\n    ["2002-05-10", 1054.98999],\n    ["2002-05-13", 1074.560059],\n    ["2002-05-14", 1097.280029],\n    ["2002-05-15", 1091.069946],\n    ["2002-05-16", 1098.22998],\n    ["2002-05-17", 1106.589966],\n    ["2002-05-20", 1091.880005],\n    ["2002-05-21", 1079.880005],\n    ["2002-05-22", 1086.02002],\n    ["2002-05-23", 1097.079956],\n    ["2002-05-24", 1083.819946],\n    ["2002-05-28", 1074.550049],\n    ["2002-05-29", 1067.660034],\n    ["2002-05-30", 1064.660034],\n    ["2002-05-31", 1067.140015],\n    ["2002-06-03", 1040.680054],\n    ["2002-06-04", 1040.689941],\n    ["2002-06-05", 1049.900024],\n    ["2002-06-06", 1029.150024],\n    ["2002-06-07", 1027.530029],\n    ["2002-06-10", 1030.73999],\n    ["2002-06-11", 1013.599976],\n    ["2002-06-12", 1020.26001],\n    ["2002-06-13", 1009.559998],\n    ["2002-06-14", 1007.27002],\n    ["2002-06-17", 1036.170044],\n    ["2002-06-18", 1037.140015],\n    ["2002-06-19", 1019.98999],\n    ["2002-06-20", 1006.289978],\n    ["2002-06-21", 989.140015],\n    ["2002-06-24", 992.719971],\n    ["2002-06-25", 976.140015],\n    ["2002-06-26", 973.530029],\n    ["2002-06-27", 990.640015],\n    ["2002-06-28", 989.820007],\n    ["2002-07-01", 968.650024],\n    ["2002-07-02", 948.090027],\n    ["2002-07-03", 953.98999],\n    ["2002-07-05", 989.030029],\n    ["2002-07-08", 976.97998],\n    ["2002-07-09", 952.830017],\n    ["2002-07-10", 920.469971],\n    ["2002-07-11", 927.369995],\n    ["2002-07-12", 921.390015],\n    ["2002-07-15", 917.929993],\n    ["2002-07-16", 900.940002],\n    ["2002-07-17", 906.039978],\n    ["2002-07-18", 881.559998],\n    ["2002-07-19", 847.75],\n    ["2002-07-22", 819.849976],\n    ["2002-07-23", 797.700012],\n    ["2002-07-24", 843.429993],\n    ["2002-07-25", 838.679993],\n    ["2002-07-26", 852.840027],\n    ["2002-07-29", 898.960022],\n    ["2002-07-30", 902.780029],\n    ["2002-07-31", 911.619995],\n    ["2002-08-01", 884.659973],\n    ["2002-08-02", 864.23999],\n    ["2002-08-05", 834.599976],\n    ["2002-08-06", 859.570007],\n    ["2002-08-07", 876.77002],\n    ["2002-08-08", 905.460022],\n    ["2002-08-09", 908.640015],\n    ["2002-08-12", 903.799988],\n    ["2002-08-13", 884.210022],\n    ["2002-08-14", 919.619995],\n    ["2002-08-15", 930.25],\n    ["2002-08-16", 928.77002],\n    ["2002-08-19", 950.700012],\n    ["2002-08-20", 937.429993],\n    ["2002-08-21", 949.359985],\n    ["2002-08-22", 962.700012],\n    ["2002-08-23", 940.859985],\n    ["2002-08-26", 947.950012],\n    ["2002-08-27", 934.820007],\n    ["2002-08-28", 917.869995],\n    ["2002-08-29", 917.799988],\n    ["2002-08-30", 916.070007],\n    ["2002-09-03", 878.02002],\n    ["2002-09-04", 893.400024],\n    ["2002-09-05", 879.150024],\n    ["2002-09-06", 893.919983],\n    ["2002-09-09", 902.960022],\n    ["2002-09-10", 909.580017],\n    ["2002-09-11", 909.450012],\n    ["2002-09-12", 886.909973],\n    ["2002-09-13", 889.809998],\n    ["2002-09-16", 891.099976],\n    ["2002-09-17", 873.52002],\n    ["2002-09-18", 869.460022],\n    ["2002-09-19", 843.320007],\n    ["2002-09-20", 845.390015],\n    ["2002-09-23", 833.700012],\n    ["2002-09-24", 819.289978],\n    ["2002-09-25", 839.659973],\n    ["2002-09-26", 854.950012],\n    ["2002-09-27", 827.369995],\n    ["2002-09-30", 815.280029],\n    ["2002-10-01", 847.909973],\n    ["2002-10-02", 827.909973],\n    ["2002-10-03", 818.950012],\n    ["2002-10-04", 800.580017],\n    ["2002-10-07", 785.280029],\n    ["2002-10-08", 798.549988],\n    ["2002-10-09", 776.76001],\n    ["2002-10-10", 803.919983],\n    ["2002-10-11", 835.320007],\n    ["2002-10-14", 841.440002],\n    ["2002-10-15", 881.27002],\n    ["2002-10-16", 860.02002],\n    ["2002-10-17", 879.200012],\n    ["2002-10-18", 884.390015],\n    ["2002-10-21", 899.719971],\n    ["2002-10-22", 890.159973],\n    ["2002-10-23", 896.140015],\n    ["2002-10-24", 882.5],\n    ["2002-10-25", 897.650024],\n    ["2002-10-28", 890.22998],\n    ["2002-10-29", 882.150024],\n    ["2002-10-30", 890.710022],\n    ["2002-10-31", 885.76001],\n    ["2002-11-01", 900.960022],\n    ["2002-11-04", 908.349976],\n    ["2002-11-05", 915.390015],\n    ["2002-11-06", 923.76001],\n    ["2002-11-07", 902.650024],\n    ["2002-11-08", 894.73999],\n    ["2002-11-11", 876.190002],\n    ["2002-11-12", 882.950012],\n    ["2002-11-13", 882.530029],\n    ["2002-11-14", 904.27002],\n    ["2002-11-15", 909.830017],\n    ["2002-11-18", 900.359985],\n    ["2002-11-19", 896.73999],\n    ["2002-11-20", 914.150024],\n    ["2002-11-21", 933.76001],\n    ["2002-11-22", 930.549988],\n    ["2002-11-25", 932.869995],\n    ["2002-11-26", 913.309998],\n    ["2002-11-27", 938.869995],\n    ["2002-11-29", 936.309998],\n    ["2002-12-02", 934.530029],\n    ["2002-12-03", 920.75],\n    ["2002-12-04", 917.580017],\n    ["2002-12-05", 906.549988],\n    ["2002-12-06", 912.22998],\n    ["2002-12-09", 892.0],\n    ["2002-12-10", 904.450012],\n    ["2002-12-11", 904.960022],\n    ["2002-12-12", 901.580017],\n    ["2002-12-13", 889.47998],\n    ["2002-12-16", 910.400024],\n    ["2002-12-17", 902.98999],\n    ["2002-12-18", 891.119995],\n    ["2002-12-19", 884.25],\n    ["2002-12-20", 895.76001],\n    ["2002-12-23", 897.380005],\n    ["2002-12-24", 892.469971],\n    ["2002-12-26", 889.659973],\n    ["2002-12-27", 875.400024],\n    ["2002-12-30", 879.390015],\n    ["2002-12-31", 879.820007],\n    ["2003-01-02", 909.030029],\n    ["2003-01-03", 908.590027],\n    ["2003-01-06", 929.01001],\n    ["2003-01-07", 922.929993],\n    ["2003-01-08", 909.929993],\n    ["2003-01-09", 927.570007],\n    ["2003-01-10", 927.570007],\n    ["2003-01-13", 926.26001],\n    ["2003-01-14", 931.659973],\n    ["2003-01-15", 918.219971],\n    ["2003-01-16", 914.599976],\n    ["2003-01-17", 901.780029],\n    ["2003-01-21", 887.619995],\n    ["2003-01-22", 878.359985],\n    ["2003-01-23", 887.340027],\n    ["2003-01-24", 861.400024],\n    ["2003-01-27", 847.47998],\n    ["2003-01-28", 858.539978],\n    ["2003-01-29", 864.359985],\n    ["2003-01-30", 844.609985],\n    ["2003-01-31", 855.700012],\n    ["2003-02-03", 860.320007],\n    ["2003-02-04", 848.200012],\n    ["2003-02-05", 843.590027],\n    ["2003-02-06", 838.150024],\n    ["2003-02-07", 829.690002],\n    ["2003-02-10", 835.969971],\n    ["2003-02-11", 829.200012],\n    ["2003-02-12", 818.679993],\n    ["2003-02-13", 817.369995],\n    ["2003-02-14", 834.890015],\n    ["2003-02-18", 851.169983],\n    ["2003-02-19", 845.130005],\n    ["2003-02-20", 837.099976],\n    ["2003-02-21", 848.169983],\n    ["2003-02-24", 832.580017],\n    ["2003-02-25", 838.570007],\n    ["2003-02-26", 827.549988],\n    ["2003-02-27", 837.280029],\n    ["2003-02-28", 841.150024],\n    ["2003-03-03", 834.809998],\n    ["2003-03-04", 821.98999],\n    ["2003-03-05", 829.849976],\n    ["2003-03-06", 822.099976],\n    ["2003-03-07", 828.890015],\n    ["2003-03-10", 807.47998],\n    ["2003-03-11", 800.72998],\n    ["2003-03-12", 804.190002],\n    ["2003-03-13", 831.900024],\n    ["2003-03-14", 833.27002],\n    ["2003-03-17", 862.789978],\n    ["2003-03-18", 866.450012],\n    ["2003-03-19", 874.02002],\n    ["2003-03-20", 875.669983],\n    ["2003-03-21", 895.789978],\n    ["2003-03-24", 864.22998],\n    ["2003-03-25", 874.73999],\n    ["2003-03-26", 869.950012],\n    ["2003-03-27", 868.52002],\n    ["2003-03-28", 863.5],\n    ["2003-03-31", 848.179993],\n    ["2003-04-01", 858.47998],\n    ["2003-04-02", 880.900024],\n    ["2003-04-03", 876.450012],\n    ["2003-04-04", 878.849976],\n    ["2003-04-07", 879.929993],\n    ["2003-04-08", 878.289978],\n    ["2003-04-09", 865.98999],\n    ["2003-04-10", 871.580017],\n    ["2003-04-11", 868.299988],\n    ["2003-04-14", 885.22998],\n    ["2003-04-15", 890.809998],\n    ["2003-04-16", 879.909973],\n    ["2003-04-17", 893.580017],\n    ["2003-04-21", 892.01001],\n    ["2003-04-22", 911.369995],\n    ["2003-04-23", 919.02002],\n    ["2003-04-24", 911.429993],\n    ["2003-04-25", 898.809998],\n    ["2003-04-28", 914.840027],\n    ["2003-04-29", 917.840027],\n    ["2003-04-30", 916.919983],\n    ["2003-05-01", 916.299988],\n    ["2003-05-02", 930.080017],\n    ["2003-05-05", 926.549988],\n    ["2003-05-06", 934.390015],\n    ["2003-05-07", 929.619995],\n    ["2003-05-08", 920.27002],\n    ["2003-05-09", 933.409973],\n    ["2003-05-12", 945.109985],\n    ["2003-05-13", 942.299988],\n    ["2003-05-14", 939.280029],\n    ["2003-05-15", 946.669983],\n    ["2003-05-16", 944.299988],\n    ["2003-05-19", 920.77002],\n    ["2003-05-20", 919.72998],\n    ["2003-05-21", 923.419983],\n    ["2003-05-22", 931.869995],\n    ["2003-05-23", 933.219971],\n    ["2003-05-27", 951.47998],\n    ["2003-05-28", 953.219971],\n    ["2003-05-29", 949.640015],\n    ["2003-05-30", 963.590027],\n    ["2003-06-02", 967.0],\n    ["2003-06-03", 971.559998],\n    ["2003-06-04", 986.23999],\n    ["2003-06-05", 990.140015],\n    ["2003-06-06", 987.76001],\n    ["2003-06-09", 975.929993],\n    ["2003-06-10", 984.840027],\n    ["2003-06-11", 997.47998],\n    ["2003-06-12", 998.51001],\n    ["2003-06-13", 988.609985],\n    ["2003-06-16", 1010.73999],\n    ["2003-06-17", 1011.659973],\n    ["2003-06-18", 1010.090027],\n    ["2003-06-19", 994.700012],\n    ["2003-06-20", 995.690002],\n    ["2003-06-23", 981.640015],\n    ["2003-06-24", 983.450012],\n    ["2003-06-25", 975.320007],\n    ["2003-06-26", 985.820007],\n    ["2003-06-27", 976.219971],\n    ["2003-06-30", 974.5],\n    ["2003-07-01", 982.320007],\n    ["2003-07-02", 993.75],\n    ["2003-07-03", 985.700012],\n    ["2003-07-07", 1004.419983],\n    ["2003-07-08", 1007.840027],\n    ["2003-07-09", 1002.210022],\n    ["2003-07-10", 988.700012],\n    ["2003-07-11", 998.140015],\n    ["2003-07-14", 1003.859985],\n    ["2003-07-15", 1000.419983],\n    ["2003-07-16", 994.090027],\n    ["2003-07-17", 981.72998],\n    ["2003-07-18", 993.320007],\n    ["2003-07-21", 978.799988],\n    ["2003-07-22", 988.109985],\n    ["2003-07-23", 988.609985],\n    ["2003-07-24", 981.599976],\n    ["2003-07-25", 998.679993],\n    ["2003-07-28", 996.52002],\n    ["2003-07-29", 989.280029],\n    ["2003-07-30", 987.48999],\n    ["2003-07-31", 990.309998],\n    ["2003-08-01", 980.150024],\n    ["2003-08-04", 982.820007],\n    ["2003-08-05", 965.460022],\n    ["2003-08-06", 967.080017],\n    ["2003-08-07", 974.119995],\n    ["2003-08-08", 977.590027],\n    ["2003-08-11", 980.590027],\n    ["2003-08-12", 990.349976],\n    ["2003-08-13", 984.030029],\n    ["2003-08-14", 990.51001],\n    ["2003-08-15", 990.669983],\n    ["2003-08-18", 999.73999],\n    ["2003-08-19", 1002.349976],\n    ["2003-08-20", 1000.299988],\n    ["2003-08-21", 1003.27002],\n    ["2003-08-22", 993.059998],\n    ["2003-08-25", 993.710022],\n    ["2003-08-26", 996.72998],\n    ["2003-08-27", 996.789978],\n    ["2003-08-28", 1002.840027],\n    ["2003-08-29", 1008.01001],\n    ["2003-09-02", 1021.98999],\n    ["2003-09-03", 1026.27002],\n    ["2003-09-04", 1027.969971],\n    ["2003-09-05", 1021.390015],\n    ["2003-09-08", 1031.640015],\n    ["2003-09-09", 1023.169983],\n    ["2003-09-10", 1010.919983],\n    ["2003-09-11", 1016.419983],\n    ["2003-09-12", 1018.630005],\n    ["2003-09-15", 1014.809998],\n    ["2003-09-16", 1029.319946],\n    ["2003-09-17", 1025.969971],\n    ["2003-09-18", 1039.579956],\n    ["2003-09-19", 1036.300049],\n    ["2003-09-22", 1022.820007],\n    ["2003-09-23", 1029.030029],\n    ["2003-09-24", 1009.380005],\n    ["2003-09-25", 1003.27002],\n    ["2003-09-26", 996.849976],\n    ["2003-09-29", 1006.580017],\n    ["2003-09-30", 995.969971],\n    ["2003-10-01", 1018.219971],\n    ["2003-10-02", 1020.23999],\n    ["2003-10-03", 1029.849976],\n    ["2003-10-06", 1034.349976],\n    ["2003-10-07", 1039.25],\n    ["2003-10-08", 1033.780029],\n    ["2003-10-09", 1038.72998],\n    ["2003-10-10", 1038.060059],\n    ["2003-10-13", 1045.349976],\n    ["2003-10-14", 1049.47998],\n    ["2003-10-15", 1046.76001],\n    ["2003-10-16", 1050.069946],\n    ["2003-10-17", 1039.319946],\n    ["2003-10-20", 1044.680054],\n    ["2003-10-21", 1046.030029],\n    ["2003-10-22", 1030.359985],\n    ["2003-10-23", 1033.77002],\n    ["2003-10-24", 1028.910034],\n    ["2003-10-27", 1031.130005],\n    ["2003-10-28", 1046.790039],\n    ["2003-10-29", 1048.109985],\n    ["2003-10-30", 1046.939941],\n    ["2003-10-31", 1050.709961],\n    ["2003-11-03", 1059.02002],\n    ["2003-11-04", 1053.25],\n    ["2003-11-05", 1051.810059],\n    ["2003-11-06", 1058.050049],\n    ["2003-11-07", 1053.209961],\n    ["2003-11-10", 1047.109985],\n    ["2003-11-11", 1046.569946],\n    ["2003-11-12", 1058.530029],\n    ["2003-11-13", 1058.410034],\n    ["2003-11-14", 1050.349976],\n    ["2003-11-17", 1043.630005],\n    ["2003-11-18", 1034.150024],\n    ["2003-11-19", 1042.439941],\n    ["2003-11-20", 1033.650024],\n    ["2003-11-21", 1035.280029],\n    ["2003-11-24", 1052.079956],\n    ["2003-11-25", 1053.890015],\n    ["2003-11-26", 1058.449951],\n    ["2003-11-28", 1058.199951],\n    ["2003-12-01", 1070.119995],\n    ["2003-12-02", 1066.619995],\n    ["2003-12-03", 1064.72998],\n    ["2003-12-04", 1069.719971],\n    ["2003-12-05", 1061.5],\n    ["2003-12-08", 1069.300049],\n    ["2003-12-09", 1060.180054],\n    ["2003-12-10", 1059.050049],\n    ["2003-12-11", 1071.209961],\n    ["2003-12-12", 1074.140015],\n    ["2003-12-15", 1068.040039],\n    ["2003-12-16", 1075.130005],\n    ["2003-12-17", 1076.47998],\n    ["2003-12-18", 1089.180054],\n    ["2003-12-19", 1088.660034],\n    ["2003-12-22", 1092.939941],\n    ["2003-12-23", 1096.02002],\n    ["2003-12-24", 1094.040039],\n    ["2003-12-26", 1095.890015],\n    ["2003-12-29", 1109.47998],\n    ["2003-12-30", 1109.640015],\n    ["2003-12-31", 1111.920044],\n    ["2004-01-02", 1108.47998],\n    ["2004-01-05", 1122.219971],\n    ["2004-01-06", 1123.670044],\n    ["2004-01-07", 1126.329956],\n    ["2004-01-08", 1131.920044],\n    ["2004-01-09", 1121.859985],\n    ["2004-01-12", 1127.22998],\n    ["2004-01-13", 1121.219971],\n    ["2004-01-14", 1130.52002],\n    ["2004-01-15", 1132.050049],\n    ["2004-01-16", 1139.829956],\n    ["2004-01-20", 1138.77002],\n    ["2004-01-21", 1147.619995],\n    ["2004-01-22", 1143.939941],\n    ["2004-01-23", 1141.550049],\n    ["2004-01-26", 1155.369995],\n    ["2004-01-27", 1144.050049],\n    ["2004-01-28", 1128.47998],\n    ["2004-01-29", 1134.109985],\n    ["2004-01-30", 1131.130005],\n    ["2004-02-02", 1135.26001],\n    ["2004-02-03", 1136.030029],\n    ["2004-02-04", 1126.52002],\n    ["2004-02-05", 1128.589966],\n    ["2004-02-06", 1142.76001],\n    ["2004-02-09", 1139.810059],\n    ["2004-02-10", 1145.540039],\n    ["2004-02-11", 1157.76001],\n    ["2004-02-12", 1152.109985],\n    ["2004-02-13", 1145.810059],\n    ["2004-02-17", 1156.98999],\n    ["2004-02-18", 1151.819946],\n    ["2004-02-19", 1147.060059],\n    ["2004-02-20", 1144.109985],\n    ["2004-02-23", 1140.98999],\n    ["2004-02-24", 1139.089966],\n    ["2004-02-25", 1143.670044],\n    ["2004-02-26", 1144.910034],\n    ["2004-02-27", 1144.939941],\n    ["2004-03-01", 1155.969971],\n    ["2004-03-02", 1149.099976],\n    ["2004-03-03", 1151.030029],\n    ["2004-03-04", 1154.869995],\n    ["2004-03-05", 1156.859985],\n    ["2004-03-08", 1147.199951],\n    ["2004-03-09", 1140.579956],\n    ["2004-03-10", 1123.890015],\n    ["2004-03-11", 1106.780029],\n    ["2004-03-12", 1120.569946],\n    ["2004-03-15", 1104.48999],\n    ["2004-03-16", 1110.699951],\n    ["2004-03-17", 1123.75],\n    ["2004-03-18", 1122.319946],\n    ["2004-03-19", 1109.780029],\n    ["2004-03-22", 1095.400024],\n    ["2004-03-23", 1093.949951],\n    ["2004-03-24", 1091.329956],\n    ["2004-03-25", 1109.189941],\n    ["2004-03-26", 1108.060059],\n    ["2004-03-29", 1122.469971],\n    ["2004-03-30", 1127.0],\n    ["2004-03-31", 1126.209961],\n    ["2004-04-01", 1132.170044],\n    ["2004-04-02", 1141.810059],\n    ["2004-04-05", 1150.569946],\n    ["2004-04-06", 1148.160034],\n    ["2004-04-07", 1140.530029],\n    ["2004-04-08", 1139.319946],\n    ["2004-04-12", 1145.199951],\n    ["2004-04-13", 1129.439941],\n    ["2004-04-14", 1128.170044],\n    ["2004-04-15", 1128.839966],\n    ["2004-04-16", 1134.609985],\n    ["2004-04-19", 1135.819946],\n    ["2004-04-20", 1118.150024],\n    ["2004-04-21", 1124.089966],\n    ["2004-04-22", 1139.930054],\n    ["2004-04-23", 1140.599976],\n    ["2004-04-26", 1135.530029],\n    ["2004-04-27", 1138.109985],\n    ["2004-04-28", 1122.410034],\n    ["2004-04-29", 1113.890015],\n    ["2004-04-30", 1107.300049],\n    ["2004-05-03", 1117.48999],\n    ["2004-05-04", 1119.550049],\n    ["2004-05-05", 1121.530029],\n    ["2004-05-06", 1113.98999],\n    ["2004-05-07", 1098.699951],\n    ["2004-05-10", 1087.119995],\n    ["2004-05-11", 1095.449951],\n    ["2004-05-12", 1097.280029],\n    ["2004-05-13", 1096.439941],\n    ["2004-05-14", 1095.699951],\n    ["2004-05-17", 1084.099976],\n    ["2004-05-18", 1091.48999],\n    ["2004-05-19", 1088.680054],\n    ["2004-05-20", 1089.189941],\n    ["2004-05-21", 1093.560059],\n    ["2004-05-24", 1095.410034],\n    ["2004-05-25", 1113.050049],\n    ["2004-05-26", 1114.939941],\n    ["2004-05-27", 1121.280029],\n    ["2004-05-28", 1120.680054],\n    ["2004-06-01", 1121.199951],\n    ["2004-06-02", 1124.98999],\n    ["2004-06-03", 1116.640015],\n    ["2004-06-04", 1122.5],\n    ["2004-06-07", 1140.420044],\n    ["2004-06-08", 1142.180054],\n    ["2004-06-09", 1131.329956],\n    ["2004-06-10", 1136.469971],\n    ["2004-06-14", 1125.290039],\n    ["2004-06-15", 1132.01001],\n    ["2004-06-16", 1133.560059],\n    ["2004-06-17", 1132.050049],\n    ["2004-06-18", 1135.02002],\n    ["2004-06-21", 1130.300049],\n    ["2004-06-22", 1134.410034],\n    ["2004-06-23", 1144.060059],\n    ["2004-06-24", 1140.650024],\n    ["2004-06-25", 1134.430054],\n    ["2004-06-28", 1133.349976],\n    ["2004-06-29", 1136.199951],\n    ["2004-06-30", 1140.839966],\n    ["2004-07-01", 1128.939941],\n    ["2004-07-02", 1125.380005],\n    ["2004-07-06", 1116.209961],\n    ["2004-07-07", 1118.329956],\n    ["2004-07-08", 1109.109985],\n    ["2004-07-09", 1112.810059],\n    ["2004-07-12", 1114.349976],\n    ["2004-07-13", 1115.140015],\n    ["2004-07-14", 1111.469971],\n    ["2004-07-15", 1106.689941],\n    ["2004-07-16", 1101.390015],\n    ["2004-07-19", 1100.900024],\n    ["2004-07-20", 1108.670044],\n    ["2004-07-21", 1093.880005],\n    ["2004-07-22", 1096.839966],\n    ["2004-07-23", 1086.199951],\n    ["2004-07-26", 1084.069946],\n    ["2004-07-27", 1094.829956],\n    ["2004-07-28", 1095.420044],\n    ["2004-07-29", 1100.430054],\n    ["2004-07-30", 1101.719971],\n    ["2004-08-02", 1106.619995],\n    ["2004-08-03", 1099.689941],\n    ["2004-08-04", 1098.630005],\n    ["2004-08-05", 1080.699951],\n    ["2004-08-06", 1063.969971],\n    ["2004-08-09", 1065.219971],\n    ["2004-08-10", 1079.040039],\n    ["2004-08-11", 1075.790039],\n    ["2004-08-12", 1063.22998],\n    ["2004-08-13", 1064.800049],\n    ["2004-08-16", 1079.339966],\n    ["2004-08-17", 1081.709961],\n    ["2004-08-18", 1095.170044],\n    ["2004-08-19", 1091.22998],\n    ["2004-08-20", 1098.349976],\n    ["2004-08-23", 1095.680054],\n    ["2004-08-24", 1096.189941],\n    ["2004-08-25", 1104.959961],\n    ["2004-08-26", 1105.089966],\n    ["2004-08-27", 1107.77002],\n    ["2004-08-30", 1099.150024],\n    ["2004-08-31", 1104.23999],\n    ["2004-09-01", 1105.910034],\n    ["2004-09-02", 1118.310059],\n    ["2004-09-03", 1113.630005],\n    ["2004-09-07", 1121.300049],\n    ["2004-09-08", 1116.27002],\n    ["2004-09-09", 1118.380005],\n    ["2004-09-10", 1123.920044],\n    ["2004-09-13", 1125.819946],\n    ["2004-09-14", 1128.329956],\n    ["2004-09-15", 1120.369995],\n    ["2004-09-16", 1123.5],\n    ["2004-09-17", 1128.550049],\n    ["2004-09-20", 1122.199951],\n    ["2004-09-21", 1129.300049],\n    ["2004-09-22", 1113.560059],\n    ["2004-09-23", 1108.359985],\n    ["2004-09-24", 1110.109985],\n    ["2004-09-27", 1103.52002],\n    ["2004-09-28", 1110.060059],\n    ["2004-09-29", 1114.800049],\n    ["2004-09-30", 1114.579956],\n    ["2004-10-01", 1131.5],\n    ["2004-10-04", 1135.170044],\n    ["2004-10-05", 1134.47998],\n    ["2004-10-06", 1142.050049],\n    ["2004-10-07", 1130.650024],\n    ["2004-10-08", 1122.140015],\n    ["2004-10-11", 1124.390015],\n    ["2004-10-12", 1121.839966],\n    ["2004-10-13", 1113.650024],\n    ["2004-10-14", 1103.290039],\n    ["2004-10-15", 1108.199951],\n    ["2004-10-18", 1114.02002],\n    ["2004-10-19", 1103.22998],\n    ["2004-10-20", 1103.660034],\n    ["2004-10-21", 1106.48999],\n    ["2004-10-22", 1095.73999],\n    ["2004-10-25", 1094.800049],\n    ["2004-10-26", 1111.089966],\n    ["2004-10-27", 1125.400024],\n    ["2004-10-28", 1127.439941],\n    ["2004-10-29", 1130.199951],\n    ["2004-11-01", 1130.51001],\n    ["2004-11-02", 1130.560059],\n    ["2004-11-03", 1143.199951],\n    ["2004-11-04", 1161.670044],\n    ["2004-11-05", 1166.170044],\n    ["2004-11-08", 1164.890015],\n    ["2004-11-09", 1164.079956],\n    ["2004-11-10", 1162.910034],\n    ["2004-11-11", 1173.47998],\n    ["2004-11-12", 1184.170044],\n    ["2004-11-15", 1183.810059],\n    ["2004-11-16", 1175.430054],\n    ["2004-11-17", 1181.939941],\n    ["2004-11-18", 1183.550049],\n    ["2004-11-19", 1170.339966],\n    ["2004-11-22", 1177.23999],\n    ["2004-11-23", 1176.939941],\n    ["2004-11-24", 1181.76001],\n    ["2004-11-26", 1182.650024],\n    ["2004-11-29", 1178.569946],\n    ["2004-11-30", 1173.819946],\n    ["2004-12-01", 1191.369995],\n    ["2004-12-02", 1190.329956],\n    ["2004-12-03", 1191.170044],\n    ["2004-12-06", 1190.25],\n    ["2004-12-07", 1177.069946],\n    ["2004-12-08", 1182.810059],\n    ["2004-12-09", 1189.23999],\n    ["2004-12-10", 1188.0],\n    ["2004-12-13", 1198.680054],\n    ["2004-12-14", 1203.380005],\n    ["2004-12-15", 1205.719971],\n    ["2004-12-16", 1203.209961],\n    ["2004-12-17", 1194.199951],\n    ["2004-12-20", 1194.650024],\n    ["2004-12-21", 1205.449951],\n    ["2004-12-22", 1209.569946],\n    ["2004-12-23", 1210.130005],\n    ["2004-12-27", 1204.920044],\n    ["2004-12-28", 1213.540039],\n    ["2004-12-29", 1213.449951],\n    ["2004-12-30", 1213.550049],\n    ["2004-12-31", 1211.920044],\n    ["2005-01-03", 1202.079956],\n    ["2005-01-04", 1188.050049],\n    ["2005-01-05", 1183.73999],\n    ["2005-01-06", 1187.890015],\n    ["2005-01-07", 1186.189941],\n    ["2005-01-10", 1190.25],\n    ["2005-01-11", 1182.98999],\n    ["2005-01-12", 1187.699951],\n    ["2005-01-13", 1177.449951],\n    ["2005-01-14", 1184.52002],\n    ["2005-01-18", 1195.97998],\n    ["2005-01-19", 1184.630005],\n    ["2005-01-20", 1175.410034],\n    ["2005-01-21", 1167.869995],\n    ["2005-01-24", 1163.75],\n    ["2005-01-25", 1168.410034],\n    ["2005-01-26", 1174.069946],\n    ["2005-01-27", 1174.550049],\n    ["2005-01-28", 1171.359985],\n    ["2005-01-31", 1181.27002],\n    ["2005-02-01", 1189.410034],\n    ["2005-02-02", 1193.189941],\n    ["2005-02-03", 1189.890015],\n    ["2005-02-04", 1203.030029],\n    ["2005-02-07", 1201.719971],\n    ["2005-02-08", 1202.300049],\n    ["2005-02-09", 1191.98999],\n    ["2005-02-10", 1197.01001],\n    ["2005-02-11", 1205.300049],\n    ["2005-02-14", 1206.140015],\n    ["2005-02-15", 1210.119995],\n    ["2005-02-16", 1210.339966],\n    ["2005-02-17", 1200.75],\n    ["2005-02-18", 1201.589966],\n    ["2005-02-22", 1184.160034],\n    ["2005-02-23", 1190.800049],\n    ["2005-02-24", 1200.199951],\n    ["2005-02-25", 1211.369995],\n    ["2005-02-28", 1203.599976],\n    ["2005-03-01", 1210.410034],\n    ["2005-03-02", 1210.079956],\n    ["2005-03-03", 1210.469971],\n    ["2005-03-04", 1222.119995],\n    ["2005-03-07", 1225.310059],\n    ["2005-03-08", 1219.430054],\n    ["2005-03-09", 1207.01001],\n    ["2005-03-10", 1209.25],\n    ["2005-03-11", 1200.079956],\n    ["2005-03-14", 1206.829956],\n    ["2005-03-15", 1197.75],\n    ["2005-03-16", 1188.069946],\n    ["2005-03-17", 1190.209961],\n    ["2005-03-18", 1189.650024],\n    ["2005-03-21", 1183.780029],\n    ["2005-03-22", 1171.709961],\n    ["2005-03-23", 1172.530029],\n    ["2005-03-24", 1171.420044],\n    ["2005-03-28", 1174.280029],\n    ["2005-03-29", 1165.359985],\n    ["2005-03-30", 1181.410034],\n    ["2005-03-31", 1180.589966],\n    ["2005-04-01", 1172.920044],\n    ["2005-04-04", 1176.119995],\n    ["2005-04-05", 1181.390015],\n    ["2005-04-06", 1184.069946],\n    ["2005-04-07", 1191.140015],\n    ["2005-04-08", 1181.199951],\n    ["2005-04-11", 1181.209961],\n    ["2005-04-12", 1187.76001],\n    ["2005-04-13", 1173.790039],\n    ["2005-04-14", 1162.050049],\n    ["2005-04-15", 1142.619995],\n    ["2005-04-18", 1145.97998],\n    ["2005-04-19", 1152.780029],\n    ["2005-04-20", 1137.5],\n    ["2005-04-21", 1159.949951],\n    ["2005-04-22", 1152.119995],\n    ["2005-04-25", 1162.099976],\n    ["2005-04-26", 1151.829956],\n    ["2005-04-27", 1156.380005],\n    ["2005-04-28", 1143.219971],\n    ["2005-04-29", 1156.849976],\n    ["2005-05-02", 1162.160034],\n    ["2005-05-03", 1161.170044],\n    ["2005-05-04", 1175.650024],\n    ["2005-05-05", 1172.630005],\n    ["2005-05-06", 1171.349976],\n    ["2005-05-09", 1178.839966],\n    ["2005-05-10", 1166.219971],\n    ["2005-05-11", 1171.109985],\n    ["2005-05-12", 1159.359985],\n    ["2005-05-13", 1154.050049],\n    ["2005-05-16", 1165.689941],\n    ["2005-05-17", 1173.800049],\n    ["2005-05-18", 1185.560059],\n    ["2005-05-19", 1191.079956],\n    ["2005-05-20", 1189.280029],\n    ["2005-05-23", 1193.859985],\n    ["2005-05-24", 1194.069946],\n    ["2005-05-25", 1190.01001],\n    ["2005-05-26", 1197.619995],\n    ["2005-05-27", 1198.780029],\n    ["2005-05-31", 1191.5],\n    ["2005-06-01", 1202.219971],\n    ["2005-06-02", 1204.290039],\n    ["2005-06-03", 1196.02002],\n    ["2005-06-06", 1197.51001],\n    ["2005-06-07", 1197.26001],\n    ["2005-06-08", 1194.670044],\n    ["2005-06-09", 1200.930054],\n    ["2005-06-10", 1198.109985],\n    ["2005-06-13", 1200.819946],\n    ["2005-06-14", 1203.910034],\n    ["2005-06-15", 1206.579956],\n    ["2005-06-16", 1210.959961],\n    ["2005-06-17", 1216.959961],\n    ["2005-06-20", 1216.099976],\n    ["2005-06-21", 1213.609985],\n    ["2005-06-22", 1213.880005],\n    ["2005-06-23", 1200.72998],\n    ["2005-06-24", 1191.569946],\n    ["2005-06-27", 1190.689941],\n    ["2005-06-28", 1201.569946],\n    ["2005-06-29", 1199.849976],\n    ["2005-06-30", 1191.329956],\n    ["2005-07-01", 1194.439941],\n    ["2005-07-05", 1204.98999],\n    ["2005-07-06", 1194.939941],\n    ["2005-07-07", 1197.869995],\n    ["2005-07-08", 1211.859985],\n    ["2005-07-11", 1219.439941],\n    ["2005-07-12", 1222.209961],\n    ["2005-07-13", 1223.290039],\n    ["2005-07-14", 1226.5],\n    ["2005-07-15", 1227.920044],\n    ["2005-07-18", 1221.130005],\n    ["2005-07-19", 1229.349976],\n    ["2005-07-20", 1235.199951],\n    ["2005-07-21", 1227.040039],\n    ["2005-07-22", 1233.680054],\n    ["2005-07-25", 1229.030029],\n    ["2005-07-26", 1231.160034],\n    ["2005-07-27", 1236.790039],\n    ["2005-07-28", 1243.719971],\n    ["2005-07-29", 1234.180054],\n    ["2005-08-01", 1235.349976],\n    ["2005-08-02", 1244.119995],\n    ["2005-08-03", 1245.040039],\n    ["2005-08-04", 1235.859985],\n    ["2005-08-05", 1226.420044],\n    ["2005-08-08", 1223.130005],\n    ["2005-08-09", 1231.380005],\n    ["2005-08-10", 1229.130005],\n    ["2005-08-11", 1237.810059],\n    ["2005-08-12", 1230.390015],\n    ["2005-08-15", 1233.869995],\n    ["2005-08-16", 1219.339966],\n    ["2005-08-17", 1220.23999],\n    ["2005-08-18", 1219.02002],\n    ["2005-08-19", 1219.709961],\n    ["2005-08-22", 1221.72998],\n    ["2005-08-23", 1217.589966],\n    ["2005-08-24", 1209.589966],\n    ["2005-08-25", 1212.369995],\n    ["2005-08-26", 1205.099976],\n    ["2005-08-29", 1212.280029],\n    ["2005-08-30", 1208.410034],\n    ["2005-08-31", 1220.329956],\n    ["2005-09-01", 1221.589966],\n    ["2005-09-02", 1218.02002],\n    ["2005-09-06", 1233.390015],\n    ["2005-09-07", 1236.359985],\n    ["2005-09-08", 1231.670044],\n    ["2005-09-09", 1241.47998],\n    ["2005-09-12", 1240.560059],\n    ["2005-09-13", 1231.199951],\n    ["2005-09-14", 1227.160034],\n    ["2005-09-15", 1227.72998],\n    ["2005-09-16", 1237.910034],\n    ["2005-09-19", 1231.02002],\n    ["2005-09-20", 1221.339966],\n    ["2005-09-21", 1210.199951],\n    ["2005-09-22", 1214.619995],\n    ["2005-09-23", 1215.290039],\n    ["2005-09-26", 1215.630005],\n    ["2005-09-27", 1215.660034],\n    ["2005-09-28", 1216.890015],\n    ["2005-09-29", 1227.680054],\n    ["2005-09-30", 1228.810059],\n    ["2005-10-03", 1226.699951],\n    ["2005-10-04", 1214.469971],\n    ["2005-10-05", 1196.390015],\n    ["2005-10-06", 1191.48999],\n    ["2005-10-07", 1195.900024],\n    ["2005-10-10", 1187.329956],\n    ["2005-10-11", 1184.869995],\n    ["2005-10-12", 1177.680054],\n    ["2005-10-13", 1176.839966],\n    ["2005-10-14", 1186.569946],\n    ["2005-10-17", 1190.099976],\n    ["2005-10-18", 1178.140015],\n    ["2005-10-19", 1195.76001],\n    ["2005-10-20", 1177.800049],\n    ["2005-10-21", 1179.589966],\n    ["2005-10-24", 1199.380005],\n    ["2005-10-25", 1196.540039],\n    ["2005-10-26", 1191.380005],\n    ["2005-10-27", 1178.900024],\n    ["2005-10-28", 1198.410034],\n    ["2005-10-31", 1207.01001],\n    ["2005-11-01", 1202.76001],\n    ["2005-11-02", 1214.76001],\n    ["2005-11-03", 1219.939941],\n    ["2005-11-04", 1220.140015],\n    ["2005-11-07", 1222.810059],\n    ["2005-11-08", 1218.589966],\n    ["2005-11-09", 1220.650024],\n    ["2005-11-10", 1230.959961],\n    ["2005-11-11", 1234.719971],\n    ["2005-11-14", 1233.76001],\n    ["2005-11-15", 1229.01001],\n    ["2005-11-16", 1231.209961],\n    ["2005-11-17", 1242.800049],\n    ["2005-11-18", 1248.27002],\n    ["2005-11-21", 1254.849976],\n    ["2005-11-22", 1261.22998],\n    ["2005-11-23", 1265.609985],\n    ["2005-11-25", 1268.25],\n    ["2005-11-28", 1257.459961],\n    ["2005-11-29", 1257.47998],\n    ["2005-11-30", 1249.47998],\n    ["2005-12-01", 1264.670044],\n    ["2005-12-02", 1265.079956],\n    ["2005-12-05", 1262.089966],\n    ["2005-12-06", 1263.699951],\n    ["2005-12-07", 1257.369995],\n    ["2005-12-08", 1255.839966],\n    ["2005-12-09", 1259.369995],\n    ["2005-12-12", 1260.430054],\n    ["2005-12-13", 1267.430054],\n    ["2005-12-14", 1272.73999],\n    ["2005-12-15", 1270.939941],\n    ["2005-12-16", 1267.319946],\n    ["2005-12-19", 1259.920044],\n    ["2005-12-20", 1259.619995],\n    ["2005-12-21", 1262.790039],\n    ["2005-12-22", 1268.119995],\n    ["2005-12-23", 1268.660034],\n    ["2005-12-27", 1256.540039],\n    ["2005-12-28", 1258.170044],\n    ["2005-12-29", 1254.420044],\n    ["2005-12-30", 1248.290039],\n    ["2006-01-03", 1268.800049],\n    ["2006-01-04", 1273.459961],\n    ["2006-01-05", 1273.47998],\n    ["2006-01-06", 1285.449951],\n    ["2006-01-09", 1290.150024],\n    ["2006-01-10", 1289.689941],\n    ["2006-01-11", 1294.180054],\n    ["2006-01-12", 1286.060059],\n    ["2006-01-13", 1287.609985],\n    ["2006-01-17", 1282.930054],\n    ["2006-01-18", 1277.930054],\n    ["2006-01-19", 1285.040039],\n    ["2006-01-20", 1261.48999],\n    ["2006-01-23", 1263.819946],\n    ["2006-01-24", 1266.859985],\n    ["2006-01-25", 1264.680054],\n    ["2006-01-26", 1273.829956],\n    ["2006-01-27", 1283.719971],\n    ["2006-01-30", 1285.189941],\n    ["2006-01-31", 1280.079956],\n    ["2006-02-01", 1282.459961],\n    ["2006-02-02", 1270.839966],\n    ["2006-02-03", 1264.030029],\n    ["2006-02-06", 1265.02002],\n    ["2006-02-07", 1254.780029],\n    ["2006-02-08", 1265.650024],\n    ["2006-02-09", 1263.780029],\n    ["2006-02-10", 1266.98999],\n    ["2006-02-13", 1262.859985],\n    ["2006-02-14", 1275.530029],\n    ["2006-02-15", 1280.0],\n    ["2006-02-16", 1289.380005],\n    ["2006-02-17", 1287.23999],\n    ["2006-02-21", 1283.030029],\n    ["2006-02-22", 1292.670044],\n    ["2006-02-23", 1287.790039],\n    ["2006-02-24", 1289.430054],\n    ["2006-02-27", 1294.119995],\n    ["2006-02-28", 1280.660034],\n    ["2006-03-01", 1291.23999],\n    ["2006-03-02", 1289.140015],\n    ["2006-03-03", 1287.22998],\n    ["2006-03-06", 1278.26001],\n    ["2006-03-07", 1275.880005],\n    ["2006-03-08", 1278.469971],\n    ["2006-03-09", 1272.22998],\n    ["2006-03-10", 1281.420044],\n    ["2006-03-13", 1284.130005],\n    ["2006-03-14", 1297.47998],\n    ["2006-03-15", 1303.02002],\n    ["2006-03-16", 1305.329956],\n    ["2006-03-17", 1307.25],\n    ["2006-03-20", 1305.079956],\n    ["2006-03-21", 1297.22998],\n    ["2006-03-22", 1305.040039],\n    ["2006-03-23", 1301.670044],\n    ["2006-03-24", 1302.949951],\n    ["2006-03-27", 1301.609985],\n    ["2006-03-28", 1293.22998],\n    ["2006-03-29", 1302.890015],\n    ["2006-03-30", 1300.25],\n    ["2006-03-31", 1294.869995],\n    ["2006-04-03", 1297.810059],\n    ["2006-04-04", 1305.930054],\n    ["2006-04-05", 1311.560059],\n    ["2006-04-06", 1309.040039],\n    ["2006-04-07", 1295.5],\n    ["2006-04-10", 1296.619995],\n    ["2006-04-11", 1286.569946],\n    ["2006-04-12", 1288.119995],\n    ["2006-04-13", 1289.119995],\n    ["2006-04-17", 1285.329956],\n    ["2006-04-18", 1307.280029],\n    ["2006-04-19", 1309.930054],\n    ["2006-04-20", 1311.459961],\n    ["2006-04-21", 1311.280029],\n    ["2006-04-24", 1308.109985],\n    ["2006-04-25", 1301.73999],\n    ["2006-04-26", 1305.410034],\n    ["2006-04-27", 1309.719971],\n    ["2006-04-28", 1310.609985],\n    ["2006-05-01", 1305.189941],\n    ["2006-05-02", 1313.209961],\n    ["2006-05-03", 1308.119995],\n    ["2006-05-04", 1312.25],\n    ["2006-05-05", 1325.76001],\n    ["2006-05-08", 1324.660034],\n    ["2006-05-09", 1325.140015],\n    ["2006-05-10", 1322.849976],\n    ["2006-05-11", 1305.920044],\n    ["2006-05-12", 1291.23999],\n    ["2006-05-15", 1294.5],\n    ["2006-05-16", 1292.079956],\n    ["2006-05-17", 1270.319946],\n    ["2006-05-18", 1261.810059],\n    ["2006-05-19", 1267.030029],\n    ["2006-05-22", 1262.069946],\n    ["2006-05-23", 1256.579956],\n    ["2006-05-24", 1258.569946],\n    ["2006-05-25", 1272.880005],\n    ["2006-05-26", 1280.160034],\n    ["2006-05-30", 1259.869995],\n    ["2006-05-31", 1270.089966],\n    ["2006-06-01", 1285.709961],\n    ["2006-06-02", 1288.219971],\n    ["2006-06-05", 1265.290039],\n    ["2006-06-06", 1263.849976],\n    ["2006-06-07", 1256.150024],\n    ["2006-06-08", 1257.930054],\n    ["2006-06-09", 1252.300049],\n    ["2006-06-12", 1237.439941],\n    ["2006-06-13", 1223.689941],\n    ["2006-06-14", 1230.040039],\n    ["2006-06-15", 1256.160034],\n    ["2006-06-16", 1251.540039],\n    ["2006-06-19", 1240.130005],\n    ["2006-06-20", 1240.119995],\n    ["2006-06-21", 1252.199951],\n    ["2006-06-22", 1245.599976],\n    ["2006-06-23", 1244.5],\n    ["2006-06-26", 1250.560059],\n    ["2006-06-27", 1239.199951],\n    ["2006-06-28", 1246.0],\n    ["2006-06-29", 1272.869995],\n    ["2006-06-30", 1270.199951],\n    ["2006-07-03", 1280.189941],\n    ["2006-07-05", 1270.910034],\n    ["2006-07-06", 1274.079956],\n    ["2006-07-07", 1265.47998],\n    ["2006-07-10", 1267.339966],\n    ["2006-07-11", 1272.430054],\n    ["2006-07-12", 1258.599976],\n    ["2006-07-13", 1242.280029],\n    ["2006-07-14", 1236.199951],\n    ["2006-07-17", 1234.48999],\n    ["2006-07-18", 1236.859985],\n    ["2006-07-19", 1259.810059],\n    ["2006-07-20", 1249.130005],\n    ["2006-07-21", 1240.290039],\n    ["2006-07-24", 1260.910034],\n    ["2006-07-25", 1268.880005],\n    ["2006-07-26", 1268.400024],\n    ["2006-07-27", 1263.199951],\n    ["2006-07-28", 1278.550049],\n    ["2006-07-31", 1276.660034],\n    ["2006-08-01", 1270.920044],\n    ["2006-08-02", 1277.410034],\n    ["2006-08-03", 1280.27002],\n    ["2006-08-04", 1279.359985],\n    ["2006-08-07", 1275.77002],\n    ["2006-08-08", 1271.47998],\n    ["2006-08-09", 1265.949951],\n    ["2006-08-10", 1271.810059],\n    ["2006-08-11", 1266.73999],\n    ["2006-08-14", 1268.209961],\n    ["2006-08-15", 1285.579956],\n    ["2006-08-16", 1295.430054],\n    ["2006-08-17", 1297.47998],\n    ["2006-08-18", 1302.300049],\n    ["2006-08-21", 1297.52002],\n    ["2006-08-22", 1298.819946],\n    ["2006-08-23", 1292.98999],\n    ["2006-08-24", 1296.060059],\n    ["2006-08-25", 1295.089966],\n    ["2006-08-28", 1301.780029],\n    ["2006-08-29", 1304.280029],\n    ["2006-08-30", 1305.369995],\n    ["2006-08-31", 1303.819946],\n    ["2006-09-01", 1311.01001],\n    ["2006-09-05", 1313.25],\n    ["2006-09-06", 1300.26001],\n    ["2006-09-07", 1294.02002],\n    ["2006-09-08", 1298.920044],\n    ["2006-09-11", 1299.540039],\n    ["2006-09-12", 1313.0],\n    ["2006-09-13", 1318.069946],\n    ["2006-09-14", 1316.280029],\n    ["2006-09-15", 1319.660034],\n    ["2006-09-18", 1321.180054],\n    ["2006-09-19", 1317.640015],\n    ["2006-09-20", 1325.180054],\n    ["2006-09-21", 1318.030029],\n    ["2006-09-22", 1314.780029],\n    ["2006-09-25", 1326.369995],\n    ["2006-09-26", 1336.349976],\n    ["2006-09-27", 1336.589966],\n    ["2006-09-28", 1338.880005],\n    ["2006-09-29", 1335.849976],\n    ["2006-10-02", 1331.319946],\n    ["2006-10-03", 1334.109985],\n    ["2006-10-04", 1350.199951],\n    ["2006-10-05", 1353.219971],\n    ["2006-10-06", 1349.589966],\n    ["2006-10-09", 1350.660034],\n    ["2006-10-10", 1353.420044],\n    ["2006-10-11", 1349.949951],\n    ["2006-10-12", 1362.829956],\n    ["2006-10-13", 1365.619995],\n    ["2006-10-16", 1369.060059],\n    ["2006-10-17", 1364.050049],\n    ["2006-10-18", 1365.800049],\n    ["2006-10-19", 1366.959961],\n    ["2006-10-20", 1368.599976],\n    ["2006-10-23", 1377.02002],\n    ["2006-10-24", 1377.380005],\n    ["2006-10-25", 1382.219971],\n    ["2006-10-26", 1389.079956],\n    ["2006-10-27", 1377.339966],\n    ["2006-10-30", 1377.930054],\n    ["2006-10-31", 1377.939941],\n    ["2006-11-01", 1367.810059],\n    ["2006-11-02", 1367.339966],\n    ["2006-11-03", 1364.300049],\n    ["2006-11-06", 1379.780029],\n    ["2006-11-07", 1382.839966],\n    ["2006-11-08", 1385.719971],\n    ["2006-11-09", 1378.329956],\n    ["2006-11-10", 1380.900024],\n    ["2006-11-13", 1384.420044],\n    ["2006-11-14", 1393.219971],\n    ["2006-11-15", 1396.569946],\n    ["2006-11-16", 1399.76001],\n    ["2006-11-17", 1401.199951],\n    ["2006-11-20", 1400.5],\n    ["2006-11-21", 1402.810059],\n    ["2006-11-22", 1406.089966],\n    ["2006-11-24", 1400.949951],\n    ["2006-11-27", 1381.959961],\n    ["2006-11-28", 1386.719971],\n    ["2006-11-29", 1399.47998],\n    ["2006-11-30", 1400.630005],\n    ["2006-12-01", 1396.709961],\n    ["2006-12-04", 1409.119995],\n    ["2006-12-05", 1414.76001],\n    ["2006-12-06", 1412.900024],\n    ["2006-12-07", 1407.290039],\n    ["2006-12-08", 1409.839966],\n    ["2006-12-11", 1413.040039],\n    ["2006-12-12", 1411.560059],\n    ["2006-12-13", 1413.209961],\n    ["2006-12-14", 1425.48999],\n    ["2006-12-15", 1427.089966],\n    ["2006-12-18", 1422.47998],\n    ["2006-12-19", 1425.550049],\n    ["2006-12-20", 1423.530029],\n    ["2006-12-21", 1418.300049],\n    ["2006-12-22", 1410.76001],\n    ["2006-12-26", 1416.900024],\n    ["2006-12-27", 1426.839966],\n    ["2006-12-28", 1424.72998],\n    ["2006-12-29", 1418.300049],\n    ["2007-01-03", 1416.599976],\n    ["2007-01-04", 1418.339966],\n    ["2007-01-05", 1409.709961],\n    ["2007-01-08", 1412.839966],\n    ["2007-01-09", 1412.109985],\n    ["2007-01-10", 1414.849976],\n    ["2007-01-11", 1423.819946],\n    ["2007-01-12", 1430.72998],\n    ["2007-01-16", 1431.900024],\n    ["2007-01-17", 1430.619995],\n    ["2007-01-18", 1426.369995],\n    ["2007-01-19", 1430.5],\n    ["2007-01-22", 1422.949951],\n    ["2007-01-23", 1427.98999],\n    ["2007-01-24", 1440.130005],\n    ["2007-01-25", 1423.900024],\n    ["2007-01-26", 1422.180054],\n    ["2007-01-29", 1420.619995],\n    ["2007-01-30", 1428.819946],\n    ["2007-01-31", 1438.23999],\n    ["2007-02-01", 1445.939941],\n    ["2007-02-02", 1448.390015],\n    ["2007-02-05", 1446.98999],\n    ["2007-02-06", 1448.0],\n    ["2007-02-07", 1450.02002],\n    ["2007-02-08", 1448.310059],\n    ["2007-02-09", 1438.060059],\n    ["2007-02-12", 1433.369995],\n    ["2007-02-13", 1444.26001],\n    ["2007-02-14", 1455.300049],\n    ["2007-02-15", 1456.810059],\n    ["2007-02-16", 1455.540039],\n    ["2007-02-20", 1459.680054],\n    ["2007-02-21", 1457.630005],\n    ["2007-02-22", 1456.380005],\n    ["2007-02-23", 1451.189941],\n    ["2007-02-26", 1449.369995],\n    ["2007-02-27", 1399.040039],\n    ["2007-02-28", 1406.819946],\n    ["2007-03-01", 1403.170044],\n    ["2007-03-02", 1387.170044],\n    ["2007-03-05", 1374.119995],\n    ["2007-03-06", 1395.410034],\n    ["2007-03-07", 1391.969971],\n    ["2007-03-08", 1401.890015],\n    ["2007-03-09", 1402.839966],\n    ["2007-03-12", 1406.599976],\n    ["2007-03-13", 1377.949951],\n    ["2007-03-14", 1387.170044],\n    ["2007-03-15", 1392.280029],\n    ["2007-03-16", 1386.949951],\n    ["2007-03-19", 1402.060059],\n    ["2007-03-20", 1410.939941],\n    ["2007-03-21", 1435.040039],\n    ["2007-03-22", 1434.540039],\n    ["2007-03-23", 1436.109985],\n    ["2007-03-26", 1437.5],\n    ["2007-03-27", 1428.609985],\n    ["2007-03-28", 1417.22998],\n    ["2007-03-29", 1422.530029],\n    ["2007-03-30", 1420.859985],\n    ["2007-04-02", 1424.550049],\n    ["2007-04-03", 1437.77002],\n    ["2007-04-04", 1439.369995],\n    ["2007-04-05", 1443.76001],\n    ["2007-04-09", 1444.609985],\n    ["2007-04-10", 1448.390015],\n    ["2007-04-11", 1438.869995],\n    ["2007-04-12", 1447.800049],\n    ["2007-04-13", 1452.849976],\n    ["2007-04-16", 1468.329956],\n    ["2007-04-17", 1471.47998],\n    ["2007-04-18", 1472.5],\n    ["2007-04-19", 1470.72998],\n    ["2007-04-20", 1484.349976],\n    ["2007-04-23", 1480.930054],\n    ["2007-04-24", 1480.410034],\n    ["2007-04-25", 1495.420044],\n    ["2007-04-26", 1494.25],\n    ["2007-04-27", 1494.069946],\n    ["2007-04-30", 1482.369995],\n    ["2007-05-01", 1486.300049],\n    ["2007-05-02", 1495.920044],\n    ["2007-05-03", 1502.390015],\n    ["2007-05-04", 1505.619995],\n    ["2007-05-07", 1509.47998],\n    ["2007-05-08", 1507.719971],\n    ["2007-05-09", 1512.579956],\n    ["2007-05-10", 1491.469971],\n    ["2007-05-11", 1505.849976],\n    ["2007-05-14", 1503.150024],\n    ["2007-05-15", 1501.189941],\n    ["2007-05-16", 1514.140015],\n    ["2007-05-17", 1512.75],\n    ["2007-05-18", 1522.75],\n    ["2007-05-21", 1525.099976],\n    ["2007-05-22", 1524.119995],\n    ["2007-05-23", 1522.280029],\n    ["2007-05-24", 1507.51001],\n    ["2007-05-25", 1515.72998],\n    ["2007-05-29", 1518.109985],\n    ["2007-05-30", 1530.22998],\n    ["2007-05-31", 1530.619995],\n    ["2007-06-01", 1536.339966],\n    ["2007-06-04", 1539.180054],\n    ["2007-06-05", 1530.949951],\n    ["2007-06-06", 1517.380005],\n    ["2007-06-07", 1490.719971],\n    ["2007-06-08", 1507.670044],\n    ["2007-06-11", 1509.119995],\n    ["2007-06-12", 1493.0],\n    ["2007-06-13", 1515.670044],\n    ["2007-06-14", 1522.969971],\n    ["2007-06-15", 1532.910034],\n    ["2007-06-18", 1531.050049],\n    ["2007-06-19", 1533.699951],\n    ["2007-06-20", 1512.839966],\n    ["2007-06-21", 1522.189941],\n    ["2007-06-22", 1502.560059],\n    ["2007-06-25", 1497.73999],\n    ["2007-06-26", 1492.890015],\n    ["2007-06-27", 1506.339966],\n    ["2007-06-28", 1505.709961],\n    ["2007-06-29", 1503.349976],\n    ["2007-07-02", 1519.430054],\n    ["2007-07-03", 1524.869995],\n    ["2007-07-05", 1525.400024],\n    ["2007-07-06", 1530.439941],\n    ["2007-07-09", 1531.849976],\n    ["2007-07-10", 1510.119995],\n    ["2007-07-11", 1518.76001],\n    ["2007-07-12", 1547.699951],\n    ["2007-07-13", 1552.5],\n    ["2007-07-16", 1549.52002],\n    ["2007-07-17", 1549.369995],\n    ["2007-07-18", 1546.170044],\n    ["2007-07-19", 1553.079956],\n    ["2007-07-20", 1534.099976],\n    ["2007-07-23", 1541.569946],\n    ["2007-07-24", 1511.040039],\n    ["2007-07-25", 1518.089966],\n    ["2007-07-26", 1482.660034],\n    ["2007-07-27", 1458.949951],\n    ["2007-07-30", 1473.910034],\n    ["2007-07-31", 1455.27002],\n    ["2007-08-01", 1465.810059],\n    ["2007-08-02", 1472.199951],\n    ["2007-08-03", 1433.060059],\n    ["2007-08-06", 1467.670044],\n    ["2007-08-07", 1476.709961],\n    ["2007-08-08", 1497.48999],\n    ["2007-08-09", 1453.089966],\n    ["2007-08-10", 1453.640015],\n    ["2007-08-13", 1452.920044],\n    ["2007-08-14", 1426.540039],\n    ["2007-08-15", 1406.699951],\n    ["2007-08-16", 1411.27002],\n    ["2007-08-17", 1445.939941],\n    ["2007-08-20", 1445.550049],\n    ["2007-08-21", 1447.119995],\n    ["2007-08-22", 1464.069946],\n    ["2007-08-23", 1462.5],\n    ["2007-08-24", 1479.369995],\n    ["2007-08-27", 1466.790039],\n    ["2007-08-28", 1432.359985],\n    ["2007-08-29", 1463.76001],\n    ["2007-08-30", 1457.640015],\n    ["2007-08-31", 1473.98999],\n    ["2007-09-04", 1489.420044],\n    ["2007-09-05", 1472.290039],\n    ["2007-09-06", 1478.550049],\n    ["2007-09-07", 1453.550049],\n    ["2007-09-10", 1451.699951],\n    ["2007-09-11", 1471.48999],\n    ["2007-09-12", 1471.560059],\n    ["2007-09-13", 1483.949951],\n    ["2007-09-14", 1484.25],\n    ["2007-09-17", 1476.650024],\n    ["2007-09-18", 1519.780029],\n    ["2007-09-19", 1529.030029],\n    ["2007-09-20", 1518.75],\n    ["2007-09-21", 1525.75],\n    ["2007-09-24", 1517.72998],\n    ["2007-09-25", 1517.209961],\n    ["2007-09-26", 1525.420044],\n    ["2007-09-27", 1531.380005],\n    ["2007-09-28", 1526.75],\n    ["2007-10-01", 1547.040039],\n    ["2007-10-02", 1546.630005],\n    ["2007-10-03", 1539.589966],\n    ["2007-10-04", 1542.839966],\n    ["2007-10-05", 1557.589966],\n    ["2007-10-08", 1552.579956],\n    ["2007-10-09", 1565.150024],\n    ["2007-10-10", 1562.469971],\n    ["2007-10-11", 1554.410034],\n    ["2007-10-12", 1561.800049],\n    ["2007-10-15", 1548.709961],\n    ["2007-10-16", 1538.530029],\n    ["2007-10-17", 1541.23999],\n    ["2007-10-18", 1540.079956],\n    ["2007-10-19", 1500.630005],\n    ["2007-10-22", 1506.329956],\n    ["2007-10-23", 1519.589966],\n    ["2007-10-24", 1515.880005],\n    ["2007-10-25", 1514.400024],\n    ["2007-10-26", 1535.280029],\n    ["2007-10-29", 1540.97998],\n    ["2007-10-30", 1531.02002],\n    ["2007-10-31", 1549.380005],\n    ["2007-11-01", 1508.439941],\n    ["2007-11-02", 1509.650024],\n    ["2007-11-05", 1502.170044],\n    ["2007-11-06", 1520.27002],\n    ["2007-11-07", 1475.619995],\n    ["2007-11-08", 1474.77002],\n    ["2007-11-09", 1453.699951],\n    ["2007-11-12", 1439.180054],\n    ["2007-11-13", 1481.050049],\n    ["2007-11-14", 1470.579956],\n    ["2007-11-15", 1451.150024],\n    ["2007-11-16", 1458.73999],\n    ["2007-11-19", 1433.27002],\n    ["2007-11-20", 1439.699951],\n    ["2007-11-21", 1416.77002],\n    ["2007-11-23", 1440.699951],\n    ["2007-11-26", 1407.219971],\n    ["2007-11-27", 1428.22998],\n    ["2007-11-28", 1469.02002],\n    ["2007-11-29", 1469.719971],\n    ["2007-11-30", 1481.140015],\n    ["2007-12-03", 1472.420044],\n    ["2007-12-04", 1462.790039],\n    ["2007-12-05", 1485.01001],\n    ["2007-12-06", 1507.339966],\n    ["2007-12-07", 1504.660034],\n    ["2007-12-10", 1515.959961],\n    ["2007-12-11", 1477.650024],\n    ["2007-12-12", 1486.589966],\n    ["2007-12-13", 1488.410034],\n    ["2007-12-14", 1467.949951],\n    ["2007-12-17", 1445.900024],\n    ["2007-12-18", 1454.97998],\n    ["2007-12-19", 1453.0],\n    ["2007-12-20", 1460.119995],\n    ["2007-12-21", 1484.459961],\n    ["2007-12-24", 1496.449951],\n    ["2007-12-26", 1497.660034],\n    ["2007-12-27", 1476.27002],\n    ["2007-12-28", 1478.48999],\n    ["2007-12-31", 1468.359985],\n    ["2008-01-02", 1447.160034],\n    ["2008-01-03", 1447.160034],\n    ["2008-01-04", 1411.630005],\n    ["2008-01-07", 1416.180054],\n    ["2008-01-08", 1390.189941],\n    ["2008-01-09", 1409.130005],\n    ["2008-01-10", 1420.329956],\n    ["2008-01-11", 1401.02002],\n    ["2008-01-14", 1416.25],\n    ["2008-01-15", 1380.949951],\n    ["2008-01-16", 1373.199951],\n    ["2008-01-17", 1333.25],\n    ["2008-01-18", 1325.189941],\n    ["2008-01-22", 1310.5],\n    ["2008-01-23", 1338.599976],\n    ["2008-01-24", 1352.069946],\n    ["2008-01-25", 1330.609985],\n    ["2008-01-28", 1353.959961],\n    ["2008-01-29", 1362.300049],\n    ["2008-01-30", 1355.810059],\n    ["2008-01-31", 1378.550049],\n    ["2008-02-01", 1395.420044],\n    ["2008-02-04", 1380.819946],\n    ["2008-02-05", 1336.640015],\n    ["2008-02-06", 1326.449951],\n    ["2008-02-07", 1336.910034],\n    ["2008-02-08", 1331.290039],\n    ["2008-02-11", 1339.130005],\n    ["2008-02-12", 1348.859985],\n    ["2008-02-13", 1367.209961],\n    ["2008-02-14", 1348.859985],\n    ["2008-02-15", 1349.98999],\n    ["2008-02-19", 1348.780029],\n    ["2008-02-20", 1360.030029],\n    ["2008-02-21", 1342.530029],\n    ["2008-02-22", 1353.109985],\n    ["2008-02-25", 1371.800049],\n    ["2008-02-26", 1381.290039],\n    ["2008-02-27", 1380.02002],\n    ["2008-02-28", 1367.680054],\n    ["2008-02-29", 1330.630005],\n    ["2008-03-03", 1331.339966],\n    ["2008-03-04", 1326.75],\n    ["2008-03-05", 1333.699951],\n    ["2008-03-06", 1304.339966],\n    ["2008-03-07", 1293.369995],\n    ["2008-03-10", 1273.369995],\n    ["2008-03-11", 1320.650024],\n    ["2008-03-12", 1308.77002],\n    ["2008-03-13", 1315.47998],\n    ["2008-03-14", 1288.140015],\n    ["2008-03-17", 1276.599976],\n    ["2008-03-18", 1330.73999],\n    ["2008-03-19", 1298.420044],\n    ["2008-03-20", 1329.51001],\n    ["2008-03-24", 1349.880005],\n    ["2008-03-25", 1352.98999],\n    ["2008-03-26", 1341.130005],\n    ["2008-03-27", 1325.76001],\n    ["2008-03-28", 1315.219971],\n    ["2008-03-31", 1322.699951],\n    ["2008-04-01", 1370.180054],\n    ["2008-04-02", 1367.530029],\n    ["2008-04-03", 1369.310059],\n    ["2008-04-04", 1370.400024],\n    ["2008-04-07", 1372.540039],\n    ["2008-04-08", 1365.540039],\n    ["2008-04-09", 1354.48999],\n    ["2008-04-10", 1360.550049],\n    ["2008-04-11", 1332.829956],\n    ["2008-04-14", 1328.319946],\n    ["2008-04-15", 1334.430054],\n    ["2008-04-16", 1364.709961],\n    ["2008-04-17", 1365.560059],\n    ["2008-04-18", 1390.329956],\n    ["2008-04-21", 1388.170044],\n    ["2008-04-22", 1375.939941],\n    ["2008-04-23", 1379.930054],\n    ["2008-04-24", 1388.819946],\n    ["2008-04-25", 1397.839966],\n    ["2008-04-28", 1396.369995],\n    ["2008-04-29", 1390.939941],\n    ["2008-04-30", 1385.589966],\n    ["2008-05-01", 1409.339966],\n    ["2008-05-02", 1413.900024],\n    ["2008-05-05", 1407.48999],\n    ["2008-05-06", 1418.26001],\n    ["2008-05-07", 1392.569946],\n    ["2008-05-08", 1397.680054],\n    ["2008-05-09", 1388.280029],\n    ["2008-05-12", 1403.579956],\n    ["2008-05-13", 1403.040039],\n    ["2008-05-14", 1408.660034],\n    ["2008-05-15", 1423.569946],\n    ["2008-05-16", 1425.349976],\n    ["2008-05-19", 1426.630005],\n    ["2008-05-20", 1413.400024],\n    ["2008-05-21", 1390.709961],\n    ["2008-05-22", 1394.349976],\n    ["2008-05-23", 1375.930054],\n    ["2008-05-27", 1385.349976],\n    ["2008-05-28", 1390.839966],\n    ["2008-05-29", 1398.26001],\n    ["2008-05-30", 1400.380005],\n    ["2008-06-02", 1385.670044],\n    ["2008-06-03", 1377.650024],\n    ["2008-06-04", 1377.199951],\n    ["2008-06-05", 1404.050049],\n    ["2008-06-06", 1360.680054],\n    ["2008-06-09", 1361.76001],\n    ["2008-06-10", 1358.439941],\n    ["2008-06-11", 1335.48999],\n    ["2008-06-12", 1339.869995],\n    ["2008-06-13", 1360.030029],\n    ["2008-06-16", 1360.140015],\n    ["2008-06-17", 1350.930054],\n    ["2008-06-18", 1337.810059],\n    ["2008-06-19", 1342.829956],\n    ["2008-06-20", 1317.930054],\n    ["2008-06-23", 1318.0],\n    ["2008-06-24", 1314.290039],\n    ["2008-06-25", 1321.969971],\n    ["2008-06-26", 1283.150024],\n    ["2008-06-27", 1278.380005],\n    ["2008-06-30", 1280.0],\n    ["2008-07-01", 1284.910034],\n    ["2008-07-02", 1261.52002],\n    ["2008-07-03", 1262.900024],\n    ["2008-07-07", 1252.310059],\n    ["2008-07-08", 1273.699951],\n    ["2008-07-09", 1244.689941],\n    ["2008-07-10", 1253.390015],\n    ["2008-07-11", 1239.48999],\n    ["2008-07-14", 1228.300049],\n    ["2008-07-15", 1214.910034],\n    ["2008-07-16", 1245.359985],\n    ["2008-07-17", 1260.319946],\n    ["2008-07-18", 1260.680054],\n    ["2008-07-21", 1260.0],\n    ["2008-07-22", 1277.0],\n    ["2008-07-23", 1282.189941],\n    ["2008-07-24", 1252.540039],\n    ["2008-07-25", 1257.76001],\n    ["2008-07-28", 1234.369995],\n    ["2008-07-29", 1263.199951],\n    ["2008-07-30", 1284.26001],\n    ["2008-07-31", 1267.380005],\n    ["2008-08-01", 1260.310059],\n    ["2008-08-04", 1249.01001],\n    ["2008-08-05", 1284.880005],\n    ["2008-08-06", 1289.189941],\n    ["2008-08-07", 1266.069946],\n    ["2008-08-08", 1296.319946],\n    ["2008-08-11", 1305.319946],\n    ["2008-08-12", 1289.589966],\n    ["2008-08-13", 1285.829956],\n    ["2008-08-14", 1292.930054],\n    ["2008-08-15", 1298.199951],\n    ["2008-08-18", 1278.599976],\n    ["2008-08-19", 1266.689941],\n    ["2008-08-20", 1274.540039],\n    ["2008-08-21", 1277.719971],\n    ["2008-08-22", 1292.199951],\n    ["2008-08-25", 1266.839966],\n    ["2008-08-26", 1271.51001],\n    ["2008-08-27", 1281.660034],\n    ["2008-08-28", 1300.680054],\n    ["2008-08-29", 1282.829956],\n    ["2008-09-02", 1277.579956],\n    ["2008-09-03", 1274.97998],\n    ["2008-09-04", 1236.829956],\n    ["2008-09-05", 1242.310059],\n    ["2008-09-08", 1267.790039],\n    ["2008-09-09", 1224.51001],\n    ["2008-09-10", 1232.040039],\n    ["2008-09-11", 1249.050049],\n    ["2008-09-12", 1251.699951],\n    ["2008-09-15", 1192.699951],\n    ["2008-09-16", 1213.599976],\n    ["2008-09-17", 1156.390015],\n    ["2008-09-18", 1206.51001],\n    ["2008-09-19", 1255.079956],\n    ["2008-09-22", 1207.089966],\n    ["2008-09-23", 1188.219971],\n    ["2008-09-24", 1185.869995],\n    ["2008-09-25", 1209.180054],\n    ["2008-09-26", 1213.27002],\n    ["2008-09-29", 1106.420044],\n    ["2008-09-30", 1166.359985],\n    ["2008-10-01", 1161.060059],\n    ["2008-10-02", 1114.280029],\n    ["2008-10-03", 1099.22998],\n    ["2008-10-06", 1056.890015],\n    ["2008-10-07", 996.22998],\n    ["2008-10-08", 984.940002],\n    ["2008-10-09", 909.919983],\n    ["2008-10-10", 899.219971],\n    ["2008-10-13", 1003.349976],\n    ["2008-10-14", 998.01001],\n    ["2008-10-15", 907.840027],\n    ["2008-10-16", 946.429993],\n    ["2008-10-17", 940.549988],\n    ["2008-10-20", 985.400024],\n    ["2008-10-21", 955.049988],\n    ["2008-10-22", 896.780029],\n    ["2008-10-23", 908.109985],\n    ["2008-10-24", 876.77002],\n    ["2008-10-27", 848.919983],\n    ["2008-10-28", 940.51001],\n    ["2008-10-29", 930.090027],\n    ["2008-10-30", 954.090027],\n    ["2008-10-31", 968.75],\n    ["2008-11-03", 966.299988],\n    ["2008-11-04", 1005.75],\n    ["2008-11-05", 952.77002],\n    ["2008-11-06", 904.880005],\n    ["2008-11-07", 930.98999],\n    ["2008-11-10", 919.210022],\n    ["2008-11-11", 898.950012],\n    ["2008-11-12", 852.299988],\n    ["2008-11-13", 911.289978],\n    ["2008-11-14", 873.289978],\n    ["2008-11-17", 850.75],\n    ["2008-11-18", 859.119995],\n    ["2008-11-19", 806.580017],\n    ["2008-11-20", 752.440002],\n    ["2008-11-21", 800.030029],\n    ["2008-11-24", 851.809998],\n    ["2008-11-25", 857.390015],\n    ["2008-11-26", 887.679993],\n    ["2008-11-28", 896.23999],\n    ["2008-12-01", 816.210022],\n    ["2008-12-02", 848.809998],\n    ["2008-12-03", 870.73999],\n    ["2008-12-04", 845.219971],\n    ["2008-12-05", 876.070007],\n    ["2008-12-08", 909.700012],\n    ["2008-12-09", 888.669983],\n    ["2008-12-10", 899.23999],\n    ["2008-12-11", 873.590027],\n    ["2008-12-12", 879.72998],\n    ["2008-12-15", 868.570007],\n    ["2008-12-16", 913.179993],\n    ["2008-12-17", 904.419983],\n    ["2008-12-18", 885.280029],\n    ["2008-12-19", 887.880005],\n    ["2008-12-22", 871.630005],\n    ["2008-12-23", 863.159973],\n    ["2008-12-24", 868.150024],\n    ["2008-12-26", 872.799988],\n    ["2008-12-29", 869.419983],\n    ["2008-12-30", 890.640015],\n    ["2008-12-31", 903.25],\n    ["2009-01-02", 931.799988],\n    ["2009-01-05", 927.450012],\n    ["2009-01-06", 934.700012],\n    ["2009-01-07", 906.650024],\n    ["2009-01-08", 909.72998],\n    ["2009-01-09", 890.349976],\n    ["2009-01-12", 870.26001],\n    ["2009-01-13", 871.789978],\n    ["2009-01-14", 842.619995],\n    ["2009-01-15", 843.73999],\n    ["2009-01-16", 850.119995],\n    ["2009-01-20", 805.219971],\n    ["2009-01-21", 840.23999],\n    ["2009-01-22", 827.5],\n    ["2009-01-23", 831.950012],\n    ["2009-01-26", 836.570007],\n    ["2009-01-27", 845.710022],\n    ["2009-01-28", 874.090027],\n    ["2009-01-29", 845.140015],\n    ["2009-01-30", 825.880005],\n    ["2009-02-02", 825.440002],\n    ["2009-02-03", 838.51001],\n    ["2009-02-04", 832.22998],\n    ["2009-02-05", 845.849976],\n    ["2009-02-06", 868.599976],\n    ["2009-02-09", 869.890015],\n    ["2009-02-10", 827.159973],\n    ["2009-02-11", 833.73999],\n    ["2009-02-12", 835.190002],\n    ["2009-02-13", 826.840027],\n    ["2009-02-17", 789.169983],\n    ["2009-02-18", 788.419983],\n    ["2009-02-19", 778.940002],\n    ["2009-02-20", 770.049988],\n    ["2009-02-23", 743.330017],\n    ["2009-02-24", 773.140015],\n    ["2009-02-25", 764.900024],\n    ["2009-02-26", 752.830017],\n    ["2009-02-27", 735.090027],\n    ["2009-03-02", 700.820007],\n    ["2009-03-03", 696.330017],\n    ["2009-03-04", 712.869995],\n    ["2009-03-05", 682.549988],\n    ["2009-03-06", 683.380005],\n    ["2009-03-09", 676.530029],\n    ["2009-03-10", 719.599976],\n    ["2009-03-11", 721.359985],\n    ["2009-03-12", 750.73999],\n    ["2009-03-13", 756.549988],\n    ["2009-03-16", 753.890015],\n    ["2009-03-17", 778.119995],\n    ["2009-03-18", 794.349976],\n    ["2009-03-19", 784.039978],\n    ["2009-03-20", 768.539978],\n    ["2009-03-23", 822.919983],\n    ["2009-03-24", 806.119995],\n    ["2009-03-25", 813.880005],\n    ["2009-03-26", 832.859985],\n    ["2009-03-27", 815.940002],\n    ["2009-03-30", 787.530029],\n    ["2009-03-31", 797.869995],\n    ["2009-04-01", 811.080017],\n    ["2009-04-02", 834.380005],\n    ["2009-04-03", 842.5],\n    ["2009-04-06", 835.47998],\n    ["2009-04-07", 815.549988],\n    ["2009-04-08", 825.159973],\n    ["2009-04-09", 856.559998],\n    ["2009-04-13", 858.72998],\n    ["2009-04-14", 841.5],\n    ["2009-04-15", 852.059998],\n    ["2009-04-16", 865.299988],\n    ["2009-04-17", 869.599976],\n    ["2009-04-20", 832.390015],\n    ["2009-04-21", 850.080017],\n    ["2009-04-22", 843.549988],\n    ["2009-04-23", 851.919983],\n    ["2009-04-24", 866.22998],\n    ["2009-04-27", 857.51001],\n    ["2009-04-28", 855.159973],\n    ["2009-04-29", 873.640015],\n    ["2009-04-30", 872.809998],\n    ["2009-05-01", 877.52002],\n    ["2009-05-04", 907.23999],\n    ["2009-05-05", 903.799988],\n    ["2009-05-06", 919.530029],\n    ["2009-05-07", 907.390015],\n    ["2009-05-08", 929.22998],\n    ["2009-05-11", 909.23999],\n    ["2009-05-12", 908.349976],\n    ["2009-05-13", 883.919983],\n    ["2009-05-14", 893.070007],\n    ["2009-05-15", 882.880005],\n    ["2009-05-18", 909.710022],\n    ["2009-05-19", 908.130005],\n    ["2009-05-20", 903.469971],\n    ["2009-05-21", 888.330017],\n    ["2009-05-22", 887.0],\n    ["2009-05-26", 910.330017],\n    ["2009-05-27", 893.059998],\n    ["2009-05-28", 906.830017],\n    ["2009-05-29", 919.140015],\n    ["2009-06-01", 942.869995],\n    ["2009-06-02", 944.73999],\n    ["2009-06-03", 931.76001],\n    ["2009-06-04", 942.460022],\n    ["2009-06-05", 940.090027],\n    ["2009-06-08", 939.140015],\n    ["2009-06-09", 942.429993],\n    ["2009-06-10", 939.150024],\n    ["2009-06-11", 944.890015],\n    ["2009-06-12", 946.210022],\n    ["2009-06-15", 923.719971],\n    ["2009-06-16", 911.969971],\n    ["2009-06-17", 910.710022],\n    ["2009-06-18", 918.369995],\n    ["2009-06-19", 921.22998],\n    ["2009-06-22", 893.039978],\n    ["2009-06-23", 895.099976],\n    ["2009-06-24", 900.940002],\n    ["2009-06-25", 920.26001],\n    ["2009-06-26", 918.900024],\n    ["2009-06-29", 927.22998],\n    ["2009-06-30", 919.320007],\n    ["2009-07-01", 923.330017],\n    ["2009-07-02", 896.419983],\n    ["2009-07-06", 898.719971],\n    ["2009-07-07", 881.030029],\n    ["2009-07-08", 879.559998],\n    ["2009-07-09", 882.679993],\n    ["2009-07-10", 879.130005],\n    ["2009-07-13", 901.049988],\n    ["2009-07-14", 905.840027],\n    ["2009-07-15", 932.679993],\n    ["2009-07-16", 940.73999],\n    ["2009-07-17", 940.380005],\n    ["2009-07-20", 951.130005],\n    ["2009-07-21", 954.580017],\n    ["2009-07-22", 954.070007],\n    ["2009-07-23", 976.289978],\n    ["2009-07-24", 979.26001],\n    ["2009-07-27", 982.179993],\n    ["2009-07-28", 979.619995],\n    ["2009-07-29", 975.150024],\n    ["2009-07-30", 986.75],\n    ["2009-07-31", 987.47998],\n    ["2009-08-03", 1002.630005],\n    ["2009-08-04", 1005.650024],\n    ["2009-08-05", 1002.719971],\n    ["2009-08-06", 997.080017],\n    ["2009-08-07", 1010.47998],\n    ["2009-08-10", 1007.099976],\n    ["2009-08-11", 994.349976],\n    ["2009-08-12", 1005.809998],\n    ["2009-08-13", 1012.72998],\n    ["2009-08-14", 1004.090027],\n    ["2009-08-17", 979.72998],\n    ["2009-08-18", 989.669983],\n    ["2009-08-19", 996.460022],\n    ["2009-08-20", 1007.369995],\n    ["2009-08-21", 1026.130005],\n    ["2009-08-24", 1025.569946],\n    ["2009-08-25", 1028.0],\n    ["2009-08-26", 1028.119995],\n    ["2009-08-27", 1030.97998],\n    ["2009-08-28", 1028.930054],\n    ["2009-08-31", 1020.619995],\n    ["2009-09-01", 998.039978],\n    ["2009-09-02", 994.75],\n    ["2009-09-03", 1003.23999],\n    ["2009-09-04", 1016.400024],\n    ["2009-09-08", 1025.390015],\n    ["2009-09-09", 1033.369995],\n    ["2009-09-10", 1044.140015],\n    ["2009-09-11", 1042.72998],\n    ["2009-09-14", 1049.339966],\n    ["2009-09-15", 1052.630005],\n    ["2009-09-16", 1068.76001],\n    ["2009-09-17", 1065.48999],\n    ["2009-09-18", 1068.300049],\n    ["2009-09-21", 1064.660034],\n    ["2009-09-22", 1071.660034],\n    ["2009-09-23", 1060.869995],\n    ["2009-09-24", 1050.780029],\n    ["2009-09-25", 1044.380005],\n    ["2009-09-28", 1062.97998],\n    ["2009-09-29", 1060.609985],\n    ["2009-09-30", 1057.079956],\n    ["2009-10-01", 1029.849976],\n    ["2009-10-02", 1025.209961],\n    ["2009-10-05", 1040.459961],\n    ["2009-10-06", 1054.719971],\n    ["2009-10-07", 1057.579956],\n    ["2009-10-08", 1065.47998],\n    ["2009-10-09", 1071.48999],\n    ["2009-10-12", 1076.189941],\n    ["2009-10-13", 1073.189941],\n    ["2009-10-14", 1092.02002],\n    ["2009-10-15", 1096.560059],\n    ["2009-10-16", 1087.680054],\n    ["2009-10-19", 1097.910034],\n    ["2009-10-20", 1091.060059],\n    ["2009-10-21", 1081.400024],\n    ["2009-10-22", 1092.910034],\n    ["2009-10-23", 1079.599976],\n    ["2009-10-26", 1066.949951],\n    ["2009-10-27", 1063.410034],\n    ["2009-10-28", 1042.630005],\n    ["2009-10-29", 1066.109985],\n    ["2009-10-30", 1036.189941],\n    ["2009-11-02", 1042.880005],\n    ["2009-11-03", 1045.410034],\n    ["2009-11-04", 1046.5],\n    ["2009-11-05", 1066.630005],\n    ["2009-11-06", 1069.300049],\n    ["2009-11-09", 1093.079956],\n    ["2009-11-10", 1093.01001],\n    ["2009-11-11", 1098.51001],\n    ["2009-11-12", 1087.23999],\n    ["2009-11-13", 1093.47998],\n    ["2009-11-16", 1109.300049],\n    ["2009-11-17", 1110.319946],\n    ["2009-11-18", 1109.800049],\n    ["2009-11-19", 1094.900024],\n    ["2009-11-20", 1091.380005],\n    ["2009-11-23", 1106.23999],\n    ["2009-11-24", 1105.650024],\n    ["2009-11-25", 1110.630005],\n    ["2009-11-27", 1091.48999],\n    ["2009-11-30", 1095.630005],\n    ["2009-12-01", 1108.859985],\n    ["2009-12-02", 1109.23999],\n    ["2009-12-03", 1099.920044],\n    ["2009-12-04", 1105.97998],\n    ["2009-12-07", 1103.25],\n    ["2009-12-08", 1091.939941],\n    ["2009-12-09", 1095.949951],\n    ["2009-12-10", 1102.349976],\n    ["2009-12-11", 1106.410034],\n    ["2009-12-14", 1114.109985],\n    ["2009-12-15", 1107.930054],\n    ["2009-12-16", 1109.180054],\n    ["2009-12-17", 1096.079956],\n    ["2009-12-18", 1102.469971],\n    ["2009-12-21", 1114.050049],\n    ["2009-12-22", 1118.02002],\n    ["2009-12-23", 1120.589966],\n    ["2009-12-24", 1126.47998],\n    ["2009-12-28", 1127.780029],\n    ["2009-12-29", 1126.199951],\n    ["2009-12-30", 1126.420044],\n    ["2009-12-31", 1115.099976],\n    ["2010-01-04", 1132.98999],\n    ["2010-01-05", 1136.52002],\n    ["2010-01-06", 1137.140015],\n    ["2010-01-07", 1141.689941],\n    ["2010-01-08", 1144.97998],\n    ["2010-01-11", 1146.97998],\n    ["2010-01-12", 1136.219971],\n    ["2010-01-13", 1145.680054],\n    ["2010-01-14", 1148.459961],\n    ["2010-01-15", 1136.030029],\n    ["2010-01-19", 1150.22998],\n    ["2010-01-20", 1138.040039],\n    ["2010-01-21", 1116.47998],\n    ["2010-01-22", 1091.76001],\n    ["2010-01-25", 1096.780029],\n    ["2010-01-26", 1092.170044],\n    ["2010-01-27", 1097.5],\n    ["2010-01-28", 1084.530029],\n    ["2010-01-29", 1073.869995],\n    ["2010-02-01", 1089.189941],\n    ["2010-02-02", 1103.319946],\n    ["2010-02-03", 1097.280029],\n    ["2010-02-04", 1063.109985],\n    ["2010-02-05", 1066.189941],\n    ["2010-02-08", 1056.73999],\n    ["2010-02-09", 1070.52002],\n    ["2010-02-10", 1068.130005],\n    ["2010-02-11", 1078.469971],\n    ["2010-02-12", 1075.51001],\n    ["2010-02-16", 1094.869995],\n    ["2010-02-17", 1099.51001],\n    ["2010-02-18", 1106.75],\n    ["2010-02-19", 1109.170044],\n    ["2010-02-22", 1108.01001],\n    ["2010-02-23", 1094.599976],\n    ["2010-02-24", 1105.23999],\n    ["2010-02-25", 1102.939941],\n    ["2010-02-26", 1104.48999],\n    ["2010-03-01", 1115.709961],\n    ["2010-03-02", 1118.310059],\n    ["2010-03-03", 1118.790039],\n    ["2010-03-04", 1122.969971],\n    ["2010-03-05", 1138.699951],\n    ["2010-03-08", 1138.5],\n    ["2010-03-09", 1140.449951],\n    ["2010-03-10", 1145.609985],\n    ["2010-03-11", 1150.23999],\n    ["2010-03-12", 1149.98999],\n    ["2010-03-15", 1150.51001],\n    ["2010-03-16", 1159.459961],\n    ["2010-03-17", 1166.209961],\n    ["2010-03-18", 1165.829956],\n    ["2010-03-19", 1159.900024],\n    ["2010-03-22", 1165.810059],\n    ["2010-03-23", 1174.170044],\n    ["2010-03-24", 1167.719971],\n    ["2010-03-25", 1165.72998],\n    ["2010-03-26", 1166.589966],\n    ["2010-03-29", 1173.219971],\n    ["2010-03-30", 1173.27002],\n    ["2010-03-31", 1169.430054],\n    ["2010-04-01", 1178.099976],\n    ["2010-04-05", 1187.439941],\n    ["2010-04-06", 1189.439941],\n    ["2010-04-07", 1182.449951],\n    ["2010-04-08", 1186.439941],\n    ["2010-04-09", 1194.369995],\n    ["2010-04-12", 1196.47998],\n    ["2010-04-13", 1197.300049],\n    ["2010-04-14", 1210.650024],\n    ["2010-04-15", 1211.670044],\n    ["2010-04-16", 1192.130005],\n    ["2010-04-19", 1197.52002],\n    ["2010-04-20", 1207.170044],\n    ["2010-04-21", 1205.939941],\n    ["2010-04-22", 1208.670044],\n    ["2010-04-23", 1217.280029],\n    ["2010-04-26", 1212.050049],\n    ["2010-04-27", 1183.709961],\n    ["2010-04-28", 1191.359985],\n    ["2010-04-29", 1206.780029],\n    ["2010-04-30", 1186.689941],\n    ["2010-05-03", 1202.26001],\n    ["2010-05-04", 1173.599976],\n    ["2010-05-05", 1165.869995],\n    ["2010-05-06", 1128.150024],\n    ["2010-05-07", 1110.880005],\n    ["2010-05-10", 1159.72998],\n    ["2010-05-11", 1155.790039],\n    ["2010-05-12", 1171.670044],\n    ["2010-05-13", 1157.439941],\n    ["2010-05-14", 1135.680054],\n    ["2010-05-17", 1136.939941],\n    ["2010-05-18", 1120.800049],\n    ["2010-05-19", 1115.050049],\n    ["2010-05-20", 1071.589966],\n    ["2010-05-21", 1087.689941],\n    ["2010-05-24", 1073.650024],\n    ["2010-05-25", 1074.030029],\n    ["2010-05-26", 1067.949951],\n    ["2010-05-27", 1103.060059],\n    ["2010-05-28", 1089.410034],\n    ["2010-06-01", 1070.709961],\n    ["2010-06-02", 1098.380005],\n    ["2010-06-03", 1102.829956],\n    ["2010-06-04", 1064.880005],\n    ["2010-06-07", 1050.469971],\n    ["2010-06-08", 1062.0],\n    ["2010-06-09", 1055.689941],\n    ["2010-06-10", 1086.839966],\n    ["2010-06-11", 1091.599976],\n    ["2010-06-14", 1089.630005],\n    ["2010-06-15", 1115.22998],\n    ["2010-06-16", 1114.609985],\n    ["2010-06-17", 1116.040039],\n    ["2010-06-18", 1117.51001],\n    ["2010-06-21", 1113.199951],\n    ["2010-06-22", 1095.310059],\n    ["2010-06-23", 1092.040039],\n    ["2010-06-24", 1073.689941],\n    ["2010-06-25", 1076.76001],\n    ["2010-06-28", 1074.569946],\n    ["2010-06-29", 1041.23999],\n    ["2010-06-30", 1030.709961],\n    ["2010-07-01", 1027.369995],\n    ["2010-07-02", 1022.580017],\n    ["2010-07-06", 1028.060059],\n    ["2010-07-07", 1060.27002],\n    ["2010-07-08", 1070.25],\n    ["2010-07-09", 1077.959961],\n    ["2010-07-12", 1078.75],\n    ["2010-07-13", 1095.339966],\n    ["2010-07-14", 1095.170044],\n    ["2010-07-15", 1096.47998],\n    ["2010-07-16", 1064.880005],\n    ["2010-07-19", 1071.25],\n    ["2010-07-20", 1083.47998],\n    ["2010-07-21", 1069.589966],\n    ["2010-07-22", 1093.670044],\n    ["2010-07-23", 1102.660034],\n    ["2010-07-26", 1115.01001],\n    ["2010-07-27", 1113.839966],\n    ["2010-07-28", 1106.130005],\n    ["2010-07-29", 1101.530029],\n    ["2010-07-30", 1101.599976],\n    ["2010-08-02", 1125.859985],\n    ["2010-08-03", 1120.459961],\n    ["2010-08-04", 1127.23999],\n    ["2010-08-05", 1125.810059],\n    ["2010-08-06", 1121.640015],\n    ["2010-08-09", 1127.790039],\n    ["2010-08-10", 1121.060059],\n    ["2010-08-11", 1089.469971],\n    ["2010-08-12", 1083.609985],\n    ["2010-08-13", 1079.25],\n    ["2010-08-16", 1079.380005],\n    ["2010-08-17", 1092.540039],\n    ["2010-08-18", 1094.160034],\n    ["2010-08-19", 1075.630005],\n    ["2010-08-20", 1071.689941],\n    ["2010-08-23", 1067.359985],\n    ["2010-08-24", 1051.869995],\n    ["2010-08-25", 1055.329956],\n    ["2010-08-26", 1047.219971],\n    ["2010-08-27", 1064.589966],\n    ["2010-08-30", 1048.920044],\n    ["2010-08-31", 1049.329956],\n    ["2010-09-01", 1080.290039],\n    ["2010-09-02", 1090.099976],\n    ["2010-09-03", 1104.51001],\n    ["2010-09-07", 1091.839966],\n    ["2010-09-08", 1098.869995],\n    ["2010-09-09", 1104.180054],\n    ["2010-09-10", 1109.550049],\n    ["2010-09-13", 1121.900024],\n    ["2010-09-14", 1121.099976],\n    ["2010-09-15", 1125.069946],\n    ["2010-09-16", 1124.660034],\n    ["2010-09-17", 1125.589966],\n    ["2010-09-20", 1142.709961],\n    ["2010-09-21", 1139.780029],\n    ["2010-09-22", 1134.280029],\n    ["2010-09-23", 1124.829956],\n    ["2010-09-24", 1148.670044],\n    ["2010-09-27", 1142.160034],\n    ["2010-09-28", 1147.699951],\n    ["2010-09-29", 1144.72998],\n    ["2010-09-30", 1141.199951],\n    ["2010-10-01", 1146.23999],\n    ["2010-10-04", 1137.030029],\n    ["2010-10-05", 1160.75],\n    ["2010-10-06", 1159.969971],\n    ["2010-10-07", 1158.060059],\n    ["2010-10-08", 1165.150024],\n    ["2010-10-11", 1165.319946],\n    ["2010-10-12", 1169.77002],\n    ["2010-10-13", 1178.099976],\n    ["2010-10-14", 1173.810059],\n    ["2010-10-15", 1176.189941],\n    ["2010-10-18", 1184.709961],\n    ["2010-10-19", 1165.900024],\n    ["2010-10-20", 1178.170044],\n    ["2010-10-21", 1180.26001],\n    ["2010-10-22", 1183.079956],\n    ["2010-10-25", 1185.619995],\n    ["2010-10-26", 1185.640015],\n    ["2010-10-27", 1182.449951],\n    ["2010-10-28", 1183.780029],\n    ["2010-10-29", 1183.26001],\n    ["2010-11-01", 1184.380005],\n    ["2010-11-02", 1193.569946],\n    ["2010-11-03", 1197.959961],\n    ["2010-11-04", 1221.060059],\n    ["2010-11-05", 1225.849976],\n    ["2010-11-08", 1223.25],\n    ["2010-11-09", 1213.400024],\n    ["2010-11-10", 1218.709961],\n    ["2010-11-11", 1213.540039],\n    ["2010-11-12", 1199.209961],\n    ["2010-11-15", 1197.75],\n    ["2010-11-16", 1178.339966],\n    ["2010-11-17", 1178.589966],\n    ["2010-11-18", 1196.689941],\n    ["2010-11-19", 1199.72998],\n    ["2010-11-22", 1197.839966],\n    ["2010-11-23", 1180.72998],\n    ["2010-11-24", 1198.349976],\n    ["2010-11-26", 1189.400024],\n    ["2010-11-29", 1187.76001],\n    ["2010-11-30", 1180.550049],\n    ["2010-12-01", 1206.069946],\n    ["2010-12-02", 1221.530029],\n    ["2010-12-03", 1224.709961],\n    ["2010-12-06", 1223.119995],\n    ["2010-12-07", 1223.75],\n    ["2010-12-08", 1228.280029],\n    ["2010-12-09", 1233.0],\n    ["2010-12-10", 1240.400024],\n    ["2010-12-13", 1240.459961],\n    ["2010-12-14", 1241.589966],\n    ["2010-12-15", 1235.22998],\n    ["2010-12-16", 1242.869995],\n    ["2010-12-17", 1243.910034],\n    ["2010-12-20", 1247.079956],\n    ["2010-12-21", 1254.599976],\n    ["2010-12-22", 1258.839966],\n    ["2010-12-23", 1256.77002],\n    ["2010-12-27", 1257.540039],\n    ["2010-12-28", 1258.51001],\n    ["2010-12-29", 1259.780029],\n    ["2010-12-30", 1257.880005],\n    ["2010-12-31", 1257.640015],\n    ["2011-01-03", 1271.869995],\n    ["2011-01-04", 1270.199951],\n    ["2011-01-05", 1276.560059],\n    ["2011-01-06", 1273.849976],\n    ["2011-01-07", 1271.5],\n    ["2011-01-10", 1269.75],\n    ["2011-01-11", 1274.47998],\n    ["2011-01-12", 1285.959961],\n    ["2011-01-13", 1283.76001],\n    ["2011-01-14", 1293.23999],\n    ["2011-01-18", 1295.02002],\n    ["2011-01-19", 1281.920044],\n    ["2011-01-20", 1280.26001],\n    ["2011-01-21", 1283.349976],\n    ["2011-01-24", 1290.839966],\n    ["2011-01-25", 1291.180054],\n    ["2011-01-26", 1296.630005],\n    ["2011-01-27", 1299.540039],\n    ["2011-01-28", 1276.339966],\n    ["2011-01-31", 1286.119995],\n    ["2011-02-01", 1307.589966],\n    ["2011-02-02", 1304.030029],\n    ["2011-02-03", 1307.099976],\n    ["2011-02-04", 1310.869995],\n    ["2011-02-07", 1319.050049],\n    ["2011-02-08", 1324.569946],\n    ["2011-02-09", 1320.880005],\n    ["2011-02-10", 1321.869995],\n    ["2011-02-11", 1329.150024],\n    ["2011-02-14", 1332.319946],\n    ["2011-02-15", 1328.01001],\n    ["2011-02-16", 1336.319946],\n    ["2011-02-17", 1340.430054],\n    ["2011-02-18", 1343.01001],\n    ["2011-02-22", 1315.439941],\n    ["2011-02-23", 1307.400024],\n    ["2011-02-24", 1306.099976],\n    ["2011-02-25", 1319.880005],\n    ["2011-02-28", 1327.219971],\n    ["2011-03-01", 1306.329956],\n    ["2011-03-02", 1308.439941],\n    ["2011-03-03", 1330.969971],\n    ["2011-03-04", 1321.150024],\n    ["2011-03-07", 1310.130005],\n    ["2011-03-08", 1321.819946],\n    ["2011-03-09", 1320.02002],\n    ["2011-03-10", 1295.109985],\n    ["2011-03-11", 1304.280029],\n    ["2011-03-14", 1296.390015],\n    ["2011-03-15", 1281.869995],\n    ["2011-03-16", 1256.880005],\n    ["2011-03-17", 1273.719971],\n    ["2011-03-18", 1279.209961],\n    ["2011-03-21", 1298.380005],\n    ["2011-03-22", 1293.77002],\n    ["2011-03-23", 1297.540039],\n    ["2011-03-24", 1309.660034],\n    ["2011-03-25", 1313.800049],\n    ["2011-03-28", 1310.189941],\n    ["2011-03-29", 1319.439941],\n    ["2011-03-30", 1328.26001],\n    ["2011-03-31", 1325.829956],\n    ["2011-04-01", 1332.410034],\n    ["2011-04-04", 1332.869995],\n    ["2011-04-05", 1332.630005],\n    ["2011-04-06", 1335.540039],\n    ["2011-04-07", 1333.51001],\n    ["2011-04-08", 1328.170044],\n    ["2011-04-11", 1324.459961],\n    ["2011-04-12", 1314.160034],\n    ["2011-04-13", 1314.410034],\n    ["2011-04-14", 1314.52002],\n    ["2011-04-15", 1319.680054],\n    ["2011-04-18", 1305.140015],\n    ["2011-04-19", 1312.619995],\n    ["2011-04-20", 1330.359985],\n    ["2011-04-21", 1337.380005],\n    ["2011-04-25", 1335.25],\n    ["2011-04-26", 1347.23999],\n    ["2011-04-27", 1355.660034],\n    ["2011-04-28", 1360.47998],\n    ["2011-04-29", 1363.609985],\n    ["2011-05-02", 1361.219971],\n    ["2011-05-03", 1356.619995],\n    ["2011-05-04", 1347.319946],\n    ["2011-05-05", 1335.099976],\n    ["2011-05-06", 1340.199951],\n    ["2011-05-09", 1346.290039],\n    ["2011-05-10", 1357.160034],\n    ["2011-05-11", 1342.079956],\n    ["2011-05-12", 1348.650024],\n    ["2011-05-13", 1337.77002],\n    ["2011-05-16", 1329.469971],\n    ["2011-05-17", 1328.97998],\n    ["2011-05-18", 1340.680054],\n    ["2011-05-19", 1343.599976],\n    ["2011-05-20", 1333.27002],\n    ["2011-05-23", 1317.369995],\n    ["2011-05-24", 1316.280029],\n    ["2011-05-25", 1320.469971],\n    ["2011-05-26", 1325.689941],\n    ["2011-05-27", 1331.099976],\n    ["2011-05-31", 1345.199951],\n    ["2011-06-01", 1314.550049],\n    ["2011-06-02", 1312.939941],\n    ["2011-06-03", 1300.160034],\n    ["2011-06-06", 1286.170044],\n    ["2011-06-07", 1284.939941],\n    ["2011-06-08", 1279.560059],\n    ["2011-06-09", 1289.0],\n    ["2011-06-10", 1270.97998],\n    ["2011-06-13", 1271.829956],\n    ["2011-06-14", 1287.869995],\n    ["2011-06-15", 1265.420044],\n    ["2011-06-16", 1267.640015],\n    ["2011-06-17", 1271.5],\n    ["2011-06-20", 1278.359985],\n    ["2011-06-21", 1295.52002],\n    ["2011-06-22", 1287.140015],\n    ["2011-06-23", 1283.5],\n    ["2011-06-24", 1268.449951],\n    ["2011-06-27", 1280.099976],\n    ["2011-06-28", 1296.670044],\n    ["2011-06-29", 1307.410034],\n    ["2011-06-30", 1320.640015],\n    ["2011-07-01", 1339.670044],\n    ["2011-07-05", 1337.880005],\n    ["2011-07-06", 1339.219971],\n    ["2011-07-07", 1353.219971],\n    ["2011-07-08", 1343.800049],\n    ["2011-07-11", 1319.48999],\n    ["2011-07-12", 1313.640015],\n    ["2011-07-13", 1317.719971],\n    ["2011-07-14", 1308.869995],\n    ["2011-07-15", 1316.140015],\n    ["2011-07-18", 1305.439941],\n    ["2011-07-19", 1326.72998],\n    ["2011-07-20", 1325.839966],\n    ["2011-07-21", 1343.800049],\n    ["2011-07-22", 1345.02002],\n    ["2011-07-25", 1337.430054],\n    ["2011-07-26", 1331.939941],\n    ["2011-07-27", 1304.890015],\n    ["2011-07-28", 1300.670044],\n    ["2011-07-29", 1292.280029],\n    ["2011-08-01", 1286.939941],\n    ["2011-08-02", 1254.050049],\n    ["2011-08-03", 1260.339966],\n    ["2011-08-04", 1200.069946],\n    ["2011-08-05", 1199.380005],\n    ["2011-08-08", 1119.459961],\n    ["2011-08-09", 1172.530029],\n    ["2011-08-10", 1120.76001],\n    ["2011-08-11", 1172.640015],\n    ["2011-08-12", 1178.810059],\n    ["2011-08-15", 1204.48999],\n    ["2011-08-16", 1192.76001],\n    ["2011-08-17", 1193.890015],\n    ["2011-08-18", 1140.650024],\n    ["2011-08-19", 1123.530029],\n    ["2011-08-22", 1123.819946],\n    ["2011-08-23", 1162.349976],\n    ["2011-08-24", 1177.599976],\n    ["2011-08-25", 1159.27002],\n    ["2011-08-26", 1176.800049],\n    ["2011-08-29", 1210.079956],\n    ["2011-08-30", 1212.920044],\n    ["2011-08-31", 1218.890015],\n    ["2011-09-01", 1204.420044],\n    ["2011-09-02", 1173.969971],\n    ["2011-09-06", 1165.23999],\n    ["2011-09-07", 1198.619995],\n    ["2011-09-08", 1185.900024],\n    ["2011-09-09", 1154.22998],\n    ["2011-09-12", 1162.27002],\n    ["2011-09-13", 1172.869995],\n    ["2011-09-14", 1188.680054],\n    ["2011-09-15", 1209.109985],\n    ["2011-09-16", 1216.01001],\n    ["2011-09-19", 1204.089966],\n    ["2011-09-20", 1202.089966],\n    ["2011-09-21", 1166.76001],\n    ["2011-09-22", 1129.560059],\n    ["2011-09-23", 1136.430054],\n    ["2011-09-26", 1162.949951],\n    ["2011-09-27", 1175.380005],\n    ["2011-09-28", 1151.060059],\n    ["2011-09-29", 1160.400024],\n    ["2011-09-30", 1131.420044],\n    ["2011-10-03", 1099.22998],\n    ["2011-10-04", 1123.949951],\n    ["2011-10-05", 1144.030029],\n    ["2011-10-06", 1164.969971],\n    ["2011-10-07", 1155.459961],\n    ["2011-10-10", 1194.890015],\n    ["2011-10-11", 1195.540039],\n    ["2011-10-12", 1207.25],\n    ["2011-10-13", 1203.660034],\n    ["2011-10-14", 1224.579956],\n    ["2011-10-17", 1200.859985],\n    ["2011-10-18", 1225.380005],\n    ["2011-10-19", 1209.880005],\n    ["2011-10-20", 1215.390015],\n    ["2011-10-21", 1238.25],\n    ["2011-10-24", 1254.189941],\n    ["2011-10-25", 1229.050049],\n    ["2011-10-26", 1242.0],\n    ["2011-10-27", 1284.589966],\n    ["2011-10-28", 1285.089966],\n    ["2011-10-31", 1253.300049],\n    ["2011-11-01", 1218.280029],\n    ["2011-11-02", 1237.900024],\n    ["2011-11-03", 1261.150024],\n    ["2011-11-04", 1253.22998],\n    ["2011-11-07", 1261.119995],\n    ["2011-11-08", 1275.920044],\n    ["2011-11-09", 1229.099976],\n    ["2011-11-10", 1239.699951],\n    ["2011-11-11", 1263.849976],\n    ["2011-11-14", 1251.780029],\n    ["2011-11-15", 1257.810059],\n    ["2011-11-16", 1236.910034],\n    ["2011-11-17", 1216.130005],\n    ["2011-11-18", 1215.650024],\n    ["2011-11-21", 1192.97998],\n    ["2011-11-22", 1188.040039],\n    ["2011-11-23", 1161.790039],\n    ["2011-11-25", 1158.670044],\n    ["2011-11-28", 1192.550049],\n    ["2011-11-29", 1195.189941],\n    ["2011-11-30", 1246.959961],\n    ["2011-12-01", 1244.579956],\n    ["2011-12-02", 1244.280029],\n    ["2011-12-05", 1257.079956],\n    ["2011-12-06", 1258.469971],\n    ["2011-12-07", 1261.01001],\n    ["2011-12-08", 1234.349976],\n    ["2011-12-09", 1255.189941],\n    ["2011-12-12", 1236.469971],\n    ["2011-12-13", 1225.72998],\n    ["2011-12-14", 1211.819946],\n    ["2011-12-15", 1215.75],\n    ["2011-12-16", 1219.660034],\n    ["2011-12-19", 1205.349976],\n    ["2011-12-20", 1241.300049],\n    ["2011-12-21", 1243.719971],\n    ["2011-12-22", 1254.0],\n    ["2011-12-23", 1265.329956],\n    ["2011-12-27", 1265.430054],\n    ["2011-12-28", 1249.640015],\n    ["2011-12-29", 1263.02002],\n    ["2011-12-30", 1257.599976],\n    ["2012-01-03", 1277.060059],\n    ["2012-01-04", 1277.300049],\n    ["2012-01-05", 1281.060059],\n    ["2012-01-06", 1277.810059],\n    ["2012-01-09", 1280.699951],\n    ["2012-01-10", 1292.079956],\n    ["2012-01-11", 1292.47998],\n    ["2012-01-12", 1295.5],\n    ["2012-01-13", 1289.089966],\n    ["2012-01-17", 1293.670044],\n    ["2012-01-18", 1308.040039],\n    ["2012-01-19", 1314.5],\n    ["2012-01-20", 1315.380005],\n    ["2012-01-23", 1316.0],\n    ["2012-01-24", 1314.650024],\n    ["2012-01-25", 1326.060059],\n    ["2012-01-26", 1318.430054],\n    ["2012-01-27", 1316.329956],\n    ["2012-01-30", 1313.01001],\n    ["2012-01-31", 1312.410034],\n    ["2012-02-01", 1324.089966],\n    ["2012-02-02", 1325.540039],\n    ["2012-02-03", 1344.900024],\n    ["2012-02-06", 1344.329956],\n    ["2012-02-07", 1347.050049],\n    ["2012-02-08", 1349.959961],\n    ["2012-02-09", 1351.949951],\n    ["2012-02-10", 1342.640015],\n    ["2012-02-13", 1351.77002],\n    ["2012-02-14", 1350.5],\n    ["2012-02-15", 1343.22998],\n    ["2012-02-16", 1358.040039],\n    ["2012-02-17", 1361.22998],\n    ["2012-02-21", 1362.209961],\n    ["2012-02-22", 1357.660034],\n    ["2012-02-23", 1363.459961],\n    ["2012-02-24", 1365.73999],\n    ["2012-02-27", 1367.589966],\n    ["2012-02-28", 1372.180054],\n    ["2012-02-29", 1365.680054],\n    ["2012-03-01", 1374.089966],\n    ["2012-03-02", 1369.630005],\n    ["2012-03-05", 1364.329956],\n    ["2012-03-06", 1343.359985],\n    ["2012-03-07", 1352.630005],\n    ["2012-03-08", 1365.910034],\n    ["2012-03-09", 1370.869995],\n    ["2012-03-12", 1371.089966],\n    ["2012-03-13", 1395.949951],\n    ["2012-03-14", 1394.280029],\n    ["2012-03-15", 1402.599976],\n    ["2012-03-16", 1404.170044],\n    ["2012-03-19", 1409.75],\n    ["2012-03-20", 1405.52002],\n    ["2012-03-21", 1402.890015],\n    ["2012-03-22", 1392.780029],\n    ["2012-03-23", 1397.109985],\n    ["2012-03-26", 1416.51001],\n    ["2012-03-27", 1412.52002],\n    ["2012-03-28", 1405.540039],\n    ["2012-03-29", 1403.280029],\n    ["2012-03-30", 1408.469971],\n    ["2012-04-02", 1419.040039],\n    ["2012-04-03", 1413.380005],\n    ["2012-04-04", 1398.959961],\n    ["2012-04-05", 1398.079956],\n    ["2012-04-09", 1382.199951],\n    ["2012-04-10", 1358.589966],\n    ["2012-04-11", 1368.709961],\n    ["2012-04-12", 1387.569946],\n    ["2012-04-13", 1370.26001],\n    ["2012-04-16", 1369.569946],\n    ["2012-04-17", 1390.780029],\n    ["2012-04-18", 1385.140015],\n    ["2012-04-19", 1376.920044],\n    ["2012-04-20", 1378.530029],\n    ["2012-04-23", 1366.939941],\n    ["2012-04-24", 1371.969971],\n    ["2012-04-25", 1390.689941],\n    ["2012-04-26", 1399.97998],\n    ["2012-04-27", 1403.359985],\n    ["2012-04-30", 1397.910034],\n    ["2012-05-01", 1405.819946],\n    ["2012-05-02", 1402.310059],\n    ["2012-05-03", 1391.569946],\n    ["2012-05-04", 1369.099976],\n    ["2012-05-07", 1369.579956],\n    ["2012-05-08", 1363.719971],\n    ["2012-05-09", 1354.579956],\n    ["2012-05-10", 1357.98999],\n    ["2012-05-11", 1353.390015],\n    ["2012-05-14", 1338.349976],\n    ["2012-05-15", 1330.660034],\n    ["2012-05-16", 1324.800049],\n    ["2012-05-17", 1304.859985],\n    ["2012-05-18", 1295.219971],\n    ["2012-05-21", 1315.98999],\n    ["2012-05-22", 1316.630005],\n    ["2012-05-23", 1318.859985],\n    ["2012-05-24", 1320.680054],\n    ["2012-05-25", 1317.819946],\n    ["2012-05-29", 1332.420044],\n    ["2012-05-30", 1313.319946],\n    ["2012-05-31", 1310.329956],\n    ["2012-06-01", 1278.040039],\n    ["2012-06-04", 1278.180054],\n    ["2012-06-05", 1285.5],\n    ["2012-06-06", 1315.130005],\n    ["2012-06-07", 1314.98999],\n    ["2012-06-08", 1325.660034],\n    ["2012-06-11", 1308.930054],\n    ["2012-06-12", 1324.180054],\n    ["2012-06-13", 1314.880005],\n    ["2012-06-14", 1329.099976],\n    ["2012-06-15", 1342.839966],\n    ["2012-06-18", 1344.780029],\n    ["2012-06-19", 1357.97998],\n    ["2012-06-20", 1355.689941],\n    ["2012-06-21", 1325.51001],\n    ["2012-06-22", 1335.02002],\n    ["2012-06-25", 1313.719971],\n    ["2012-06-26", 1319.98999],\n    ["2012-06-27", 1331.849976],\n    ["2012-06-28", 1329.040039],\n    ["2012-06-29", 1362.160034],\n    ["2012-07-02", 1365.51001],\n    ["2012-07-03", 1374.02002],\n    ["2012-07-05", 1367.579956],\n    ["2012-07-06", 1354.680054],\n    ["2012-07-09", 1352.459961],\n    ["2012-07-10", 1341.469971],\n    ["2012-07-11", 1341.449951],\n    ["2012-07-12", 1334.76001],\n    ["2012-07-13", 1356.780029],\n    ["2012-07-16", 1353.640015],\n    ["2012-07-17", 1363.670044],\n    ["2012-07-18", 1372.780029],\n    ["2012-07-19", 1376.51001],\n    ["2012-07-20", 1362.660034],\n    ["2012-07-23", 1350.52002],\n    ["2012-07-24", 1338.310059],\n    ["2012-07-25", 1337.890015],\n    ["2012-07-26", 1360.02002],\n    ["2012-07-27", 1385.969971],\n    ["2012-07-30", 1385.300049],\n    ["2012-07-31", 1379.319946],\n    ["2012-08-01", 1375.319946],\n    ["2012-08-02", 1365.0],\n    ["2012-08-03", 1390.98999],\n    ["2012-08-06", 1394.22998],\n    ["2012-08-07", 1401.349976],\n    ["2012-08-08", 1402.219971],\n    ["2012-08-09", 1402.800049],\n    ["2012-08-10", 1405.869995],\n    ["2012-08-13", 1404.109985],\n    ["2012-08-14", 1403.930054],\n    ["2012-08-15", 1405.530029],\n    ["2012-08-16", 1415.51001],\n    ["2012-08-17", 1418.160034],\n    ["2012-08-20", 1418.130005],\n    ["2012-08-21", 1413.170044],\n    ["2012-08-22", 1413.48999],\n    ["2012-08-23", 1402.079956],\n    ["2012-08-24", 1411.130005],\n    ["2012-08-27", 1410.439941],\n    ["2012-08-28", 1409.300049],\n    ["2012-08-29", 1410.48999],\n    ["2012-08-30", 1399.47998],\n    ["2012-08-31", 1406.579956],\n    ["2012-09-04", 1404.939941],\n    ["2012-09-05", 1403.439941],\n    ["2012-09-06", 1432.119995],\n    ["2012-09-07", 1437.920044],\n    ["2012-09-10", 1429.079956],\n    ["2012-09-11", 1433.560059],\n    ["2012-09-12", 1436.560059],\n    ["2012-09-13", 1459.98999],\n    ["2012-09-14", 1465.77002],\n    ["2012-09-17", 1461.189941],\n    ["2012-09-18", 1459.319946],\n    ["2012-09-19", 1461.050049],\n    ["2012-09-20", 1460.26001],\n    ["2012-09-21", 1460.150024],\n    ["2012-09-24", 1456.890015],\n    ["2012-09-25", 1441.589966],\n    ["2012-09-26", 1433.319946],\n    ["2012-09-27", 1447.150024],\n    ["2012-09-28", 1440.670044],\n    ["2012-10-01", 1444.48999],\n    ["2012-10-02", 1445.75],\n    ["2012-10-03", 1450.98999],\n    ["2012-10-04", 1461.400024],\n    ["2012-10-05", 1460.930054],\n    ["2012-10-08", 1455.880005],\n    ["2012-10-09", 1441.47998],\n    ["2012-10-10", 1432.560059],\n    ["2012-10-11", 1432.839966],\n    ["2012-10-12", 1428.589966],\n    ["2012-10-15", 1440.130005],\n    ["2012-10-16", 1454.920044],\n    ["2012-10-17", 1460.910034],\n    ["2012-10-18", 1457.339966],\n    ["2012-10-19", 1433.189941],\n    ["2012-10-22", 1433.819946],\n    ["2012-10-23", 1413.109985],\n    ["2012-10-24", 1408.75],\n    ["2012-10-25", 1412.969971],\n    ["2012-10-26", 1411.939941],\n    ["2012-10-31", 1412.160034],\n    ["2012-11-01", 1427.589966],\n    ["2012-11-02", 1414.199951],\n    ["2012-11-05", 1417.26001],\n    ["2012-11-06", 1428.390015],\n    ["2012-11-07", 1394.530029],\n    ["2012-11-08", 1377.51001],\n    ["2012-11-09", 1379.849976],\n    ["2012-11-12", 1380.030029],\n    ["2012-11-13", 1374.530029],\n    ["2012-11-14", 1355.48999],\n    ["2012-11-15", 1353.329956],\n    ["2012-11-16", 1359.880005],\n    ["2012-11-19", 1386.890015],\n    ["2012-11-20", 1387.810059],\n    ["2012-11-21", 1391.030029],\n    ["2012-11-23", 1409.150024],\n    ["2012-11-26", 1406.290039],\n    ["2012-11-27", 1398.939941],\n    ["2012-11-28", 1409.930054],\n    ["2012-11-29", 1415.949951],\n    ["2012-11-30", 1416.180054],\n    ["2012-12-03", 1409.459961],\n    ["2012-12-04", 1407.050049],\n    ["2012-12-05", 1409.280029],\n    ["2012-12-06", 1413.939941],\n    ["2012-12-07", 1418.069946],\n    ["2012-12-10", 1418.550049],\n    ["2012-12-11", 1427.839966],\n    ["2012-12-12", 1428.47998],\n    ["2012-12-13", 1419.449951],\n    ["2012-12-14", 1413.579956],\n    ["2012-12-17", 1430.359985],\n    ["2012-12-18", 1446.790039],\n    ["2012-12-19", 1435.810059],\n    ["2012-12-20", 1443.689941],\n    ["2012-12-21", 1430.150024],\n    ["2012-12-24", 1426.660034],\n    ["2012-12-26", 1419.829956],\n    ["2012-12-27", 1418.099976],\n    ["2012-12-28", 1402.430054],\n    ["2012-12-31", 1426.189941],\n    ["2013-01-02", 1462.420044],\n    ["2013-01-03", 1459.369995],\n    ["2013-01-04", 1466.469971],\n    ["2013-01-07", 1461.890015],\n    ["2013-01-08", 1457.150024],\n    ["2013-01-09", 1461.02002],\n    ["2013-01-10", 1472.119995],\n    ["2013-01-11", 1472.050049],\n    ["2013-01-14", 1470.680054],\n    ["2013-01-15", 1472.339966],\n    ["2013-01-16", 1472.630005],\n    ["2013-01-17", 1480.939941],\n    ["2013-01-18", 1485.97998],\n    ["2013-01-22", 1492.560059],\n    ["2013-01-23", 1494.810059],\n    ["2013-01-24", 1494.819946],\n    ["2013-01-25", 1502.959961],\n    ["2013-01-28", 1500.180054],\n    ["2013-01-29", 1507.839966],\n    ["2013-01-30", 1501.959961],\n    ["2013-01-31", 1498.109985],\n    ["2013-02-01", 1513.170044],\n    ["2013-02-04", 1495.709961],\n    ["2013-02-05", 1511.290039],\n    ["2013-02-06", 1512.119995],\n    ["2013-02-07", 1509.390015],\n    ["2013-02-08", 1517.930054],\n    ["2013-02-11", 1517.01001],\n    ["2013-02-12", 1519.430054],\n    ["2013-02-13", 1520.329956],\n    ["2013-02-14", 1521.380005],\n    ["2013-02-15", 1519.790039],\n    ["2013-02-19", 1530.939941],\n    ["2013-02-20", 1511.949951],\n    ["2013-02-21", 1502.420044],\n    ["2013-02-22", 1515.599976],\n    ["2013-02-25", 1487.849976],\n    ["2013-02-26", 1496.939941],\n    ["2013-02-27", 1515.98999],\n    ["2013-02-28", 1514.680054],\n    ["2013-03-01", 1518.199951],\n    ["2013-03-04", 1525.199951],\n    ["2013-03-05", 1539.790039],\n    ["2013-03-06", 1541.459961],\n    ["2013-03-07", 1544.26001],\n    ["2013-03-08", 1551.180054],\n    ["2013-03-11", 1556.219971],\n    ["2013-03-12", 1552.47998],\n    ["2013-03-13", 1554.52002],\n    ["2013-03-14", 1563.22998],\n    ["2013-03-15", 1560.699951],\n    ["2013-03-18", 1552.099976],\n    ["2013-03-19", 1548.339966],\n    ["2013-03-20", 1558.709961],\n    ["2013-03-21", 1545.800049],\n    ["2013-03-22", 1556.890015],\n    ["2013-03-25", 1551.689941],\n    ["2013-03-26", 1563.77002],\n    ["2013-03-27", 1562.849976],\n    ["2013-03-28", 1569.189941],\n    ["2013-04-01", 1562.170044],\n    ["2013-04-02", 1570.25],\n    ["2013-04-03", 1553.689941],\n    ["2013-04-04", 1559.97998],\n    ["2013-04-05", 1553.280029],\n    ["2013-04-08", 1563.069946],\n    ["2013-04-09", 1568.609985],\n    ["2013-04-10", 1587.72998],\n    ["2013-04-11", 1593.369995],\n    ["2013-04-12", 1588.849976],\n    ["2013-04-15", 1552.359985],\n    ["2013-04-16", 1574.569946],\n    ["2013-04-17", 1552.01001],\n    ["2013-04-18", 1541.609985],\n    ["2013-04-19", 1555.25],\n    ["2013-04-22", 1562.5],\n    ["2013-04-23", 1578.780029],\n    ["2013-04-24", 1578.790039],\n    ["2013-04-25", 1585.160034],\n    ["2013-04-26", 1582.23999],\n    ["2013-04-29", 1593.609985],\n    ["2013-04-30", 1597.569946],\n    ["2013-05-01", 1582.699951],\n    ["2013-05-02", 1597.589966],\n    ["2013-05-03", 1614.420044],\n    ["2013-05-06", 1617.5],\n    ["2013-05-07", 1625.959961],\n    ["2013-05-08", 1632.689941],\n    ["2013-05-09", 1626.670044],\n    ["2013-05-10", 1633.699951],\n    ["2013-05-13", 1633.77002],\n    ["2013-05-14", 1650.339966],\n    ["2013-05-15", 1658.780029],\n    ["2013-05-16", 1650.469971],\n    ["2013-05-17", 1667.469971],\n    ["2013-05-20", 1666.290039],\n    ["2013-05-21", 1669.160034],\n    ["2013-05-22", 1655.349976],\n    ["2013-05-23", 1650.51001],\n    ["2013-05-24", 1649.599976],\n    ["2013-05-28", 1660.060059],\n    ["2013-05-29", 1648.359985],\n    ["2013-05-30", 1654.410034],\n    ["2013-05-31", 1630.73999],\n    ["2013-06-03", 1640.420044],\n    ["2013-06-04", 1631.380005],\n    ["2013-06-05", 1608.900024],\n    ["2013-06-06", 1622.560059],\n    ["2013-06-07", 1643.380005],\n    ["2013-06-10", 1642.810059],\n    ["2013-06-11", 1626.130005],\n    ["2013-06-12", 1612.52002],\n    ["2013-06-13", 1636.359985],\n    ["2013-06-14", 1626.72998],\n    ["2013-06-17", 1639.040039],\n    ["2013-06-18", 1651.810059],\n    ["2013-06-19", 1628.930054],\n    ["2013-06-20", 1588.189941],\n    ["2013-06-21", 1592.430054],\n    ["2013-06-24", 1573.089966],\n    ["2013-06-25", 1588.030029],\n    ["2013-06-26", 1603.26001],\n    ["2013-06-27", 1613.199951],\n    ["2013-06-28", 1606.280029],\n    ["2013-07-01", 1614.959961],\n    ["2013-07-02", 1614.079956],\n    ["2013-07-03", 1615.410034],\n    ["2013-07-05", 1631.890015],\n    ["2013-07-08", 1640.459961],\n    ["2013-07-09", 1652.319946],\n    ["2013-07-10", 1652.619995],\n    ["2013-07-11", 1675.02002],\n    ["2013-07-12", 1680.189941],\n    ["2013-07-15", 1682.5],\n    ["2013-07-16", 1676.26001],\n    ["2013-07-17", 1680.910034],\n    ["2013-07-18", 1689.369995],\n    ["2013-07-19", 1692.089966],\n    ["2013-07-22", 1695.530029],\n    ["2013-07-23", 1692.390015],\n    ["2013-07-24", 1685.939941],\n    ["2013-07-25", 1690.25],\n    ["2013-07-26", 1691.650024],\n    ["2013-07-29", 1685.329956],\n    ["2013-07-30", 1685.959961],\n    ["2013-07-31", 1685.72998],\n    ["2013-08-01", 1706.869995],\n    ["2013-08-02", 1709.670044],\n    ["2013-08-05", 1707.140015],\n    ["2013-08-06", 1697.369995],\n    ["2013-08-07", 1690.910034],\n    ["2013-08-08", 1697.47998],\n    ["2013-08-09", 1691.420044],\n    ["2013-08-12", 1689.469971],\n    ["2013-08-13", 1694.160034],\n    ["2013-08-14", 1685.390015],\n    ["2013-08-15", 1661.319946],\n    ["2013-08-16", 1655.829956],\n    ["2013-08-19", 1646.060059],\n    ["2013-08-20", 1652.349976],\n    ["2013-08-21", 1642.800049],\n    ["2013-08-22", 1656.959961],\n    ["2013-08-23", 1663.5],\n    ["2013-08-26", 1656.780029],\n    ["2013-08-27", 1630.47998],\n    ["2013-08-28", 1634.959961],\n    ["2013-08-29", 1638.170044],\n    ["2013-08-30", 1632.969971],\n    ["2013-09-03", 1639.77002],\n    ["2013-09-04", 1653.079956],\n    ["2013-09-05", 1655.079956],\n    ["2013-09-06", 1655.170044],\n    ["2013-09-09", 1671.709961],\n    ["2013-09-10", 1683.98999],\n    ["2013-09-11", 1689.130005],\n    ["2013-09-12", 1683.420044],\n    ["2013-09-13", 1687.98999],\n    ["2013-09-16", 1697.599976],\n    ["2013-09-17", 1704.76001],\n    ["2013-09-18", 1725.52002],\n    ["2013-09-19", 1722.339966],\n    ["2013-09-20", 1709.910034],\n    ["2013-09-23", 1701.839966],\n    ["2013-09-24", 1697.420044],\n    ["2013-09-25", 1692.77002],\n    ["2013-09-26", 1698.670044],\n    ["2013-09-27", 1691.75],\n    ["2013-09-30", 1681.550049],\n    ["2013-10-01", 1695.0],\n    ["2013-10-02", 1693.869995],\n    ["2013-10-03", 1678.660034],\n    ["2013-10-04", 1690.5],\n    ["2013-10-07", 1676.119995],\n    ["2013-10-08", 1655.449951],\n    ["2013-10-09", 1656.400024],\n    ["2013-10-10", 1692.560059],\n    ["2013-10-11", 1703.199951],\n    ["2013-10-14", 1710.140015],\n    ["2013-10-15", 1698.060059],\n    ["2013-10-16", 1721.540039],\n    ["2013-10-17", 1733.150024],\n    ["2013-10-18", 1744.5],\n    ["2013-10-21", 1744.660034],\n    ["2013-10-22", 1754.670044],\n    ["2013-10-23", 1746.380005],\n    ["2013-10-24", 1752.069946],\n    ["2013-10-25", 1759.77002],\n    ["2013-10-28", 1762.109985],\n    ["2013-10-29", 1771.949951],\n    ["2013-10-30", 1763.310059],\n    ["2013-10-31", 1756.540039],\n    ["2013-11-01", 1761.640015],\n    ["2013-11-04", 1767.930054],\n    ["2013-11-05", 1762.969971],\n    ["2013-11-06", 1770.48999],\n    ["2013-11-07", 1747.150024],\n    ["2013-11-08", 1770.609985],\n    ["2013-11-11", 1771.890015],\n    ["2013-11-12", 1767.689941],\n    ["2013-11-13", 1782.0],\n    ["2013-11-14", 1790.619995],\n    ["2013-11-15", 1798.180054],\n    ["2013-11-18", 1791.530029],\n    ["2013-11-19", 1787.869995],\n    ["2013-11-20", 1781.369995],\n    ["2013-11-21", 1795.849976],\n    ["2013-11-22", 1804.76001],\n    ["2013-11-25", 1802.47998],\n    ["2013-11-26", 1802.75],\n    ["2013-11-27", 1807.22998],\n    ["2013-11-29", 1805.810059],\n    ["2013-12-02", 1800.900024],\n    ["2013-12-03", 1795.150024],\n    ["2013-12-04", 1792.810059],\n    ["2013-12-05", 1785.030029],\n    ["2013-12-06", 1805.089966],\n    ["2013-12-09", 1808.369995],\n    ["2013-12-10", 1802.619995],\n    ["2013-12-11", 1782.219971],\n    ["2013-12-12", 1775.5],\n    ["2013-12-13", 1775.319946],\n    ["2013-12-16", 1786.540039],\n    ["2013-12-17", 1781.0],\n    ["2013-12-18", 1810.650024],\n    ["2013-12-19", 1809.599976],\n    ["2013-12-20", 1818.319946],\n    ["2013-12-23", 1827.98999],\n    ["2013-12-24", 1833.319946],\n    ["2013-12-26", 1842.02002],\n    ["2013-12-27", 1841.400024],\n    ["2013-12-30", 1841.069946],\n    ["2013-12-31", 1848.359985],\n    ["2014-01-02", 1831.97998],\n    ["2014-01-03", 1831.369995],\n    ["2014-01-06", 1826.77002],\n    ["2014-01-07", 1837.880005],\n    ["2014-01-08", 1837.48999],\n    ["2014-01-09", 1838.130005],\n    ["2014-01-10", 1842.369995],\n    ["2014-01-13", 1819.199951],\n    ["2014-01-14", 1838.880005],\n    ["2014-01-15", 1848.380005],\n    ["2014-01-16", 1845.890015],\n    ["2014-01-17", 1838.699951],\n    ["2014-01-21", 1843.800049],\n    ["2014-01-22", 1844.859985],\n    ["2014-01-23", 1828.459961],\n    ["2014-01-24", 1790.290039],\n    ["2014-01-27", 1781.560059],\n    ["2014-01-28", 1792.5],\n    ["2014-01-29", 1774.199951],\n    ["2014-01-30", 1794.189941],\n    ["2014-01-31", 1782.589966],\n    ["2014-02-03", 1741.890015],\n    ["2014-02-04", 1755.199951],\n    ["2014-02-05", 1751.640015],\n    ["2014-02-06", 1773.430054],\n    ["2014-02-07", 1797.02002],\n    ["2014-02-10", 1799.839966],\n    ["2014-02-11", 1819.75],\n    ["2014-02-12", 1819.26001],\n    ["2014-02-13", 1829.829956],\n    ["2014-02-14", 1838.630005],\n    ["2014-02-18", 1840.76001],\n    ["2014-02-19", 1828.75],\n    ["2014-02-20", 1839.780029],\n    ["2014-02-21", 1836.25],\n    ["2014-02-24", 1847.609985],\n    ["2014-02-25", 1845.119995],\n    ["2014-02-26", 1845.160034],\n    ["2014-02-27", 1854.290039],\n    ["2014-02-28", 1859.449951],\n    ["2014-03-03", 1845.72998],\n    ["2014-03-04", 1873.910034],\n    ["2014-03-05", 1873.810059],\n    ["2014-03-06", 1877.030029],\n    ["2014-03-07", 1878.040039],\n    ["2014-03-10", 1877.170044],\n    ["2014-03-11", 1867.630005],\n    ["2014-03-12", 1868.199951],\n    ["2014-03-13", 1846.339966],\n    ["2014-03-14", 1841.130005],\n    ["2014-03-17", 1858.829956],\n    ["2014-03-18", 1872.25],\n    ["2014-03-19", 1860.77002],\n    ["2014-03-20", 1872.01001],\n    ["2014-03-21", 1866.52002],\n    ["2014-03-24", 1857.439941],\n    ["2014-03-25", 1865.619995],\n    ["2014-03-26", 1852.560059],\n    ["2014-03-27", 1849.040039],\n    ["2014-03-28", 1857.619995],\n    ["2014-03-31", 1872.339966],\n    ["2014-04-01", 1885.52002],\n    ["2014-04-02", 1890.900024],\n    ["2014-04-03", 1888.77002],\n    ["2014-04-04", 1865.089966],\n    ["2014-04-07", 1845.040039],\n    ["2014-04-08", 1851.959961],\n    ["2014-04-09", 1872.180054],\n    ["2014-04-10", 1833.079956],\n    ["2014-04-11", 1815.689941],\n    ["2014-04-14", 1830.609985],\n    ["2014-04-15", 1842.97998],\n    ["2014-04-16", 1862.310059],\n    ["2014-04-17", 1864.849976],\n    ["2014-04-21", 1871.890015],\n    ["2014-04-22", 1879.550049],\n    ["2014-04-23", 1875.390015],\n    ["2014-04-24", 1878.609985],\n    ["2014-04-25", 1863.400024],\n    ["2014-04-28", 1869.430054],\n    ["2014-04-29", 1878.329956],\n    ["2014-04-30", 1883.949951],\n    ["2014-05-01", 1883.680054],\n    ["2014-05-02", 1881.140015],\n    ["2014-05-05", 1884.660034],\n    ["2014-05-06", 1867.719971],\n    ["2014-05-07", 1878.209961],\n    ["2014-05-08", 1875.630005],\n    ["2014-05-09", 1878.47998],\n    ["2014-05-12", 1896.650024],\n    ["2014-05-13", 1897.449951],\n    ["2014-05-14", 1888.530029],\n    ["2014-05-15", 1870.849976],\n    ["2014-05-16", 1877.859985],\n    ["2014-05-19", 1885.079956],\n    ["2014-05-20", 1872.829956],\n    ["2014-05-21", 1888.030029],\n    ["2014-05-22", 1892.48999],\n    ["2014-05-23", 1900.530029],\n    ["2014-05-27", 1911.910034],\n    ["2014-05-28", 1909.780029],\n    ["2014-05-29", 1920.030029],\n    ["2014-05-30", 1923.569946],\n    ["2014-06-02", 1924.969971],\n    ["2014-06-03", 1924.23999],\n    ["2014-06-04", 1927.880005],\n    ["2014-06-05", 1940.459961],\n    ["2014-06-06", 1949.439941],\n    ["2014-06-09", 1951.27002],\n    ["2014-06-10", 1950.790039],\n    ["2014-06-11", 1943.890015],\n    ["2014-06-12", 1930.109985],\n    ["2014-06-13", 1936.160034],\n    ["2014-06-16", 1937.780029],\n    ["2014-06-17", 1941.98999],\n    ["2014-06-18", 1956.97998],\n    ["2014-06-19", 1959.47998],\n    ["2014-06-20", 1962.869995],\n    ["2014-06-23", 1962.609985],\n    ["2014-06-24", 1949.97998],\n    ["2014-06-25", 1959.530029],\n    ["2014-06-26", 1957.219971],\n    ["2014-06-27", 1960.959961],\n    ["2014-06-30", 1960.22998],\n    ["2014-07-01", 1973.319946],\n    ["2014-07-02", 1974.619995],\n    ["2014-07-03", 1985.439941],\n    ["2014-07-07", 1977.650024],\n    ["2014-07-08", 1963.709961],\n    ["2014-07-09", 1972.829956],\n    ["2014-07-10", 1964.680054],\n    ["2014-07-11", 1967.569946],\n    ["2014-07-14", 1977.099976],\n    ["2014-07-15", 1973.280029],\n    ["2014-07-16", 1981.569946],\n    ["2014-07-17", 1958.119995],\n    ["2014-07-18", 1978.219971],\n    ["2014-07-21", 1973.630005],\n    ["2014-07-22", 1983.530029],\n    ["2014-07-23", 1987.01001],\n    ["2014-07-24", 1987.97998],\n    ["2014-07-25", 1978.339966],\n    ["2014-07-28", 1978.910034],\n    ["2014-07-29", 1969.949951],\n    ["2014-07-30", 1970.069946],\n    ["2014-07-31", 1930.670044],\n    ["2014-08-01", 1925.150024],\n    ["2014-08-04", 1938.98999],\n    ["2014-08-05", 1920.209961],\n    ["2014-08-06", 1920.23999],\n    ["2014-08-07", 1909.569946],\n    ["2014-08-08", 1931.589966],\n    ["2014-08-11", 1936.920044],\n    ["2014-08-12", 1933.75],\n    ["2014-08-13", 1946.719971],\n    ["2014-08-14", 1955.180054],\n    ["2014-08-15", 1955.060059],\n    ["2014-08-18", 1971.73999],\n    ["2014-08-19", 1981.599976],\n    ["2014-08-20", 1986.51001],\n    ["2014-08-21", 1992.369995],\n    ["2014-08-22", 1988.400024],\n    ["2014-08-25", 1997.920044],\n    ["2014-08-26", 2000.02002],\n    ["2014-08-27", 2000.119995],\n    ["2014-08-28", 1996.73999],\n    ["2014-08-29", 2003.369995],\n    ["2014-09-02", 2002.280029],\n    ["2014-09-03", 2000.719971],\n    ["2014-09-04", 1997.650024],\n    ["2014-09-05", 2007.709961],\n    ["2014-09-08", 2001.540039],\n    ["2014-09-09", 1988.439941],\n    ["2014-09-10", 1995.689941],\n    ["2014-09-11", 1997.449951],\n    ["2014-09-12", 1985.540039],\n    ["2014-09-15", 1984.130005],\n    ["2014-09-16", 1998.97998],\n    ["2014-09-17", 2001.569946],\n    ["2014-09-18", 2011.359985],\n    ["2014-09-19", 2010.400024],\n    ["2014-09-22", 1994.290039],\n    ["2014-09-23", 1982.77002],\n    ["2014-09-24", 1998.300049],\n    ["2014-09-25", 1965.98999],\n    ["2014-09-26", 1982.849976],\n    ["2014-09-29", 1977.800049],\n    ["2014-09-30", 1972.290039],\n    ["2014-10-01", 1946.160034],\n    ["2014-10-02", 1946.170044],\n    ["2014-10-03", 1967.900024],\n    ["2014-10-06", 1964.819946],\n    ["2014-10-07", 1935.099976],\n    ["2014-10-08", 1968.890015],\n    ["2014-10-09", 1928.209961],\n    ["2014-10-10", 1906.130005],\n    ["2014-10-13", 1874.73999],\n    ["2014-10-14", 1877.699951],\n    ["2014-10-15", 1862.48999],\n    ["2014-10-16", 1862.76001],\n    ["2014-10-17", 1886.76001],\n    ["2014-10-20", 1904.01001],\n    ["2014-10-21", 1941.280029],\n    ["2014-10-22", 1927.109985],\n    ["2014-10-23", 1950.819946],\n    ["2014-10-24", 1964.579956],\n    ["2014-10-27", 1961.630005],\n    ["2014-10-28", 1985.050049],\n    ["2014-10-29", 1982.300049],\n    ["2014-10-30", 1994.650024],\n    ["2014-10-31", 2018.050049],\n    ["2014-11-03", 2017.810059],\n    ["2014-11-04", 2012.099976],\n    ["2014-11-05", 2023.569946],\n    ["2014-11-06", 2031.209961],\n    ["2014-11-07", 2031.920044],\n    ["2014-11-10", 2038.26001],\n    ["2014-11-11", 2039.680054],\n    ["2014-11-12", 2038.25],\n    ["2014-11-13", 2039.329956],\n    ["2014-11-14", 2039.819946],\n    ["2014-11-17", 2041.319946],\n    ["2014-11-18", 2051.800049],\n    ["2014-11-19", 2048.719971],\n    ["2014-11-20", 2052.75],\n    ["2014-11-21", 2063.5],\n    ["2014-11-24", 2069.409912],\n    ["2014-11-25", 2067.030029],\n    ["2014-11-26", 2072.830078],\n    ["2014-11-28", 2067.560059],\n    ["2014-12-01", 2053.439941],\n    ["2014-12-02", 2066.550049],\n    ["2014-12-03", 2074.330078],\n    ["2014-12-04", 2071.919922],\n    ["2014-12-05", 2075.370117],\n    ["2014-12-08", 2060.310059],\n    ["2014-12-09", 2059.820068],\n    ["2014-12-10", 2026.140015],\n    ["2014-12-11", 2035.329956],\n    ["2014-12-12", 2002.329956],\n    ["2014-12-15", 1989.630005],\n    ["2014-12-16", 1972.73999],\n    ["2014-12-17", 2012.890015],\n    ["2014-12-18", 2061.22998],\n    ["2014-12-19", 2070.649902],\n    ["2014-12-22", 2078.540039],\n    ["2014-12-23", 2082.169922],\n    ["2014-12-24", 2081.879883],\n    ["2014-12-26", 2088.77002],\n    ["2014-12-29", 2090.570068],\n    ["2014-12-30", 2080.350098],\n    ["2014-12-31", 2058.899902],\n    ["2015-01-02", 2058.199951],\n    ["2015-01-05", 2020.579956],\n    ["2015-01-06", 2002.609985],\n    ["2015-01-07", 2025.900024],\n    ["2015-01-08", 2062.139893],\n    ["2015-01-09", 2044.810059],\n    ["2015-01-12", 2028.26001],\n    ["2015-01-13", 2023.030029],\n    ["2015-01-14", 2011.27002],\n    ["2015-01-15", 1992.670044],\n    ["2015-01-16", 2019.420044],\n    ["2015-01-20", 2022.550049],\n    ["2015-01-21", 2032.119995],\n    ["2015-01-22", 2063.149902],\n    ["2015-01-23", 2051.820068],\n    ["2015-01-26", 2057.090088],\n    ["2015-01-27", 2029.550049],\n    ["2015-01-28", 2002.160034],\n    ["2015-01-29", 2021.25],\n    ["2015-01-30", 1994.98999],\n    ["2015-02-02", 2020.849976],\n    ["2015-02-03", 2050.030029],\n    ["2015-02-04", 2041.51001],\n    ["2015-02-05", 2062.52002],\n    ["2015-02-06", 2055.469971],\n    ["2015-02-09", 2046.73999],\n    ["2015-02-10", 2068.590088],\n    ["2015-02-11", 2068.530029],\n    ["2015-02-12", 2088.47998],\n    ["2015-02-13", 2096.98999],\n    ["2015-02-17", 2100.340088],\n    ["2015-02-18", 2099.679932],\n    ["2015-02-19", 2097.449951],\n    ["2015-02-20", 2110.300049],\n    ["2015-02-23", 2109.659912],\n    ["2015-02-24", 2115.47998],\n    ["2015-02-25", 2113.860107],\n    ["2015-02-26", 2110.73999],\n    ["2015-02-27", 2104.5],\n    ["2015-03-02", 2117.389893],\n    ["2015-03-03", 2107.780029],\n    ["2015-03-04", 2098.530029],\n    ["2015-03-05", 2101.040039],\n    ["2015-03-06", 2071.26001],\n    ["2015-03-09", 2079.429932],\n    ["2015-03-10", 2044.160034],\n    ["2015-03-11", 2040.23999],\n    ["2015-03-12", 2065.949951],\n    ["2015-03-13", 2053.399902],\n    ["2015-03-16", 2081.189941],\n    ["2015-03-17", 2074.280029],\n    ["2015-03-18", 2099.5],\n    ["2015-03-19", 2089.27002],\n    ["2015-03-20", 2108.100098],\n    ["2015-03-23", 2104.419922],\n    ["2015-03-24", 2091.5],\n    ["2015-03-25", 2061.050049],\n    ["2015-03-26", 2056.149902],\n    ["2015-03-27", 2061.02002],\n    ["2015-03-30", 2086.23999],\n    ["2015-03-31", 2067.889893],\n    ["2015-04-01", 2059.689941],\n    ["2015-04-02", 2066.959961],\n    ["2015-04-06", 2080.620117],\n    ["2015-04-07", 2076.330078],\n    ["2015-04-08", 2081.899902],\n    ["2015-04-09", 2091.179932],\n    ["2015-04-10", 2102.060059],\n    ["2015-04-13", 2092.429932],\n    ["2015-04-14", 2095.840088],\n    ["2015-04-15", 2106.629883],\n    ["2015-04-16", 2104.98999],\n    ["2015-04-17", 2081.179932],\n    ["2015-04-20", 2100.399902],\n    ["2015-04-21", 2097.290039],\n    ["2015-04-22", 2107.959961],\n    ["2015-04-23", 2112.929932],\n    ["2015-04-24", 2117.689941],\n    ["2015-04-27", 2108.919922],\n    ["2015-04-28", 2114.76001],\n    ["2015-04-29", 2106.850098],\n    ["2015-04-30", 2085.51001],\n    ["2015-05-01", 2108.290039],\n    ["2015-05-04", 2114.48999],\n    ["2015-05-05", 2089.459961],\n    ["2015-05-06", 2080.149902],\n    ["2015-05-07", 2088.0],\n    ["2015-05-08", 2116.100098],\n    ["2015-05-11", 2105.330078],\n    ["2015-05-12", 2099.120117],\n    ["2015-05-13", 2098.47998],\n    ["2015-05-14", 2121.100098],\n    ["2015-05-15", 2122.72998],\n    ["2015-05-18", 2129.199951],\n    ["2015-05-19", 2127.830078],\n    ["2015-05-20", 2125.850098],\n    ["2015-05-21", 2130.820068],\n    ["2015-05-22", 2126.060059],\n    ["2015-05-26", 2104.199951],\n    ["2015-05-27", 2123.47998],\n    ["2015-05-28", 2120.790039],\n    ["2015-05-29", 2107.389893],\n    ["2015-06-01", 2111.72998],\n    ["2015-06-02", 2109.600098],\n    ["2015-06-03", 2114.070068],\n    ["2015-06-04", 2095.840088],\n    ["2015-06-05", 2092.830078],\n    ["2015-06-08", 2079.280029],\n    ["2015-06-09", 2080.149902],\n    ["2015-06-10", 2105.199951],\n    ["2015-06-11", 2108.860107],\n    ["2015-06-12", 2094.110107],\n    ["2015-06-15", 2084.429932],\n    ["2015-06-16", 2096.290039],\n    ["2015-06-17", 2100.439941],\n    ["2015-06-18", 2121.23999],\n    ["2015-06-19", 2109.98999],\n    ["2015-06-22", 2122.850098],\n    ["2015-06-23", 2124.199951],\n    ["2015-06-24", 2108.580078],\n    ["2015-06-25", 2102.310059],\n    ["2015-06-26", 2101.48999],\n    ["2015-06-29", 2057.639893],\n    ["2015-06-30", 2063.110107],\n    ["2015-07-01", 2077.419922],\n    ["2015-07-02", 2076.780029],\n    ["2015-07-06", 2068.76001],\n    ["2015-07-07", 2081.340088],\n    ["2015-07-08", 2046.680054],\n    ["2015-07-09", 2051.310059],\n    ["2015-07-10", 2076.620117],\n    ["2015-07-13", 2099.600098],\n    ["2015-07-14", 2108.949951],\n    ["2015-07-15", 2107.399902],\n    ["2015-07-16", 2124.290039],\n    ["2015-07-17", 2126.639893],\n    ["2015-07-20", 2128.280029],\n    ["2015-07-21", 2119.209961],\n    ["2015-07-22", 2114.149902],\n    ["2015-07-23", 2102.149902],\n    ["2015-07-24", 2079.649902],\n    ["2015-07-27", 2067.639893],\n    ["2015-07-28", 2093.25],\n    ["2015-07-29", 2108.570068],\n    ["2015-07-30", 2108.629883],\n    ["2015-07-31", 2103.840088],\n    ["2015-08-03", 2098.040039],\n    ["2015-08-04", 2093.320068],\n    ["2015-08-05", 2099.840088],\n    ["2015-08-06", 2083.560059],\n    ["2015-08-07", 2077.570068],\n    ["2015-08-10", 2104.179932],\n    ["2015-08-11", 2084.070068],\n    ["2015-08-12", 2086.050049],\n    ["2015-08-13", 2083.389893],\n    ["2015-08-14", 2091.540039],\n    ["2015-08-17", 2102.439941],\n    ["2015-08-18", 2096.919922],\n    ["2015-08-19", 2079.610107],\n    ["2015-08-20", 2035.72998],\n    ["2015-08-21", 1970.890015],\n    ["2015-08-24", 1893.209961],\n    ["2015-08-25", 1867.609985],\n    ["2015-08-26", 1940.51001],\n    ["2015-08-27", 1987.660034],\n    ["2015-08-28", 1988.869995],\n    ["2015-08-31", 1972.180054],\n    ["2015-09-01", 1913.849976],\n    ["2015-09-02", 1948.859985],\n    ["2015-09-03", 1951.130005],\n    ["2015-09-04", 1921.219971],\n    ["2015-09-08", 1969.410034],\n    ["2015-09-09", 1942.040039],\n    ["2015-09-10", 1952.290039],\n    ["2015-09-11", 1961.050049],\n    ["2015-09-14", 1953.030029],\n    ["2015-09-15", 1978.089966],\n    ["2015-09-16", 1995.310059],\n    ["2015-09-17", 1990.199951],\n    ["2015-09-18", 1958.030029],\n    ["2015-09-21", 1966.969971],\n    ["2015-09-22", 1942.73999],\n    ["2015-09-23", 1938.76001],\n    ["2015-09-24", 1932.23999],\n    ["2015-09-25", 1931.339966],\n    ["2015-09-28", 1881.77002],\n    ["2015-09-29", 1884.089966],\n    ["2015-09-30", 1920.030029],\n    ["2015-10-01", 1923.819946],\n    ["2015-10-02", 1951.359985],\n    ["2015-10-05", 1987.050049],\n    ["2015-10-06", 1979.920044],\n    ["2015-10-07", 1995.829956],\n    ["2015-10-08", 2013.430054],\n    ["2015-10-09", 2014.890015],\n    ["2015-10-12", 2017.459961],\n    ["2015-10-13", 2003.689941],\n    ["2015-10-14", 1994.23999],\n    ["2015-10-15", 2023.859985],\n    ["2015-10-16", 2033.109985],\n    ["2015-10-19", 2033.660034],\n    ["2015-10-20", 2030.77002],\n    ["2015-10-21", 2018.939941],\n    ["2015-10-22", 2052.51001],\n    ["2015-10-23", 2075.149902],\n    ["2015-10-26", 2071.179932],\n    ["2015-10-27", 2065.889893],\n    ["2015-10-28", 2090.350098],\n    ["2015-10-29", 2089.409912],\n    ["2015-10-30", 2079.360107],\n    ["2015-11-02", 2104.050049],\n    ["2015-11-03", 2109.790039],\n    ["2015-11-04", 2102.310059],\n    ["2015-11-05", 2099.929932],\n    ["2015-11-06", 2099.199951],\n    ["2015-11-09", 2078.580078],\n    ["2015-11-10", 2081.719971],\n    ["2015-11-11", 2075.0],\n    ["2015-11-12", 2045.969971],\n    ["2015-11-13", 2023.040039],\n    ["2015-11-16", 2053.189941],\n    ["2015-11-17", 2050.439941],\n    ["2015-11-18", 2083.580078],\n    ["2015-11-19", 2081.23999],\n    ["2015-11-20", 2089.169922],\n    ["2015-11-23", 2086.590088],\n    ["2015-11-24", 2089.139893],\n    ["2015-11-25", 2088.870117],\n    ["2015-11-27", 2090.110107],\n    ["2015-11-30", 2080.409912],\n    ["2015-12-01", 2102.629883],\n    ["2015-12-02", 2079.51001],\n    ["2015-12-03", 2049.620117],\n    ["2015-12-04", 2091.689941],\n    ["2015-12-07", 2077.070068],\n    ["2015-12-08", 2063.590088],\n    ["2015-12-09", 2047.619995],\n    ["2015-12-10", 2052.22998],\n    ["2015-12-11", 2012.369995],\n    ["2015-12-14", 2021.939941],\n    ["2015-12-15", 2043.410034],\n    ["2015-12-16", 2073.070068],\n    ["2015-12-17", 2041.890015],\n    ["2015-12-18", 2005.550049],\n    ["2015-12-21", 2021.150024],\n    ["2015-12-22", 2038.969971],\n    ["2015-12-23", 2064.290039],\n    ["2015-12-24", 2060.98999],\n    ["2015-12-28", 2056.5],\n    ["2015-12-29", 2078.360107],\n    ["2015-12-30", 2063.360107],\n    ["2015-12-31", 2043.939941],\n    ["2016-01-04", 2012.660034],\n    ["2016-01-05", 2016.709961],\n    ["2016-01-06", 1990.26001],\n    ["2016-01-07", 1943.089966],\n    ["2016-01-08", 1922.030029],\n    ["2016-01-11", 1923.670044],\n    ["2016-01-12", 1938.680054],\n    ["2016-01-13", 1890.280029],\n    ["2016-01-14", 1921.839966],\n    ["2016-01-15", 1880.329956],\n    ["2016-01-19", 1881.329956],\n    ["2016-01-20", 1859.329956],\n    ["2016-01-21", 1868.98999],\n    ["2016-01-22", 1906.900024],\n    ["2016-01-25", 1877.079956],\n    ["2016-01-26", 1903.630005],\n    ["2016-01-27", 1882.949951],\n    ["2016-01-28", 1893.359985],\n    ["2016-01-29", 1940.23999],\n    ["2016-02-01", 1939.380005],\n    ["2016-02-02", 1903.030029],\n    ["2016-02-03", 1912.530029],\n    ["2016-02-04", 1915.449951],\n    ["2016-02-05", 1880.050049],\n    ["2016-02-08", 1853.439941],\n    ["2016-02-09", 1852.209961],\n    ["2016-02-10", 1851.859985],\n    ["2016-02-11", 1829.079956],\n    ["2016-02-12", 1864.780029],\n    ["2016-02-16", 1895.579956],\n    ["2016-02-17", 1926.819946],\n    ["2016-02-18", 1917.829956],\n    ["2016-02-19", 1917.780029],\n    ["2016-02-22", 1945.5],\n    ["2016-02-23", 1921.27002],\n    ["2016-02-24", 1929.800049],\n    ["2016-02-25", 1951.699951],\n    ["2016-02-26", 1948.050049],\n    ["2016-02-29", 1932.22998],\n    ["2016-03-01", 1978.349976],\n    ["2016-03-02", 1986.449951],\n    ["2016-03-03", 1993.400024],\n    ["2016-03-04", 1999.98999],\n    ["2016-03-07", 2001.76001],\n    ["2016-03-08", 1979.26001],\n    ["2016-03-09", 1989.26001],\n    ["2016-03-10", 1989.569946],\n    ["2016-03-11", 2022.189941],\n    ["2016-03-14", 2019.640015],\n    ["2016-03-15", 2015.930054],\n    ["2016-03-16", 2027.219971],\n    ["2016-03-17", 2040.589966],\n    ["2016-03-18", 2049.580078],\n    ["2016-03-21", 2051.600098],\n    ["2016-03-22", 2049.800049],\n    ["2016-03-23", 2036.709961],\n    ["2016-03-24", 2035.939941],\n    ["2016-03-28", 2037.050049],\n    ["2016-03-29", 2055.01001],\n    ["2016-03-30", 2063.949951],\n    ["2016-03-31", 2059.73999],\n    ["2016-04-01", 2072.780029],\n    ["2016-04-04", 2066.129883],\n    ["2016-04-05", 2045.170044],\n    ["2016-04-06", 2066.659912],\n    ["2016-04-07", 2041.910034],\n    ["2016-04-08", 2047.599976],\n    ["2016-04-11", 2041.98999],\n    ["2016-04-12", 2061.719971],\n    ["2016-04-13", 2082.419922],\n    ["2016-04-14", 2082.780029],\n    ["2016-04-15", 2080.72998],\n    ["2016-04-18", 2094.340088],\n    ["2016-04-19", 2100.800049],\n    ["2016-04-20", 2102.399902],\n    ["2016-04-21", 2091.47998],\n    ["2016-04-22", 2091.580078],\n    ["2016-04-25", 2087.790039],\n    ["2016-04-26", 2091.699951],\n    ["2016-04-27", 2095.149902],\n    ["2016-04-28", 2075.810059],\n    ["2016-04-29", 2065.300049],\n    ["2016-05-02", 2081.429932],\n    ["2016-05-03", 2063.370117],\n    ["2016-05-04", 2051.120117],\n    ["2016-05-05", 2050.629883],\n    ["2016-05-06", 2057.139893],\n    ["2016-05-09", 2058.689941],\n    ["2016-05-10", 2084.389893],\n    ["2016-05-11", 2064.459961],\n    ["2016-05-12", 2064.110107],\n    ["2016-05-13", 2046.609985],\n    ["2016-05-16", 2066.659912],\n    ["2016-05-17", 2047.209961],\n    ["2016-05-18", 2047.630005],\n    ["2016-05-19", 2040.040039],\n    ["2016-05-20", 2052.320068],\n    ["2016-05-23", 2048.040039],\n    ["2016-05-24", 2076.060059],\n    ["2016-05-25", 2090.540039],\n    ["2016-05-26", 2090.100098],\n    ["2016-05-27", 2099.060059],\n    ["2016-05-31", 2096.949951],\n    ["2016-06-01", 2099.330078],\n    ["2016-06-02", 2105.26001],\n    ["2016-06-03", 2099.129883],\n    ["2016-06-06", 2109.409912],\n    ["2016-06-07", 2112.129883],\n    ["2016-06-08", 2119.120117],\n    ["2016-06-09", 2115.47998],\n    ["2016-06-10", 2096.070068],\n    ["2016-06-13", 2079.060059],\n    ["2016-06-14", 2075.320068],\n    ["2016-06-15", 2071.5],\n    ["2016-06-16", 2077.98999],\n    ["2016-06-17", 2071.219971],\n    ["2016-06-20", 2083.25],\n    ["2016-06-21", 2088.899902],\n    ["2016-06-22", 2085.449951],\n    ["2016-06-23", 2113.320068],\n    ["2016-06-24", 2037.410034],\n    ["2016-06-27", 2000.540039],\n    ["2016-06-28", 2036.089966],\n    ["2016-06-29", 2070.77002],\n    ["2016-06-30", 2098.860107],\n    ["2016-07-01", 2102.949951],\n    ["2016-07-05", 2088.550049],\n    ["2016-07-06", 2099.72998],\n    ["2016-07-07", 2097.899902],\n    ["2016-07-08", 2129.899902],\n    ["2016-07-11", 2137.159912],\n    ["2016-07-12", 2152.139893],\n    ["2016-07-13", 2152.429932],\n    ["2016-07-14", 2163.75],\n    ["2016-07-15", 2161.73999],\n    ["2016-07-18", 2166.889893],\n    ["2016-07-19", 2163.780029],\n    ["2016-07-20", 2173.02002],\n    ["2016-07-21", 2165.169922],\n    ["2016-07-22", 2175.030029],\n    ["2016-07-25", 2168.47998],\n    ["2016-07-26", 2169.179932],\n    ["2016-07-27", 2166.580078],\n    ["2016-07-28", 2170.060059],\n    ["2016-07-29", 2173.600098],\n    ["2016-08-01", 2170.840088],\n    ["2016-08-02", 2157.030029],\n    ["2016-08-03", 2163.790039],\n    ["2016-08-04", 2164.25],\n    ["2016-08-05", 2182.870117],\n    ["2016-08-08", 2180.889893],\n    ["2016-08-09", 2181.73999],\n    ["2016-08-10", 2175.48999],\n    ["2016-08-11", 2185.790039],\n    ["2016-08-12", 2184.050049],\n    ["2016-08-15", 2190.149902],\n    ["2016-08-16", 2178.149902],\n    ["2016-08-17", 2182.219971],\n    ["2016-08-18", 2187.02002],\n    ["2016-08-19", 2183.870117],\n    ["2016-08-22", 2182.639893],\n    ["2016-08-23", 2186.899902],\n    ["2016-08-24", 2175.439941],\n    ["2016-08-25", 2172.469971],\n    ["2016-08-26", 2169.040039],\n    ["2016-08-29", 2180.379883],\n    ["2016-08-30", 2176.120117],\n    ["2016-08-31", 2170.949951],\n    ["2016-09-01", 2170.860107],\n    ["2016-09-02", 2179.97998],\n    ["2016-09-06", 2186.47998],\n    ["2016-09-07", 2186.159912],\n    ["2016-09-08", 2181.300049],\n    ["2016-09-09", 2127.810059],\n    ["2016-09-12", 2159.040039],\n    ["2016-09-13", 2127.02002],\n    ["2016-09-14", 2125.77002],\n    ["2016-09-15", 2147.26001],\n    ["2016-09-16", 2139.159912],\n    ["2016-09-19", 2139.120117],\n    ["2016-09-20", 2139.76001],\n    ["2016-09-21", 2163.120117],\n    ["2016-09-22", 2177.179932],\n    ["2016-09-23", 2164.689941],\n    ["2016-09-26", 2146.100098],\n    ["2016-09-27", 2159.929932],\n    ["2016-09-28", 2171.370117],\n    ["2016-09-29", 2151.129883],\n    ["2016-09-30", 2168.27002],\n    ["2016-10-03", 2161.199951],\n    ["2016-10-04", 2150.48999],\n    ["2016-10-05", 2159.72998],\n    ["2016-10-06", 2160.77002],\n    ["2016-10-07", 2153.73999],\n    ["2016-10-10", 2163.659912],\n    ["2016-10-11", 2136.72998],\n    ["2016-10-12", 2139.179932],\n    ["2016-10-13", 2132.550049],\n    ["2016-10-14", 2132.97998],\n    ["2016-10-17", 2126.5],\n    ["2016-10-18", 2139.600098],\n    ["2016-10-19", 2144.290039],\n    ["2016-10-20", 2141.340088],\n    ["2016-10-21", 2141.159912],\n    ["2016-10-24", 2151.330078],\n    ["2016-10-25", 2143.159912],\n    ["2016-10-26", 2139.429932],\n    ["2016-10-27", 2133.040039],\n    ["2016-10-28", 2126.409912],\n    ["2016-10-31", 2126.149902],\n    ["2016-11-01", 2111.719971],\n    ["2016-11-02", 2097.939941],\n    ["2016-11-03", 2088.659912],\n    ["2016-11-04", 2085.179932],\n    ["2016-11-07", 2131.52002],\n    ["2016-11-08", 2139.560059],\n    ["2016-11-09", 2163.26001],\n    ["2016-11-10", 2167.47998],\n    ["2016-11-11", 2164.449951],\n    ["2016-11-14", 2164.199951],\n    ["2016-11-15", 2180.389893],\n    ["2016-11-16", 2176.939941],\n    ["2016-11-17", 2187.120117],\n    ["2016-11-18", 2181.899902],\n    ["2016-11-21", 2198.179932],\n    ["2016-11-22", 2202.939941],\n    ["2016-11-23", 2204.719971],\n    ["2016-11-25", 2213.350098],\n    ["2016-11-28", 2201.719971],\n    ["2016-11-29", 2204.659912],\n    ["2016-11-30", 2198.810059],\n    ["2016-12-01", 2191.080078],\n    ["2016-12-02", 2191.949951],\n    ["2016-12-05", 2204.709961],\n    ["2016-12-06", 2212.22998],\n    ["2016-12-07", 2241.350098],\n    ["2016-12-08", 2246.189941],\n    ["2016-12-09", 2259.530029],\n    ["2016-12-12", 2256.959961],\n    ["2016-12-13", 2271.719971],\n    ["2016-12-14", 2253.280029],\n    ["2016-12-15", 2262.030029],\n    ["2016-12-16", 2258.070068],\n    ["2016-12-19", 2262.530029],\n    ["2016-12-20", 2270.76001],\n    ["2016-12-21", 2265.179932],\n    ["2016-12-22", 2260.959961],\n    ["2016-12-23", 2263.790039],\n    ["2016-12-27", 2268.879883],\n    ["2016-12-28", 2249.919922],\n    ["2016-12-29", 2249.26001],\n    ["2016-12-30", 2238.830078],\n    ["2017-01-03", 2257.830078],\n    ["2017-01-04", 2270.75],\n    ["2017-01-05", 2269.0],\n    ["2017-01-06", 2276.97998],\n    ["2017-01-09", 2268.899902],\n    ["2017-01-10", 2268.899902],\n    ["2017-01-11", 2275.320068],\n    ["2017-01-12", 2270.439941],\n    ["2017-01-13", 2274.639893],\n    ["2017-01-17", 2267.889893],\n    ["2017-01-18", 2271.889893],\n    ["2017-01-19", 2263.689941],\n    ["2017-01-20", 2271.310059],\n    ["2017-01-23", 2265.199951],\n    ["2017-01-24", 2280.070068],\n    ["2017-01-25", 2298.370117],\n    ["2017-01-26", 2296.679932],\n    ["2017-01-27", 2294.689941],\n    ["2017-01-30", 2280.899902],\n    ["2017-01-31", 2278.870117],\n    ["2017-02-01", 2279.550049],\n    ["2017-02-02", 2280.850098],\n    ["2017-02-03", 2297.419922],\n    ["2017-02-06", 2292.560059],\n    ["2017-02-07", 2293.080078],\n    ["2017-02-08", 2294.669922],\n    ["2017-02-09", 2307.870117],\n    ["2017-02-10", 2316.100098],\n    ["2017-02-13", 2328.25],\n    ["2017-02-14", 2337.580078],\n    ["2017-02-15", 2349.25],\n    ["2017-02-16", 2347.219971],\n    ["2017-02-17", 2351.159912],\n    ["2017-02-21", 2365.379883],\n    ["2017-02-22", 2362.820068],\n    ["2017-02-23", 2363.810059],\n    ["2017-02-24", 2367.340088],\n    ["2017-02-27", 2369.75],\n    ["2017-02-28", 2363.639893],\n    ["2017-03-01", 2395.959961],\n    ["2017-03-02", 2381.919922],\n    ["2017-03-03", 2383.120117],\n    ["2017-03-06", 2375.310059],\n    ["2017-03-07", 2368.389893],\n    ["2017-03-08", 2362.97998],\n    ["2017-03-09", 2364.870117],\n    ["2017-03-10", 2372.600098],\n    ["2017-03-13", 2373.469971],\n    ["2017-03-14", 2365.449951],\n    ["2017-03-15", 2385.26001],\n    ["2017-03-16", 2381.379883],\n    ["2017-03-17", 2378.25],\n    ["2017-03-20", 2373.469971],\n    ["2017-03-21", 2344.02002],\n    ["2017-03-22", 2348.449951],\n    ["2017-03-23", 2345.959961],\n    ["2017-03-24", 2343.97998],\n    ["2017-03-27", 2341.590088],\n    ["2017-03-28", 2358.570068],\n    ["2017-03-29", 2361.129883],\n    ["2017-03-30", 2368.060059],\n    ["2017-03-31", 2362.719971],\n    ["2017-04-03", 2358.840088],\n    ["2017-04-04", 2360.159912],\n    ["2017-04-05", 2352.949951],\n    ["2017-04-06", 2357.48999],\n    ["2017-04-07", 2355.540039],\n    ["2017-04-10", 2357.159912],\n    ["2017-04-11", 2353.780029],\n    ["2017-04-12", 2344.929932],\n    ["2017-04-13", 2328.949951],\n    ["2017-04-17", 2349.01001],\n    ["2017-04-18", 2342.189941],\n    ["2017-04-19", 2338.169922],\n    ["2017-04-20", 2355.840088],\n    ["2017-04-21", 2348.689941],\n    ["2017-04-24", 2374.149902],\n    ["2017-04-25", 2388.610107],\n    ["2017-04-26", 2387.449951],\n    ["2017-04-27", 2388.77002],\n    ["2017-04-28", 2384.199951],\n    ["2017-05-01", 2388.330078],\n    ["2017-05-02", 2391.169922],\n    ["2017-05-03", 2388.129883],\n    ["2017-05-04", 2389.52002],\n    ["2017-05-05", 2399.290039],\n    ["2017-05-08", 2399.379883],\n    ["2017-05-09", 2396.919922],\n    ["2017-05-10", 2399.629883],\n    ["2017-05-11", 2394.439941],\n    ["2017-05-12", 2390.899902],\n    ["2017-05-15", 2402.320068],\n    ["2017-05-16", 2400.669922],\n    ["2017-05-17", 2357.030029],\n    ["2017-05-18", 2365.719971],\n    ["2017-05-19", 2381.72998],\n    ["2017-05-22", 2394.02002],\n    ["2017-05-23", 2398.419922],\n    ["2017-05-24", 2404.389893],\n    ["2017-05-25", 2415.070068],\n    ["2017-05-26", 2415.820068],\n    ["2017-05-30", 2412.909912],\n    ["2017-05-31", 2411.800049],\n    ["2017-06-01", 2430.060059],\n    ["2017-06-02", 2439.070068],\n    ["2017-06-05", 2436.100098],\n    ["2017-06-06", 2429.330078],\n    ["2017-06-07", 2433.139893],\n    ["2017-06-08", 2433.790039],\n    ["2017-06-09", 2431.77002],\n    ["2017-06-12", 2429.389893],\n    ["2017-06-13", 2440.350098],\n    ["2017-06-14", 2437.919922],\n    ["2017-06-15", 2432.459961],\n    ["2017-06-16", 2433.149902],\n    ["2017-06-19", 2453.459961],\n    ["2017-06-20", 2437.030029],\n    ["2017-06-21", 2435.610107],\n    ["2017-06-22", 2434.5],\n    ["2017-06-23", 2438.300049],\n    ["2017-06-26", 2439.070068],\n    ["2017-06-27", 2419.379883],\n    ["2017-06-28", 2440.689941],\n    ["2017-06-29", 2419.699951],\n    ["2017-06-30", 2423.409912],\n    ["2017-07-03", 2429.01001],\n    ["2017-07-05", 2432.540039],\n    ["2017-07-06", 2409.75],\n    ["2017-07-07", 2425.179932],\n    ["2017-07-10", 2427.429932],\n    ["2017-07-11", 2425.530029],\n    ["2017-07-12", 2443.25],\n    ["2017-07-13", 2447.830078],\n    ["2017-07-14", 2459.27002],\n    ["2017-07-17", 2459.139893],\n    ["2017-07-18", 2460.610107],\n    ["2017-07-19", 2473.830078],\n    ["2017-07-20", 2473.449951],\n    ["2017-07-21", 2472.540039],\n    ["2017-07-24", 2469.909912],\n    ["2017-07-25", 2477.129883],\n    ["2017-07-26", 2477.830078],\n    ["2017-07-27", 2475.419922],\n    ["2017-07-28", 2472.100098],\n    ["2017-07-31", 2470.300049],\n    ["2017-08-01", 2476.350098],\n    ["2017-08-02", 2477.570068],\n    ["2017-08-03", 2472.159912],\n    ["2017-08-04", 2476.830078],\n    ["2017-08-07", 2480.909912],\n    ["2017-08-08", 2474.919922],\n    ["2017-08-09", 2474.02002],\n    ["2017-08-10", 2438.209961],\n    ["2017-08-11", 2441.320068],\n    ["2017-08-14", 2465.840088],\n    ["2017-08-15", 2464.610107],\n    ["2017-08-16", 2468.110107],\n    ["2017-08-17", 2430.01001],\n    ["2017-08-18", 2425.550049],\n    ["2017-08-21", 2428.370117],\n    ["2017-08-22", 2452.51001],\n    ["2017-08-23", 2444.040039],\n    ["2017-08-24", 2438.969971],\n    ["2017-08-25", 2443.050049],\n    ["2017-08-28", 2444.23999],\n    ["2017-08-29", 2446.300049],\n    ["2017-08-30", 2457.590088],\n    ["2017-08-31", 2471.649902],\n    ["2017-09-01", 2476.550049],\n    ["2017-09-05", 2457.850098],\n    ["2017-09-06", 2465.540039],\n    ["2017-09-07", 2465.100098],\n    ["2017-09-08", 2461.429932],\n    ["2017-09-11", 2488.110107],\n    ["2017-09-12", 2496.47998],\n    ["2017-09-13", 2498.370117],\n    ["2017-09-14", 2495.620117],\n    ["2017-09-15", 2500.22998],\n    ["2017-09-18", 2503.870117],\n    ["2017-09-19", 2506.649902],\n    ["2017-09-20", 2508.23999],\n    ["2017-09-21", 2500.600098],\n    ["2017-09-22", 2502.219971],\n    ["2017-09-25", 2496.659912],\n    ["2017-09-26", 2496.840088],\n    ["2017-09-27", 2507.040039],\n    ["2017-09-28", 2510.060059],\n    ["2017-09-29", 2519.360107],\n    ["2017-10-02", 2529.120117],\n    ["2017-10-03", 2534.580078],\n    ["2017-10-04", 2537.73999],\n    ["2017-10-05", 2552.070068],\n    ["2017-10-06", 2549.330078],\n    ["2017-10-09", 2544.72998],\n    ["2017-10-10", 2550.639893],\n    ["2017-10-11", 2555.23999],\n    ["2017-10-12", 2550.929932],\n    ["2017-10-13", 2553.169922],\n    ["2017-10-16", 2557.639893],\n    ["2017-10-17", 2559.360107],\n    ["2017-10-18", 2561.26001],\n    ["2017-10-19", 2562.100098],\n    ["2017-10-20", 2575.209961],\n    ["2017-10-23", 2564.97998],\n    ["2017-10-24", 2569.129883],\n    ["2017-10-25", 2557.149902],\n    ["2017-10-26", 2560.399902],\n    ["2017-10-27", 2581.070068],\n    ["2017-10-30", 2572.830078],\n    ["2017-10-31", 2575.26001],\n    ["2017-11-01", 2579.360107],\n    ["2017-11-02", 2579.850098],\n    ["2017-11-03", 2587.840088],\n    ["2017-11-06", 2591.129883],\n    ["2017-11-07", 2590.639893],\n    ["2017-11-08", 2594.379883],\n    ["2017-11-09", 2584.620117],\n    ["2017-11-10", 2582.300049],\n    ["2017-11-13", 2584.840088],\n    ["2017-11-14", 2578.870117],\n    ["2017-11-15", 2564.620117],\n    ["2017-11-16", 2585.639893],\n    ["2017-11-17", 2578.850098],\n    ["2017-11-20", 2582.139893],\n    ["2017-11-21", 2599.030029],\n    ["2017-11-22", 2597.080078],\n    ["2017-11-24", 2602.419922],\n    ["2017-11-27", 2601.419922],\n    ["2017-11-28", 2627.040039],\n    ["2017-11-29", 2626.070068],\n    ["2017-11-30", 2647.580078],\n    ["2017-12-01", 2642.219971],\n    ["2017-12-04", 2639.439941],\n    ["2017-12-05", 2629.570068],\n    ["2017-12-06", 2629.27002],\n    ["2017-12-07", 2636.97998],\n    ["2017-12-08", 2651.5],\n    ["2017-12-11", 2659.98999],\n    ["2017-12-12", 2664.110107],\n    ["2017-12-13", 2662.850098],\n    ["2017-12-14", 2652.01001],\n    ["2017-12-15", 2675.810059],\n    ["2017-12-18", 2690.159912],\n    ["2017-12-19", 2681.469971],\n    ["2017-12-20", 2679.25],\n    ["2017-12-21", 2684.570068],\n    ["2017-12-22", 2683.340088],\n    ["2017-12-26", 2680.5],\n    ["2017-12-27", 2682.620117],\n    ["2017-12-28", 2687.540039],\n    ["2017-12-29", 2673.610107],\n    ["2018-01-02", 2695.810059],\n    ["2018-01-03", 2713.060059],\n    ["2018-01-04", 2723.98999],\n    ["2018-01-05", 2743.149902],\n    ["2018-01-08", 2747.709961],\n    ["2018-01-09", 2751.290039],\n    ["2018-01-10", 2748.22998],\n    ["2018-01-11", 2767.560059],\n    ["2018-01-12", 2786.23999],\n    ["2018-01-16", 2776.419922],\n    ["2018-01-17", 2802.560059],\n    ["2018-01-18", 2798.030029],\n    ["2018-01-19", 2810.300049],\n    ["2018-01-22", 2832.969971],\n    ["2018-01-23", 2839.129883],\n    ["2018-01-24", 2837.540039],\n    ["2018-01-25", 2839.25],\n    ["2018-01-26", 2872.870117],\n    ["2018-01-29", 2853.530029],\n    ["2018-01-30", 2822.429932],\n    ["2018-01-31", 2823.810059],\n    ["2018-02-01", 2821.97998],\n    ["2018-02-02", 2762.129883],\n    ["2018-02-05", 2648.939941],\n    ["2018-02-06", 2695.139893],\n    ["2018-02-07", 2681.659912],\n    ["2018-02-08", 2581.0],\n    ["2018-02-09", 2619.550049],\n    ["2018-02-12", 2656.0],\n    ["2018-02-13", 2662.939941],\n    ["2018-02-14", 2698.629883],\n    ["2018-02-15", 2731.199951],\n    ["2018-02-16", 2732.219971],\n    ["2018-02-20", 2716.26001],\n    ["2018-02-21", 2701.330078],\n    ["2018-02-22", 2703.959961],\n    ["2018-02-23", 2747.300049],\n    ["2018-02-26", 2779.600098],\n    ["2018-02-27", 2744.280029],\n    ["2018-02-28", 2713.830078],\n    ["2018-03-01", 2677.669922],\n    ["2018-03-02", 2691.25],\n    ["2018-03-05", 2720.939941],\n    ["2018-03-06", 2728.120117],\n    ["2018-03-07", 2726.800049],\n    ["2018-03-08", 2738.969971],\n    ["2018-03-09", 2786.570068],\n    ["2018-03-12", 2783.02002],\n    ["2018-03-13", 2765.310059],\n    ["2018-03-14", 2749.47998],\n    ["2018-03-15", 2747.330078],\n    ["2018-03-16", 2752.01001],\n    ["2018-03-19", 2712.919922],\n    ["2018-03-20", 2716.939941],\n    ["2018-03-21", 2711.929932],\n    ["2018-03-22", 2643.689941],\n    ["2018-03-23", 2588.26001],\n    ["2018-03-26", 2658.550049],\n    ["2018-03-27", 2612.620117],\n    ["2018-03-28", 2605.0],\n    ["2018-03-29", 2640.870117],\n    ["2018-04-02", 2581.879883],\n    ["2018-04-03", 2614.449951],\n    ["2018-04-04", 2644.689941],\n    ["2018-04-05", 2662.840088],\n    ["2018-04-06", 2604.469971],\n    ["2018-04-09", 2613.159912],\n    ["2018-04-10", 2656.870117],\n    ["2018-04-11", 2642.189941],\n    ["2018-04-12", 2663.98999],\n    ["2018-04-13", 2656.300049],\n    ["2018-04-16", 2677.840088],\n    ["2018-04-17", 2706.389893],\n    ["2018-04-18", 2708.639893],\n    ["2018-04-19", 2693.129883],\n    ["2018-04-20", 2670.139893],\n    ["2018-04-23", 2670.290039],\n    ["2018-04-24", 2634.560059],\n    ["2018-04-25", 2639.399902],\n    ["2018-04-26", 2666.939941],\n    ["2018-04-27", 2669.909912],\n    ["2018-04-30", 2648.050049],\n    ["2018-05-01", 2654.800049],\n    ["2018-05-02", 2635.669922],\n    ["2018-05-03", 2629.72998],\n    ["2018-05-04", 2663.419922],\n    ["2018-05-07", 2672.629883],\n    ["2018-05-08", 2671.919922],\n    ["2018-05-09", 2697.790039],\n    ["2018-05-10", 2723.070068],\n    ["2018-05-11", 2727.719971],\n    ["2018-05-14", 2730.129883],\n    ["2018-05-15", 2711.449951],\n    ["2018-05-16", 2722.459961],\n    ["2018-05-17", 2720.129883],\n    ["2018-05-18", 2712.969971],\n    ["2018-05-21", 2733.01001],\n    ["2018-05-22", 2724.439941],\n    ["2018-05-23", 2733.290039],\n    ["2018-05-24", 2727.76001],\n    ["2018-05-25", 2721.330078],\n    ["2018-05-29", 2689.860107],\n    ["2018-05-30", 2724.01001],\n    ["2018-05-31", 2705.27002],\n    ["2018-06-01", 2734.620117],\n    ["2018-06-04", 2746.870117],\n    ["2018-06-05", 2748.800049],\n    ["2018-06-06", 2772.350098],\n    ["2018-06-07", 2770.370117],\n    ["2018-06-08", 2779.030029],\n    ["2018-06-11", 2782.0],\n    ["2018-06-12", 2786.850098],\n    ["2018-06-13", 2775.629883],\n    ["2018-06-14", 2782.48999],\n    ["2018-06-15", 2779.659912],\n    ["2018-06-18", 2773.75],\n    ["2018-06-19", 2762.590088],\n    ["2018-06-20", 2767.320068],\n    ["2018-06-21", 2749.76001],\n    ["2018-06-22", 2754.879883],\n    ["2018-06-25", 2717.070068],\n    ["2018-06-26", 2723.060059],\n    ["2018-06-27", 2699.629883],\n    ["2018-06-28", 2716.310059],\n    ["2018-06-29", 2718.370117],\n    ["2018-07-02", 2726.709961],\n    ["2018-07-03", 2713.219971],\n    ["2018-07-05", 2736.610107],\n    ["2018-07-06", 2759.820068],\n    ["2018-07-09", 2784.169922],\n    ["2018-07-10", 2793.840088],\n    ["2018-07-11", 2774.02002],\n    ["2018-07-12", 2798.290039],\n    ["2018-07-13", 2801.310059],\n    ["2018-07-16", 2798.429932],\n    ["2018-07-17", 2809.550049],\n    ["2018-07-18", 2815.620117],\n    ["2018-07-19", 2804.48999],\n    ["2018-07-20", 2801.830078],\n    ["2018-07-23", 2806.97998],\n    ["2018-07-24", 2820.399902],\n    ["2018-07-25", 2846.070068],\n    ["2018-07-26", 2837.439941],\n    ["2018-07-27", 2818.820068],\n    ["2018-07-30", 2802.600098],\n    ["2018-07-31", 2816.290039],\n    ["2018-08-01", 2813.360107],\n    ["2018-08-02", 2827.219971],\n    ["2018-08-03", 2840.350098],\n    ["2018-08-06", 2850.399902],\n    ["2018-08-07", 2858.449951],\n    ["2018-08-08", 2857.699951],\n    ["2018-08-09", 2853.580078],\n    ["2018-08-10", 2833.280029],\n    ["2018-08-13", 2821.929932],\n    ["2018-08-14", 2839.959961],\n    ["2018-08-15", 2818.370117],\n    ["2018-08-16", 2840.689941],\n    ["2018-08-17", 2850.129883],\n    ["2018-08-20", 2857.050049],\n    ["2018-08-21", 2862.959961],\n    ["2018-08-22", 2861.820068],\n    ["2018-08-23", 2856.97998],\n    ["2018-08-24", 2874.689941],\n    ["2018-08-27", 2896.73999],\n    ["2018-08-28", 2897.52002],\n    ["2018-08-29", 2914.040039],\n    ["2018-08-30", 2901.129883],\n    ["2018-08-31", 2901.52002],\n    ["2018-09-04", 2896.719971],\n    ["2018-09-05", 2888.600098],\n    ["2018-09-06", 2878.050049],\n    ["2018-09-07", 2871.679932],\n    ["2018-09-10", 2877.129883],\n    ["2018-09-11", 2887.889893],\n    ["2018-09-12", 2888.919922],\n    ["2018-09-13", 2904.179932],\n    ["2018-09-14", 2904.97998],\n    ["2018-09-17", 2888.800049],\n    ["2018-09-18", 2904.310059],\n    ["2018-09-19", 2907.949951],\n    ["2018-09-20", 2930.75],\n    ["2018-09-21", 2929.669922],\n    ["2018-09-24", 2919.370117],\n    ["2018-09-25", 2915.560059],\n    ["2018-09-26", 2905.969971],\n    ["2018-09-27", 2914.0],\n    ["2018-09-28", 2913.97998],\n    ["2018-10-01", 2924.590088],\n    ["2018-10-02", 2923.429932],\n    ["2018-10-03", 2925.51001],\n    ["2018-10-04", 2901.610107],\n    ["2018-10-05", 2885.570068],\n    ["2018-10-08", 2884.429932],\n    ["2018-10-09", 2880.340088],\n    ["2018-10-10", 2785.679932],\n    ["2018-10-11", 2728.370117],\n    ["2018-10-12", 2767.129883],\n    ["2018-10-15", 2750.790039],\n    ["2018-10-16", 2809.919922],\n    ["2018-10-17", 2809.209961],\n    ["2018-10-18", 2768.780029],\n    ["2018-10-19", 2767.780029],\n    ["2018-10-22", 2755.879883],\n    ["2018-10-23", 2740.689941],\n    ["2018-10-24", 2656.100098],\n    ["2018-10-25", 2705.570068],\n    ["2018-10-26", 2658.689941],\n    ["2018-10-29", 2641.25],\n    ["2018-10-30", 2682.629883],\n    ["2018-10-31", 2711.73999],\n    ["2018-11-01", 2740.370117],\n    ["2018-11-02", 2723.060059],\n    ["2018-11-05", 2738.310059],\n    ["2018-11-06", 2755.449951],\n    ["2018-11-07", 2813.889893],\n    ["2018-11-08", 2806.830078],\n    ["2018-11-09", 2781.01001],\n    ["2018-11-12", 2726.219971],\n    ["2018-11-13", 2722.179932],\n    ["2018-11-14", 2701.580078],\n    ["2018-11-15", 2730.199951],\n    ["2018-11-16", 2736.27002],\n    ["2018-11-19", 2690.72998],\n    ["2018-11-20", 2641.889893],\n    ["2018-11-21", 2649.929932],\n    ["2018-11-23", 2632.560059],\n    ["2018-11-26", 2673.449951],\n    ["2018-11-27", 2682.169922],\n    ["2018-11-28", 2743.790039],\n    ["2018-11-29", 2737.800049],\n    ["2018-11-30", 2760.169922],\n    ["2018-12-03", 2790.370117],\n    ["2018-12-04", 2700.060059],\n    ["2018-12-06", 2695.949951],\n    ["2018-12-07", 2633.080078],\n    ["2018-12-10", 2637.719971],\n    ["2018-12-11", 2636.780029],\n    ["2018-12-12", 2651.070068],\n    ["2018-12-13", 2650.540039],\n    ["2018-12-14", 2599.949951],\n    ["2018-12-17", 2545.939941],\n    ["2018-12-18", 2546.159912],\n    ["2018-12-19", 2506.959961],\n    ["2018-12-20", 2467.419922],\n    ["2018-12-21", 2416.620117],\n    ["2018-12-24", 2351.100098],\n    ["2018-12-26", 2467.699951],\n    ["2018-12-27", 2488.830078],\n    ["2018-12-28", 2485.73999],\n    ["2018-12-31", 2506.850098]\n  ]\n}\n'
assert hashlib.sha256(sp500_snapshot_text.encode()).hexdigest() == '6f662b7d0e2eefcb142f6fffb6996a0e42b915b66cba6bcc03e944ea322d91ed'
sp500_data = json.loads(sp500_snapshot_text)
sp500_prices = sp500_data['prices']
sp500_dates = [row[0] for row in sp500_prices[1:]]
sp500_returns = [math.log(current[1]/previous[1]) for previous,current in zip(sp500_prices,sp500_prices[1:])]
assert len(sp500_prices) == 5031 and len(sp500_returns) == 5030
assert sp500_dates[0] == '1999-01-05' and sp500_dates[-1] == '2018-12-31'
print("Loaded embedded S&P 500 snapshot and self-contained functions; no remote downloads.")
print("Original source SHA-256:", sp500_data['source_sha256'])


Loaded embedded S&P 500 snapshot and self-contained functions; no remote downloads.
Original source SHA-256: 1e028cbb9c400cc018c816ccc439b33c919387e726c3ed5ca2c05c82746059de


## จากราคาไปเป็นผลตอบแทน

ให้ \(P_{t-1}\) เป็นราคาก่อนเริ่มช่วง \(P_t\) เป็นราคาปลายช่วง และ \(D_t\) เป็นเงินปันผลที่ได้รับเมื่อสิ้นช่วงต่อหนึ่งหน่วยสินทรัพย์ โดยยังไม่หักต้นทุนซื้อขาย

$$
R_t=\frac{P_t+D_t-P_{t-1}}{P_{t-1}},\qquad
r_t=\log(1+R_t)=\log\left(\frac{P_t+D_t}{P_{t-1}}\right).
$$

R คือ simple return ส่วน r คือ log return ตัวอย่างซื้อที่ 100 ขายที่ 99 และรับปันผล 2 จะได้ R=1% และ r≈0.9950% ถ้าใช้ราคาดิบแล้วละปันผล เราจะคำนวณ R เป็น −1% ทั้งที่ความมั่งคั่งรวมเพิ่มขึ้น

เมื่อ R ใกล้ศูนย์ ค่า r ใกล้ R แต่ความต่างเพิ่มตามขนาดการเปลี่ยนแปลง การขึ้น 20% แล้วลง 20% ให้ simple return สะสม −4% ส่วน log returns บวกกันได้ \(\log(1.2)+\log(0.8)=\log(0.96)\) ไม่ใช่ศูนย์

$$
R_{1:T}=\prod_{t=1}^{T}(1+R_t)-1,\qquad
r_{1:T}=\sum_{t=1}^{T}r_t=\log(1+R_{1:T}).
$$

เมื่อคำนวณผลตอบแทนหลายช่วงที่มีปันผล สูตรทบต้นนี้สมมติว่านำปันผลกลับไปลงทุน หากใช้ราคาที่ปรับปันผลและแตกหุ้นไว้แล้ว ให้ตรวจวิธีปรับราคาก่อน เพื่อไม่ให้นับปันผลซ้ำ

บทนี้ใช้ **log return** สำหรับอนุกรมเวลาและ realized variance เป็นหลัก ส่วนการรวมสินทรัพย์ด้วยน้ำหนักพอร์ต ณ ต้นช่วงยังใช้ simple returns: \(R_p=\sum_iw_iR_i\) การเฉลี่ย log returns ของสินทรัพย์ด้วยน้ำหนักเดียวกันไม่ได้ให้ log return ของพอร์ตตรง ๆ

In [2]:
simple, log_return = return_pair(100,99,2)
close(simple,.01)
close(log_return,math.log(1.01))
print(f"Dividend-inclusive simple return {simple:.6%}; log return {log_return:.6%}")
compound = 1.2*.8-1
sum_log = math.log(1.2)+math.log(.8)
close(math.expm1(sum_log),compound)
print(f"+20%, then -20%: simple {compound:.4%}; log {sum_log:.4%}")

Dividend-inclusive simple return 1.000000%; log return 0.995033%
+20%, then -20%: simple -4.0000%; log -4.0822%


## รูปแบบสามอย่างในผลตอบแทนรายวัน

Taylor จัดข้อสังเกตหลักของผลตอบแทนรายวันไว้สามข้อ

| สิ่งที่พบซ้ำ | ดูจากอะไร | สมมติฐานที่ต้องตรวจต่อ |
|---|---|---|
| การแจกแจงมีหางหนากว่า Normal | Histogram, Q–Q plot, tail probabilities และ kurtosis | Normal ที่ใช้ประมาณขาดทุนครอบคลุมหางหรือไม่ |
| ผลตอบแทนคนละวันมักมี linear correlation ต่ำ | Scatter plot และ ACF ของ r | Correlation ต่ำไม่ได้รับรอง independence |
| ขนาดผลตอบแทนมีความสัมพันธ์บวกข้ามวัน | ACF ของ \(|r|\) และ \(r^2\) | Volatility คงที่อาจไม่เหมาะกับข้อมูล |

ผลตอบแทนหุ้นยังอาจเบ้ด้านขาดทุน ขนาดความเบ้และความหนาของหางเปลี่ยนตามตลาดและช่วงตัวอย่าง ส่วนคำว่า correlation ต่ำไม่ได้หมายถึงเท่ากับศูนย์ทุก lag โดยเฉพาะสินทรัพย์ที่ซื้อขายบางหรือข้อมูลระยะสั้นมาก

ค่าเฉลี่ยรายวันมักมีขนาดเล็กเมื่อเทียบกับ SD เมื่อใช้ข้อมูลช่วงสั้นแล้วแปลงค่าเฉลี่ยเป็นรายปี ความคลาดเคลื่อนของค่าประมาณก็ขยายตามไปด้วย ส่วน SD ที่คำนวณจากช่วงวิกฤตย่อมต่างจากช่วงตลาดสงบ

Taylor ยังยกตัวอย่าง calendar effects เช่น ความต่างของผลตอบแทนตามวันในสัปดาห์ ต้นเดือน เดือนมกราคม และก่อนวันหยุด ผลที่พบอาจอ่อนลงเมื่อเปลี่ยนช่วงข้อมูล และการทดสอบหลายเงื่อนไขย้อนหลังอาจทำให้เราเลือกผลที่เด่นเพราะความบังเอิญ ก่อนนำไปใช้จึงต้องทดสอบกับข้อมูลนอกช่วงประมาณค่าและหักต้นทุนซื้อขายด้วย

## สถิติสรุปต้องบอกทั้งค่าและนิยาม

ให้ r₁,…,rₙ เป็น log returns ในช่วงเวลาเดียวกัน กำหนดค่าเฉลี่ยและ central moments ของตัวอย่างเป็น

$$
\bar r=\frac1n\sum_{t=1}^nr_t,\qquad
m_j=\frac1n\sum_{t=1}^n(r_t-\bar r)^j,\qquad
s^2=\frac{n}{n-1}m_2.
$$

s² ใช้ตัวหาร n−1 ส่วน m₂ ใช้ n การประมาณ variance แบบ s² ไม่มี bias ภายใต้ iid ที่มี variance จำกัด แต่เมื่อข้อมูลพึ่งพากัน คุณสมบัตินี้ต้องพิจารณาใหม่ การรายงานต้องระบุด้วยว่าเป็น simple หรือ log return และวัดเป็นทศนิยมหรือเปอร์เซ็นต์

ตัวอย่างสมมติห้าค่า −2%, −1%, 0%, 1%, 6% ให้

| สถิติ | ค่า | อ่านอย่างไร |
|---|---:|---|
| จำนวนข้อมูล | 5 | ใช้สาธิตสูตร ไม่พอประมาณหางของตลาด |
| ค่าเฉลี่ย | 0.8000% | ไวต่อวันที่ +6% |
| Median | 0% | ค่ากลางเมื่อเรียงข้อมูล |
| Sample SD | 3.1145% | ขนาดการกระจายรอบ mean ใช้ n−1 |
| Minimum / Maximum | −2% / 6% | ช่วงที่เกิดในตัวอย่าง ไม่ใช่ขอบเขตของประชากร |
| Moment skewness | 1.0392 | ตัวอย่างมีหางยาวด้านบวก |
| Moment kurtosis | 2.6688 | ใช้ m₄/m₂²; Normal ประชากรมีค่า 3 |

สถิติแต่ละตัวใช้ข้อมูลคนละด้าน Median ไม่ได้บอก variance และ SD ไม่ได้บอกว่าหางด้านไหนยาวกว่า หากสองชุดมี mean และ SD เท่ากัน ยังต้องดู histogram, quantiles และลำดับเวลา

เมื่อเพิ่มข้อมูลใหม่ ให้ใช้ช่วงตัวอย่างเดียวกันในการเทียบสินทรัพย์ และรายงานจำนวนข้อมูลที่ถูกตัดหรือขาดไปด้วย การเลือกช่วงหลังเห็นผลแล้วอาจทำให้สถิติดูสอดคล้องกับข้อสรุปที่ต้องการเกินจริง

In [3]:
values=[-.02,-.01,0,.01,.06]
stats=summary_stats(values)
print('Five hypothetical returns:',stats)
close(stats['mean'],.008);close(stats['sd'],math.sqrt(.00097))
flipped=summary_stats([-v for v in values])
close(flipped['skewness'],-stats['skewness']);close(flipped['kurtosis'],stats['kurtosis'])

Five hypothetical returns: {'mean': 0.008, 'sd': 0.03114482300479487, 'variance': 0.000776, 'kurtosis': 2.6687745775321496, 'skewness': 1.0391889223704167}


## ค่าเฉลี่ยผลตอบแทนกับ risk premium

ผลตอบแทนส่วนเกินที่เกิดขึ้นจริงคือ \(R_t^e=R_t-R_{f,t}\) โดย Rf เป็นผลตอบแทนสินทรัพย์ปลอดความเสี่ยงสำหรับช่วงถือและสกุลเงินเดียวกัน ส่วน risk premium ที่คาดไว้ก่อนลงทุนคือ

$$
\operatorname{RP}_t=\mathbb E[R_t-R_{f,t}\mid\mathcal F_{t-1}].
$$

ค่าเฉลี่ยผลตอบแทนส่วนเกินในอดีตใช้ประมาณ premium ได้ภายใต้สมมติฐานว่าช่วงข้อมูลนั้นยังเกี่ยวข้องกับอนาคต แต่ผลตอบแทนที่เกิดขึ้นจริงรวมช็อกที่ไม่คาดไว้ด้วย จึงอาจติดลบแม้ premium ที่คาดไว้เป็นบวก

การใช้ log excess return \(r_t-r_{f,t}\) ต้องระบุให้ชัด เพราะไม่เท่ากับ simple excess return แม้ค่าจะใกล้กันเมื่อผลตอบแทนเล็ก หาก log return แจกแจง N(μ,σ²) จะมี \(\mathbb E[R]=e^{\mu+\sigma^2/2}-1\) ขณะที่อัตราเติบโตจาก mean log เท่ากับ \(e^\mu-1\)

การประมาณ mean ต้องใช้ข้อมูลมากเพราะสัญญาณรายวันเล็กเมื่อเทียบกับความผันผวน ภายใต้ iid ค่า standard error ของ mean เท่ากับ s/√n เช่น สมมติ mean รายวัน 0.04%, SD 1.2% และ n=2,520 วัน

$$
252\bar r=10.08\%,\qquad
\operatorname{SE}(252\bar r)=\frac{252(0.012)}{\sqrt{2520}}\approx6.024\%.
$$

ตัวเลข 10.08% เป็นค่าเฉลี่ย log return ที่แปลงเป็นรายปี ส่วน 6.024% เป็น standard error ของค่าประมาณนั้น ไม่ใช่ SD ของผลตอบแทนรายปี ตัวอย่างนี้แสดงว่าข้อมูลสิบปีตามสมมติฐาน 252 วันต่อปีอาจยังให้ mean ที่ไม่แม่น หากมี serial dependence ต้องรวม autocovariance หรือใช้วิธีประมาณ standard error ที่รองรับ dependence

การเทียบ premium หุ้น พันธบัตร หรือสกุลเงินต้องระบุช่วงเวลา ปันผล ต้นทุน และ benchmark ควบคู่กัน ผลตอบแทนย้อนหลังสูงอาจมาจากความเสี่ยงที่รับหรือช็อกที่ดีในช่วงนั้น การเรียกว่า alpha ต้องระบุโมเดลผลตอบแทนที่ใช้เทียบเพิ่มด้วย

In [4]:
daily_mean,daily_sd,n=.0004,.012,2520
annual_mean=252*daily_mean
annual_mean_se=252*daily_sd/math.sqrt(n)
close(annual_mean,.1008)
print(f'Annualized mean log return {annual_mean:.4%}; iid standard error {annual_mean_se:.4%}')
print('This standard error is uncertainty about the estimated mean, not annual return volatility.')

Annualized mean log return 10.0800%; iid standard error 6.0240%
This standard error is uncertainty about the estimated mean, not annual return volatility.


## SD เปลี่ยนตามความถี่และช่วงตัวอย่าง

Sample SD จากข้อมูลทั้งช่วงให้ขนาดความผันผวนเฉลี่ยในช่วงนั้น ส่วน rolling SD คำนวณใหม่จากหน้าต่างล่าสุด เช่น n วัน จึงเปลี่ยนเมื่อข้อมูลใหม่เข้ามาและข้อมูลเก่าออกไป

$$
s_{t,n}^2=\frac1{n-1}\sum_{j=0}^{n-1}(r_{t-j}-\bar r_{t,n})^2.
$$

หน้าต่างสั้นตอบสนองเร็วแต่ค่าประมาณแกว่งมาก หน้าต่างยาวเรียบกว่าแต่รวมสภาวะเก่ามากขึ้น วันที่รุนแรงหนึ่งวันจะมีผลต่อ rolling SD จนกว่าจะหลุดจากหน้าต่าง อ่านตัวทดลองประกอบใน [บทพฤติกรรมแบบสุ่มของสินทรัพย์](../random-assets.html)

สำหรับกระบวนการ weak stationary ความแปรปรวนของผลตอบแทนรวม h ช่วงคือ

$$
\operatorname{Var}\!\left(\sum_{j=1}^{h}r_{t+j}\right)
=h\gamma_0+2\sum_{k=1}^{h-1}(h-k)\gamma_k.
$$

สูตร √h ใช้ได้เมื่อ covariance ข้ามช่วงเป็นศูนย์และ variance ต่อช่วงเท่ากัน การมี marginal Normal ไม่ได้ทำให้พจน์ covariance หายไปเอง ตัวอย่างสองวันมี SD วันละ 1% และ correlation 0.3 จะมี SD รวม \(\sqrt{2(0.01)^2(1+0.3)}\approx1.6125\%\) เทียบกับ 1.4142% เมื่อ correlation เป็นศูนย์

การ annualize SD รายวันด้วย √252 จึงต้องบอกสมมติฐานและจำนวนวัน ส่วน SD กับ standard error ตอบคนละคำถาม: SD วัดการกระจายของผลตอบแทน แต่ standard error วัดความคลาดเคลื่อนของค่าประมาณ

In [5]:
sd,rho=.01,.3
variance=2*sd**2+2*rho*sd**2
print(f'Two-day SD with correlation .3: {math.sqrt(variance):.6%}; uncorrelated: {math.sqrt(2)*sd:.6%}')
close(variance,.00026)

Two-day SD with correlation .3: 1.612452%; uncorrelated: 1.414214%


## Calendar effects และการทดสอบซ้ำหลายครั้ง

หัวข้อ calendar effects ครอบคลุมทั้งค่าเฉลี่ยและความผันผวนตามวันในสัปดาห์ ช่วงเปลี่ยนเดือน เดือนมกราคม และรอบวันหยุด ผลของปฏิทินอาจต่างกันตามตลาดและช่วงศึกษา วันที่อยู่ติดวันหยุดยังครอบคลุมเวลาปฏิทินไม่เท่ากับวันซื้อขายทั่วไป

วิธีประมาณ mean ตามวันในสัปดาห์คือใช้ dummy regression

$$
r_t=\alpha+\sum_{d=2}^{5}\beta_d\,1\{D_t=d\}+u_t.
$$

Dₜ=1 เป็นวันอ้างอิง α คือ mean ของวันนั้น ส่วน βd คือความต่างจากวันอ้างอิง หากใช้ intercept พร้อม dummy ครบทั้งห้าวัน จะมีตัวแปรซ้ำเชิงเส้น ต้องตัดวันหนึ่งออกหรือใช้ dummy ห้าตัวโดยไม่มี intercept

การทดสอบว่า mean ต่างกันตามวันพิจารณาสมมติฐานร่วม β₂=⋯=β₅=0 และเลือก standard errors ให้รองรับ heteroskedasticity หรือ autocorrelation ตามข้อมูล หากสนใจ volatility ให้ศึกษาขนาดผลตอบแทนหรือ variance ตามวันแยกจาก regression ของ mean

การลองหลายเดือน หลายวัน และหลายตลาดเพิ่มโอกาสพบผลที่ดูมีนัยสำคัญโดยบังเอิญ ต้องแยกช่วงค้นหารูปแบบออกจากช่วงทดสอบ ระบุจำนวนสมมติฐานที่ลอง และประเมินหลังหักต้นทุนซื้อขาย ตารางผลที่เลือกมาเฉพาะข้อที่ผ่านไม่แสดงความเสี่ยงจากการค้นหานี้

แม้ช็อกของแต่ละวันเป็นอิสระ mean ที่เปลี่ยนตามวันก็สร้าง ACF แบบคาบได้เมื่อใช้ mean รวม ดูสูตรและตัวทดลองใน [ภาคผนวก calendar effects](#calendar-acf-appendix)

## วันที่แกว่งแรงมักอยู่ใกล้กัน

[Volatility clustering](../glossary.html#volatility-clustering) หมายถึงช่วงที่ผลตอบแทนมีขนาดใหญ่เกิดติดกัน สลับกับช่วงที่ขนาดเล็ก คำว่า “ขนาดใหญ่” ครอบคลุมทั้งบวกและลบ หลังวันที่ลงแรง วันถัดไปอาจลงต่อหรือดีดกลับแรงก็ได้

แบบจำลองง่าย ๆ ที่แยกสองส่วนนี้คือ

$$
r_t=\mu_t+\sigma_tz_t,\qquad
\mathbb E[z_t\mid\mathcal F_{t-1}]=0,\qquad
\mathbb E[z_t^2\mid\mathcal F_{t-1}]=1.
$$

\(\mathcal F_{t-1}\) คือข้อมูลที่รู้ก่อนเริ่มช่วง t ส่วน \(\mu_t\) และ \(\sigma_t\) เป็นค่าที่กำหนดจากข้อมูลนั้นได้ เมื่อ \(\sigma_t\) สูง ขนาด \(|r_t-\mu_t|\) มีแนวโน้มสูงขึ้น แต่เครื่องหมายยังขึ้นกับช็อก \(z_t\) การคาดการณ์ volatility จึงไม่ได้ให้คำตอบทิศทางผลตอบแทนโดยอัตโนมัติ



ราคาปิดดัชนี S&P 500 จาก Yahoo Finance ในชุดข้อมูลตัวอย่าง arch 8.0.0 ใช้ข้อมูลทั้งชุดตั้งแต่ 4 มกราคม 1999 ถึง 31 ธันวาคม 2018 จำนวน 5,031 ราคา คำนวณ log returns ได้ 5,030 ค่า โดยไม่รวมปันผล

คำนวณแต่ละค่าด้วย \(r_t=\log(P_t/P_{t-1})\) จากราคาปิดของวันซื้อขายที่ติดกัน ใช้ราคาแรกเป็นฐาน ไม่เติมวันหยุดและไม่ตัดวันที่แกว่งแรงออก ในตัวอย่างนี้ ACF ที่ lag 1 ของ r เท่ากับประมาณ −0.070 ส่วนของ |r| เท่ากับ 0.244 จึงเห็นความสัมพันธ์ของขนาดผลตอบแทนชัดกว่าความสัมพันธ์ของผลตอบแทนดิบ ค่านี้เป็นผลของช่วงข้อมูลที่แสดง ไม่ใช่ค่าคงที่ของตลาด

ดู [ที่มาของชุดข้อมูล](https://bashtage.github.io/arch/univariate/univariate_volatility_modeling.html#setup) หรือ [ดาวน์โหลดราคาปิดและรายละเอียดการคำนวณ](../data/sp500-daily.json) เพื่อคำนวณซ้ำได้

ถ้าเก็บผลตอบแทนทุกค่าไว้แล้วสับลำดับวัน ค่าเฉลี่ย SD และ histogram จะเหมือนเดิม แต่วันที่แกว่งแรงจะกระจายไปอยู่คนละตำแหน่ง Histogram จึงแยกข้อมูลสองลำดับนี้ไม่ได้

In [6]:
returns = sp500_returns
print(f"S&P 500: {len(returns):,} daily log returns, {sp500_dates[0]} to {sp500_dates[-1]}")
print("Closing price index; excludes dividends. Source:", sp500_data['source'])
permuted = shuffle(returns)
assert sorted(returns)==sorted(permuted)
before, after = moments(returns),moments(permuted)
for key in before:
    close(before[key],after[key])
for label, values in [('Original',returns),('Shuffled',permuted)]:
    print(label,moments(values))
    print(f"lag 1: r={acf(values)[1]:.6f}, |r|={acf([abs(r) for r in values])[1]:.6f}")
print("The same values in a different order: moments/histogram unchanged.")

S&P 500: 5,030 daily log returns, 1999-01-05 to 2018-12-31
Closing price index; excludes dividends. Source: Yahoo Finance via the arch 8.0.0 bundled example
Original {'mean': 0.00014186059322427583, 'sd': 0.01203839301555574, 'variance': 0.0001448940946859679, 'kurtosis': 11.169196103558116}
lag 1: r=-0.070084, |r|=0.244257
Shuffled {'mean': 0.00014186059322427583, 'sd': 0.01203839301555574, 'variance': 0.0001448940946859679, 'kurtosis': 11.169196103558116}
lag 1: r=-0.005068, |r|=0.002085
The same values in a different order: moments/histogram unchanged.


## วัดความสัมพันธ์ข้ามเวลาด้วย ACF

[Autocorrelation function หรือ ACF](../glossary.html#autocorrelation) วัดความสัมพันธ์ระหว่างค่าที่ห่างกัน k ช่วงเวลา นิยาม sample ACF ที่ใช้ในบทนี้คือ

$$
\widehat\rho_k=
\frac{\sum_{t=k+1}^{n}(r_t-\bar r)(r_{t-k}-\bar r)}
{\sum_{t=1}^{n}(r_t-\bar r)^2},\qquad k=1,\ldots,m.
$$

เราใช้ค่าเฉลี่ยของชุดเต็มและตัวหารชุดเต็มทุก lag นิยามนี้อาจต่างเล็กน้อยจากการใช้ Pearson correlation กับสองช่วงที่ตัดแล้ว เช่นฟังก์ชัน CORREL ที่หา mean ของแต่ละช่วงใหม่ สำหรับข้อมูลคงที่ทุกค่า ตัวหารเป็นศูนย์และ ACF ไม่มีนิยาม

ถ้าค่าต่าง ๆ เป็น iid และมีเงื่อนไขโมเมนต์ที่เหมาะสม กรอบอ้างอิงราย lag สำหรับตัวอย่างขนาดใหญ่ประมาณได้ด้วย \(\pm1.96/\sqrt n\) เช่น n=5,030 ในตัวอย่าง S&P 500 ให้ประมาณ ±0.028 ดูวิธีสร้างกรอบใน [NIST, Autocorrelation Plot](https://www.itl.nist.gov/div898/handbook/eda/section3/eda331.htm)

กรอบนี้ใช้กับแต่ละ lag แยกกัน เมื่อดู 20 หรือ 30 lag พร้อมกัน โอกาสเห็นจุดหลุดกรอบโดยบังเอิญจะเพิ่มขึ้น ถ้า variance เปลี่ยนตามข้อมูลในอดีต หรือมี conditional heteroskedasticity การใช้กรอบ iid กับผลตอบแทนดิบก็อาจทำให้สรุปผลคลาดเคลื่อน ส่วนค่าที่อยู่ในกรอบยังไม่เพียงพอจะยืนยันว่าข้อมูลเป็นอิสระ

Box–Pierce และ Ljung–Box รวม ACF หลาย lag เป็นสถิติเดียว

$$
Q_{\rm BP}=n\sum_{k=1}^{m}\widehat\rho_k^2,\qquad
Q_{\rm LB}=n(n+2)\sum_{k=1}^{m}\frac{\widehat\rho_k^2}{n-k}.
$$

ภายใต้สมมติฐานหลักและเงื่อนไขที่เหมาะสมของ white-noise/iid benchmark ค่าสถิติมีการแจกแจงอ้างอิงโดยประมาณแบบ \(\chi_m^2\) หากทดสอบ residuals จากโมเดลที่ประมาณพารามิเตอร์แล้ว ต้องปรับองศาอิสระและเงื่อนไขตามโมเดล ผลทดสอบใช้ตัดสินสมมติฐานเรื่องความสัมพันธ์ที่ตั้งไว้ การจะสรุปว่าทำนายราคาได้ยังต้องทดสอบการพยากรณ์แยกต่างหาก

ลองคำนวณ ACF และ Q อีกครั้งหลังแทน r ด้วย \(|r|\) หรือ \(r^2\) ถ้า r เป็น iid ฟังก์ชันเหล่านี้ก็ควรเป็น iid เช่นกัน ความสัมพันธ์ในขนาดผลตอบแทนจึงเป็นหลักฐานที่ใช้โต้แย้งสมมติฐาน iid ของ r ได้ แม้ ACF ของ r เองจะต่ำ

In [7]:
for label, values in [('r',returns),('|r|',[abs(r) for r in returns]),('r^2',[r*r for r in returns])]:
    bp, lb = portmanteau(values,20)
    print(f"{label}: Box-Pierce Q(20)={bp:.4f}; Ljung-Box Q(20)={lb:.4f}")
print(f"Pointwise iid reference +/-1.96/sqrt(n) = {1.96/math.sqrt(len(returns)):.6f}")
print("No iid-based p-values claimed for this market return series with changing volatility.")
assert acf([3,3,3],1)==[None,None]

r: Box-Pierce Q(20)=115.9251; Ljung-Box Q(20)=116.1892
|r|: Box-Pierce Q(20)=7858.5035; Ljung-Box Q(20)=7876.6039
r^2: Box-Pierce Q(20)=7012.6240; Ljung-Box Q(20)=7028.4653
Pointwise iid reference +/-1.96/sqrt(n) = 0.027636
No iid-based p-values claimed for this market return series with changing volatility.


## หางหนาและ kurtosis

สำหรับ Standard Normal โอกาสอยู่ห่างค่าเฉลี่ยเกิน 3 SD ทั้งสองด้านรวมประมาณ 0.2700% และเกิน 4 SD ประมาณ 0.00633% ถ้าสุ่มอิสระวันละหนึ่งค่าตลอด 252 วัน จำนวนเหตุการณ์เกิน 3 SD คาดหมายอยู่ที่ประมาณ 0.68 ครั้งต่อปี ตัวเลขนี้มาจากแบบจำลอง ไม่ใช่อัตราที่ตลาดต้องเกิดจริง

การแจกแจงที่มี [fat tails](../glossary.html#fat-tails) ให้โอกาสเหตุการณ์รุนแรงมากกว่า Normal ที่ใช้เทียบ หากตัดวันที่ร่วงแรงออกเพียงเพราะอยู่นอก 3 SD เราอาจเสียข้อมูลที่ต้องใช้ประเมินความเสี่ยงไปด้วย ก่อนตัดข้อมูล ให้แยกราคาที่ผิดหรือค้างจากครั้งก่อน และราคาที่ยังไม่ปรับการแตกหุ้น ออกจากราคาที่เคลื่อนไหวแรงจริง

Kurtosis แบบ population คือ

$$
\kappa=\frac{\mathbb E[(r-\mu)^4]}{\operatorname{Var}(r)^2}.
$$

Normal มี κ=3 และ excess kurtosis เท่ากับ κ−3 ในตัวทดลองใช้ moment estimator \(\widehat\kappa=m_4/m_2^2\) โดย \(m_j=n^{-1}\sum_t(r_t-\bar r)^j\) ส่วน SD ที่รายงานใช้ตัวหาร n−1 โปรแกรมที่ปรับ small-sample bias หรือรายงานเฉพาะ excess kurtosis จะให้ตัวเลขคนละนิยาม

Kurtosis ไวต่อข้อมูลปลายหางเพราะยกกำลังสี่ ตัวเลขสูงไม่ได้บอกความเบ้ และไม่ได้ระบุว่า distribution ต้องเป็น Student-t หรือมี variance อนันต์ การทดสอบ normality ด้วย \((\widehat\kappa-3)/\sqrt{24/n}\) อาศัย iid Normal null และการประมาณตัวอย่างใหญ่ จึงไม่ควรใช้ standard error นี้ตรง ๆ กับข้อมูลที่มี volatility clustering

In [8]:
for threshold in [3,4]:
    probability = math.erfc(threshold/math.sqrt(2))
    print(f"Normal two-sided P(|Z|>{threshold})={probability:.8%}; expected per 252 draws={252*probability:.6f}")
print(f"Moment kurtosis: {moments(returns)['kurtosis']:.6f}; excess: {moments(returns)['kurtosis']-3:.6f}")
print("Moment estimator uses m4/m2^2; reported sample SD uses n-1.")

Normal two-sided P(|Z|>3)=0.26997961%; expected per 252 draws=0.680349
Normal two-sided P(|Z|>4)=0.00633425%; expected per 252 draws=0.015962
Moment kurtosis: 11.169196; excess: 8.169196
Moment estimator uses m4/m2^2; reported sample SD uses n-1.


## Skewness บอกว่าหางด้านไหนยาวกว่า

เมื่อโมเมนต์อันดับสามมีค่าจำกัด population skewness และ moment estimator ของตัวอย่างคือ

$$
S=\frac{\mathbb E[(r-\mu)^3]}{\sigma^3},\qquad
\widehat S=\frac{m_3}{m_2^{3/2}}.
$$

ค่าบวกสอดคล้องกับความไม่สมมาตรด้านบวก ส่วนค่าลบสอดคล้องกับด้านขาดทุน โปรแกรมบางตัวปรับ bias ของตัวอย่าง จึงควรตรวจนิยามก่อนเทียบตัวเลข ถ้าสลับเครื่องหมายผลตอบแทนทุกค่า skewness จะเปลี่ยนเครื่องหมาย แต่ SD และ kurtosis เท่าเดิม

ตัวอย่างห้าค่า −2%, −1%, 0%, 1%, 6% มี skewness ประมาณ 1.0392 เมื่อกลับเครื่องหมายทุกค่าได้ −1.0392 ขณะที่ sample SD ยังเป็น 3.1145% และ kurtosis 2.6688 ทั้งคู่

Skewness ศูนย์อย่างเดียวไม่รับรอง symmetry และ symmetry ก็ยังไม่ได้แปลว่าเป็น Normal การมี finite sample moments ยังไม่พิสูจน์ว่า population moments มีค่าจำกัด เพราะข้อมูลที่เก็บมามีจำนวนจำกัดเสมอ

ภายใต้ iid Normal และตัวอย่างใหญ่ standard error ของ moment skewness ประมาณ √(6/n) ส่วน kurtosis ใช้ √(24/n) ข้อจำกัดเรื่อง volatility clustering ใช้กับทั้งสองสูตร ดูนิยามใน [NIST, Measures of Skewness and Kurtosis](https://www.itl.nist.gov/div898/handbook/eda/section3/eda35b.htm)

## ดูรูปการแจกแจงให้พ้นจากสถิติไม่กี่ค่า

Histogram ขึ้นกับความกว้างและจุดเริ่มของ bin ถ้าเปลี่ยน bin แล้วข้อสรุปเรื่องหลายยอดหายไป ควรตรวจวิธีแสดงผลก่อนตีความว่าเป็นหลายสภาวะ ส่วน density estimate ก็ขึ้นกับ bandwidth เช่นกัน

Q–Q plot เทียบ quantiles ของข้อมูลกับการแจกแจงที่เลือก จุดปลายที่เบนจากเส้นตรงช่วยให้เห็นความต่างของหาง ดูตัวอย่างใน [บท VaR/ES](../value-at-risk-expected-shortfall.html#distribution-checks) ถ้าต้องการวัดหางโดยตรง อาจรายงานสัดส่วนที่เกิน ±3 SD และแยกสองหางออกจากกันด้วย

การมี kurtosis สูงบอกว่ากำลังสี่ของค่าที่ห่าง mean มีน้ำหนักมาก แต่ไม่ได้บอกรูปทรงทุกส่วนหรือรับรองว่าจุดยอดต้องสูงเสมอไป ควรอ่านควบคู่กับ quantiles, ความเบ้ และกราฟ

การเพิ่มช่วงถืออาจทำให้ส่วนกลางดูใกล้ Normal ขึ้น ขณะที่หางยังต่างอยู่ อีกทั้งผลตอบแทนหลายวันที่คำนวณแบบหน้าต่างซ้อนกันใช้ข้อมูลร่วมกัน จึงเกิด dependence จากการสร้างข้อมูลได้เอง ต้องระบุว่าใช้ช่วงทับกันหรือไม่

## เลือก probability distribution สำหรับผลตอบแทน

การแจกแจง marginal อธิบายค่าที่พบเมื่อรวมข้อมูล แต่ยังต้องมีแบบจำลองความสัมพันธ์ข้ามเวลาเพิ่มเติม ตารางนี้เปรียบเทียบตัวเลือกโดยระบุเงื่อนไขของโมเมนต์

| การแจกแจง | จุดที่ใช้ได้ | ข้อจำกัดหรือเงื่อนไข |
|---|---|---|
| Normal | คำนวณสะดวก มี mean/variance และทุกโมเมนต์ | Symmetric และ kurtosis 3 จึงอาจครอบคลุมหางไม่พอ |
| Student-t | Symmetric และหางลดช้ากว่า Normal | Mean มีเมื่อ ν>1, variance มีเมื่อ ν>2, kurtosis มีเมื่อ ν>4 |
| Skewed distributions | แยกพฤติกรรมหางบวกและลบได้ | ต้องระบุ parameterization และตรวจโมเมนต์ของแบบที่ใช้ |
| Normal variance mixture | รวมหลายสเกลของ volatility | ต้องกำหนด dynamics ของสภาวะเพิ่มเพื่ออธิบาย clustering |
| Stable distributions ที่ α<2 | ใช้ศึกษาหางแบบกำลังและการรวมตัวแปรในตระกูล stable | Variance อนันต์ จึงใช้ SD และ √time scaling แบบ finite variance ไม่ได้ |

ถ้า T มี Student-t degrees of freedom ν>2 ตัวแปร \(Z=T\sqrt{(\nu-2)/\nu}\) จะมี variance 1 จึงเทียบกับ Standard Normal ที่สเกลเท่ากันได้ โดยมี density

$$
f_Z(z)=\frac{\Gamma((\nu+1)/2)}{\sqrt{\pi(\nu-2)}\,\Gamma(\nu/2)}
\left(1+\frac{z^2}{\nu-2}\right)^{-(\nu+1)/2}.
$$

สำหรับ ν>4 ค่า kurtosis เท่ากับ \(3+6/(\nu-4)\) เช่น ν=5 ให้ 9 และ ν=10 ให้ 4 ที่ 2<ν≤4 variance ยังมีค่าจำกัด แต่ fourth moment ไม่มีค่าจำกัด ส่วน symmetry ของ Student-t ยังอยู่แม้ third moment ไม่ได้มีอยู่ ดูเงื่อนไขจาก [NIST, t Distribution](https://www.itl.nist.gov/div898/handbook/eda/section3/eda3664.htm)



เส้นทฤษฎีที่ mean 0 และ variance 1 เท่ากัน ใช้แกนแนวตั้ง logarithmic ในช่องหางเพื่อเห็นความต่างที่ค่า density ต่ำ ไม่มีข้อมูลตลาดในภาพ

ราคาที่เป็น Lognormal ไม่ได้หมายความว่า returns เป็น Lognormal: ภายใต้ GBM พารามิเตอร์คงที่ log return เป็น Normal ส่วน 1+simple return เป็น Lognormal การใช้ Normal หรือ Student-t ที่รองรับค่าทั้งเส้นจำนวนกับ simple return ยังอาจให้ R<−100% จึงต้องตรวจความหมายทางเศรษฐกิจของตัวแปรที่เลือก

การเลือก distribution ควรเทียบทั้งส่วนกลางและหางที่ใช้ตัดสินใจ รวมถึงตรวจข้อมูลนอกช่วงประมาณค่า การเพิ่มพารามิเตอร์ให้ fit ข้อมูลเดิมดีขึ้นอย่างเดียวไม่รับรองว่าพยากรณ์ดีขึ้น

In [9]:
for nu in [5,10]:
    print('Student-t degrees of freedom',nu,'variance-normalized kurtosis',3+6/(nu-4))
# Midpoint integration checks unit area and variance. Finite tail cutoff is explicit.
step=.002
area=second=0
for i in range(100000):
    x=-100+(i+.5)*step;mass=standardized_t_pdf(x,5)*step
    area+=mass;second+=x*x*mass
assert abs(area-1)<1e-7 and abs(second-1)<5e-5
print('t(5) density numerical integral on [-100,100]:',area,'second moment:',second)

Student-t degrees of freedom 5 variance-normalized kurtosis 9.0
Student-t degrees of freedom 10 variance-normalized kurtosis 4.0
t(5) density numerical integral on [-100,100]: 0.9999999994710693 second moment: 0.9999911834991227


## Normal หลายสเกลรวมกันให้หางหนาได้

สมมติว่าในแต่ละสภาวะผลตอบแทนมีค่าเฉลี่ยศูนย์ และ

$$
r\mid\sigma\sim N(0,\sigma^2).
$$

เมื่อรวมหลายสภาวะเข้าด้วยกัน \(\mathbb E[r^2]=\mathbb E[\sigma^2]\) และ \(\mathbb E[r^4]=3\mathbb E[\sigma^4]\) จึงได้

$$
\kappa=3\frac{\mathbb E[\sigma^4]}{\mathbb E[\sigma^2]^2}
=3\left(1+\frac{\operatorname{Var}(\sigma^2)}{\mathbb E[\sigma^2]^2}\right)\geq3.
$$

ค่า κ จะมากกว่า 3 เมื่อ variance ของแต่ละสภาวะต่างกันจริง โดยโมเมนต์ที่ใช้ต้องมีค่าจำกัด สูตรนี้ยังสมมติให้ทุกสภาวะมีค่าเฉลี่ยเดียวกัน หากค่าเฉลี่ยต่างกัน ต้องคำนวณโมเมนต์ของ mixture ใหม่

ตัวอย่างให้ 80% ของวันมี SD 0.5% และอีก 20% มี SD 2.5% จะได้ variance รวม \(0.8(0.005)^2+0.2(0.025)^2=0.000145\) หรือ SD ประมาณ 1.2042% ต่อวัน ส่วน kurtosis เท่ากับประมาณ 11.219 เทียบกับ 3 ของ Normal ที่มี SD เท่ากัน

การสุ่มเลือกสภาวะใหม่อย่างอิสระทุกวันให้ mixture แบบนี้ได้ โดยไม่เกิด volatility clustering หากให้สภาวะเดิมอยู่นานหลายวัน ก็เกิด clustering ได้เช่นกัน สัดส่วนของแต่ละสภาวะกำหนดการแจกแจงรวม ส่วนการเรียงสภาวะตามเวลากำหนดความสัมพันธ์ข้ามวัน

การเปลี่ยน volatility เป็นกลไกหนึ่งที่สร้างหางหนาได้ ส่วน jumps ความเบ้ และหางของช็อก \(z_t\) อาจเพิ่มความผิดปกติจาก Normal อีก การหารด้วย volatility ที่ประมาณมาแล้วไม่ได้รับประกันว่าข้อมูลที่เหลือจะเป็น Normal

In [10]:
mixture = variance_mixture(.2,5)
variance = .8*.005**2+.2*.025**2
close(variance,.000145)
close(mixture['kurtosis'],11.218787158145064)
close(variance_mixture(.2,1)['kurtosis'],3)
print(f"Daily SD={math.sqrt(variance):.6%}; kurtosis={mixture['kurtosis']:.6f}")
print(f"Two-sided tail beyond 3 pooled SD={mixture['tail'](3):.6%}")
print("Mixture probabilities alone do not specify temporal dependence.")

Daily SD=1.204159%; kurtosis=11.218787
Two-sided tail beyond 3 pooled SD=2.969206%
Mixture probabilities alone do not specify temporal dependence.


## เมื่อรวมผลตอบแทนหลายวัน

เมื่อรวมผลตอบแทนหลายวัน รูปการแจกแจงอาจเข้าใกล้ Normal ภายใต้เงื่อนไข central limit theorem เช่น variance จำกัดและ dependence ที่ไม่รุนแรงเกินไป แต่จำนวนวันเท่าใดจึงใกล้พอขึ้นกับข้อมูล โดยเฉพาะความแม่นที่ต้องการตรงหาง การเห็น monthly kurtosis ใกล้ 3 ไม่ได้พิสูจน์ว่า daily returns เป็น Normal

GBM ที่มีพารามิเตอร์คงที่อย่างในบทก่อนอธิบาย volatility clustering ไม่ได้ เราอาจขยายแบบจำลอง continuous-time ให้ volatility เปลี่ยนตามเวลา หรือใช้แบบจำลอง discrete-time อย่าง ARCH/GARCH ที่ให้ conditional variance ตอบสนองต่อข้อมูลอดีต

สำหรับ VaR/ES การเปลี่ยนจาก unconditional SD หนึ่งค่าไปใช้ volatility ที่คาดการณ์ ณ วันนั้นจะเปลี่ยนความเสี่ยงที่รายงาน การเลือก distribution ของ standardized shocks ก็ยังมีผลต่อหาง ส่วนการปรับสูตรราคา Option ต้องพิจารณาความน่าจะเป็นสำหรับการตั้งราคาและ volatility risk premium เพิ่มเติม ไม่สามารถนำผลประมาณภายใต้ physical measure ไปแทน risk-neutral dynamics ทั้งชุดทันที

## เมื่อหนึ่งวันมีราคาหลายพันค่า

ข้อมูลซื้อขายระหว่างวันมีทั้งราคา bid, ask และ transaction price ผู้ซื้ออาจซื้อที่ ask ผู้ขายอาจขายที่ bid และบางธุรกรรมเกิดระหว่างสองราคา ระยะห่างระหว่างรายการซื้อขายไม่คงที่ บางนาทีอาจไม่มี trade แต่มีการปรับ quote หลายครั้ง

การคำนวณผลตอบแทนจาก trade ทุกคู่จึงผสมการเคลื่อนของมูลค่ากับผลของ bid–ask spread ตัวอย่างมูลค่าแฝงอยู่ที่ 100 คงที่ แต่ราคา trade สลับ 99.99 กับ 100.01 จะให้ผลตอบแทนสลับบวกและลบ ทั้งที่ราคาแฝงไม่ได้เปลี่ยน การเห็น short-lag correlation ติดลบในข้อมูลแบบนี้ยังไม่ใช่กลยุทธ์กำไรหลังหัก spread

ก่อนคำนวณต้องเลือกว่าใช้ราคาซื้อขายจริง ราคากึ่งกลาง bid–ask หรือ quote ฝั่งใด แล้วจัดเขตเวลาและ daylight saving ให้ตรงกัน พร้อมทำเครื่องหมายราคาที่ค้างจากครั้งก่อนและแยกช่วงตลาดเปิดกับปิด การเติมราคาล่าสุดในนาทีที่ไม่มีการซื้อขายอาจเพิ่มผลตอบแทนศูนย์จำนวนมาก ส่วน tick size ทำให้ราคาขยับเป็นขั้น

Stylized facts ที่เห็นในรายวันยังพบในข้อมูลระหว่างวันได้ แต่ microstructure มีอิทธิพลมากขึ้นตามความถี่ การดู dependence ยังต้องคำนึงถึงเวลาในวัน เพราะนาทีใกล้เปิดตลาดกับนาทีกลางวันอาจมีสเกลความผันผวนต่างกันซ้ำทุกวัน

## จังหวะในวันและข่าวที่ออกตามเวลา

หลายตลาดมี volatility สูงใกล้เปิดหรือปิดตลาด และมีจุดสูงขึ้นรอบข่าวเศรษฐกิจบางรายการ รูปแบบขึ้นกับตลาด ตารางซื้อขาย และช่วงที่ศึกษา ตลาด FX ตลอดวันยังมีช่วงที่ผู้ค้าจากหลายภูมิภาคทำงานทับกัน จึงไม่ควรใช้รูป U หรือเวลาออกข่าวชุดเดียวกับทุกตลาด

ตัวอย่างในงานของ Taylor แยกผลของเวลาเปิดตลาด ข่าวเศรษฐกิจ และช่วงที่ตลาดต่างประเทศเปิดซ้อนกัน เวลาที่บันทึกในตัวอย่างเก่าเป็นข้อมูลของตลาดและช่วงศึกษานั้น การเปรียบเทียบข้ามประเทศต้องจัดการช่วงที่เปลี่ยน daylight saving คนละวันด้วย

ให้ \(v_j\) เป็น variance ของผลตอบแทนช่วง j ที่ประมาณจากหลายวัน สัดส่วน variance ภายในช่วงเปิดตลาดคือ

$$
a_j=\frac{v_j}{\sum_{k=1}^{M}v_k},\qquad \sum_{j=1}^{M}a_j=1.
$$

สมมติว่าผลตอบแทนแต่ละช่วงไม่มี covariance ต่อกัน และ variance รวมตลอดช่วงเปิดตลาดของวันเป้าหมายเท่ากับ h เราจะจัด variance ให้ช่วง j ตามสัดส่วนนี้ได้เป็น \(h a_j\) ส่วน SD เท่ากับ \(\sqrt h\sqrt{a_j}\)

หากผลตอบแทนระหว่างช่วงมี covariance ต้องบวกพจน์เหล่านั้นด้วย จึงจะได้ variance ของผลตอบแทนรวมตลอดช่วงเปิดตลาด



สร้าง profile สมมติจากเส้นลดลงหลังเปิดตลาด เส้นเพิ่มขึ้นก่อนปิด และส่วนที่สูงขึ้นใกล้ช่วงที่ 31 แล้ว normalize ให้แต่ละชุดรวมเป็น 100% เส้นนี้แสดงการจัดสัดส่วนเท่านั้น ไม่ได้อ้างว่า variance รวมของวันข่าวเท่ากับวันอื่น และเวลาในภาพไม่ใช่ตารางประกาศข่าวจริง

ความเคลื่อนไหวหลังข่าวขึ้นกับส่วนที่ต่างจากความคาดหวัง ไม่ใช่เพียงตัวเลขที่ประกาศหรือจำนวนพาดหัวข่าว หากจะศึกษาผลของข่าว ต้องเก็บ announcement time, ค่าที่ประกาศ และค่าคาดการณ์ที่มีอยู่ก่อนข่าวให้ตรงกัน ดูตัวอย่างการแยก news surprise ใน [Andersen, Bollerslev, Diebold และ Vega (2003)](https://public.econ.duke.edu/~boller/research.html)

In [11]:
for news in [False,True]:
    profile = intraday_profile(news)
    close(sum(profile),1)
    daily_sd = .01
    interval_sd = [daily_sd*math.sqrt(a) for a in profile]
    close(sum(s*s for s in interval_sd),daily_sd**2)
    print(f"News bump={news}: largest share {max(profile):.6%}; sum {sum(profile):.6%}")
print("Variance weights sum to one; SD multipliers are their square roots.")

News bump=False: largest share 3.553526%; sum 100.000000%
News bump=True: largest share 4.719755%; sum 100.000000%
Variance weights sum to one; SD multipliers are their square roots.


## วัดความผันผวนระหว่างวันด้วย realized variance

ให้ \(r_{t,j}\) เป็น log return ระหว่างสองราคาที่เก็บต่อกันในวัน t ใช้ N ช่วงย่อย เรานิยาม

$$
\operatorname{RV}_t=\sum_{j=1}^{N}r_{t,j}^2,\qquad
\operatorname{RVol}_t=\sqrt{\operatorname{RV}_t}.
$$

[Realized variance](../glossary.html#realized-variance) มีหน่วยเป็นผลตอบแทนยกกำลังสอง ส่วน realized volatility หรือ realized SD มีหน่วยเดียวกับผลตอบแทน ชื่อ RV ในงานบางชิ้นใช้เรียกไม่เหมือนกัน จึงควรอ่านนิยามก่อนเทียบตัวเลข บทนี้สงวน RV ไว้สำหรับผลรวมกำลังสอง

ตัวอย่าง log returns ภายในวันเป็น 1%, −1%, 1%, −1% จะรวมได้ศูนย์ ราคาจึงกลับมาที่เดิม แต่ \(\operatorname{RV}=4(0.01)^2=0.0004\) และ \(\sqrt{\operatorname{RV}}=2\%\) การยกกำลังสองผลตอบแทนต้น–ปลายวันจะให้ศูนย์ เพราะไม่เห็นการแกว่งระหว่างทาง



ทั้งสองเส้นมีผลตอบแทนย่อยสี่ช่วง และผลตอบแทนต้น–ปลายวันเป็นศูนย์เหมือนกัน คำนวณ √RV จากค่าทศนิยม แล้วจึงแปลงเป็นเปอร์เซ็นต์

สำหรับ log price ที่เป็น continuous semimartingale และไม่มี measurement noise เมื่อเก็บถี่ขึ้น RV จะลู่เข้า quadratic variation ซึ่งเท่ากับ integrated variance \(\int\sigma_s^2ds\) ของช่วงนั้น หากมี jumps ขีดจำกัดจะรวมกำลังสองของขนาด jump ด้วย

$$
\operatorname{RV}_t\ \longrightarrow\
\int_t^{t+1}\sigma_s^2\,ds+\sum_{t<s\leq t+1}(\Delta\log P_s)^2.
$$

ความเชื่อมโยงกับ quadratic variation อธิบายไว้ในบท [Applied Stochastic Calculus](../applied-stochastic-calculus.html) ส่วนการใช้ realized measures ประมาณและพยากรณ์ volatility พัฒนาต่อใน [Andersen, Bollerslev, Diebold และ Labys (2003)](https://econ.duke.edu/~boller/Published_Papers/ecta_03.pdf)

RV ที่คำนวณจากข้อมูลช่วงตลาดเปิดครอบคลุมเฉพาะช่วงนั้น หากคูณ \(\sqrt{252}\) เพื่อแปลงเป็นรายปี เช่น จาก 1% เป็นประมาณ 15.87% ก็ยังไม่รวมความเสี่ยงข้ามคืน การแปลงนี้อาศัยสมมติฐานเรื่องการปรับสเกลและจำนวนวันซื้อขาย

เราอาจบวก overnight return ยกกำลังสองเพื่อรวมการเปลี่ยนแปลงข้ามคืนได้บางส่วน แต่ยังขาดรายละเอียดการเคลื่อนไหวระหว่างทาง

In [12]:
for scale in [.01,.002]:
    log_prices = [0]
    for r in [scale,-scale,scale,-scale]:
        log_prices.append(log_prices[-1]+r)
    stats = realized_variance(log_prices)
    close(log_prices[-1],0)
    close(stats['volatility'],2*scale)
    print(f"Four returns +/-{scale:.2%}: RV={stats['variance']:.7f}, sqrt(RV)={stats['volatility']:.4%}, closing return=0")
print(f"1% session SD * sqrt(252) = {.01*math.sqrt(252):.4%}; excludes overnight")

Four returns +/-1.00%: RV=0.0004000, sqrt(RV)=2.0000%, closing return=0
Four returns +/-0.20%: RV=0.0000160, sqrt(RV)=0.4000%, closing return=0
1% session SD * sqrt(252) = 15.8745%; excludes overnight


## ราคาถี่ขึ้นมีทั้งข้อมูลเพิ่มและ noise เพิ่ม

ให้ \(p_j^*\) เป็น log price แฝง และราคาที่สังเกตเป็น \(p_j=p_j^*+\epsilon_j\) โดย \(\epsilon_j\) แทน [microstructure noise](../glossary.html#microstructure-noise) สมมติ noise มีค่าเฉลี่ยศูนย์ variance \(\eta^2\) เป็นอิสระข้ามเวลาและจากราคาแฝง จะได้ observed return

$$
r_j=\Delta p_j^*+\epsilon_j-\epsilon_{j-1}.
$$

ผลต่าง noise มี variance \(2\eta^2\) ดังนั้นบน sampling grid ที่มี N ช่วง

$$
\mathbb E[\operatorname{RV}_{\rm observed}]
=\mathbb E[\operatorname{RV}_{\rm latent}]+2N\eta^2.
$$

เมื่อเก็บราคาถี่ขึ้น N เพิ่ม ส่วน bias จาก noise จึงเพิ่มตามในแบบจำลองนี้ ถ้า latent returns ไม่มี serial covariance ส่วน noise ยังทำให้ covariance ของผลตอบแทนติดกันเท่ากับ \(-\eta^2\) ได้ งานเรื่องการเลือก sampling frequency จึงต้องพิจารณาทั้งความละเอียดและ microstructure noise ดู [Aït-Sahalia, Mykland และ Zhang, *How Often to Sample a Continuous-Time Process…*](https://www.nber.org/papers/w9611)

เมื่อ noise เป็นศูนย์ การเก็บถี่ขึ้นช่วยประมาณ integrated variance ภายใต้สมมติฐานของโมเดลได้ แต่ RV บนเส้นทางเดียวไม่จำเป็นต้องขยับเข้าหาค่าจริงทีละขั้นอย่างสม่ำเสมอ เมื่อ noise ไม่เป็นศูนย์ การเลือกข้อมูลถี่ที่สุดก็อาจเพิ่มความผิดพลาด ยังมีวิธีอย่าง subsampling, realized kernels และ pre-averaging สำหรับจัดการปัญหานี้ โดยแต่ละวิธีมีเงื่อนไขของตนเอง

In [13]:
for noise_bps in [0,3,10]:
    data = intraday_sample(noise_bps)
    for stride in [1,5,15,30,390]:
        latent = realized_variance(data['latent'],stride)
        observed = realized_variance(data['observed'],stride)
        bias = 2*observed['count']*data['eta']**2
        if noise_bps==0:
            close(latent['variance'],observed['variance'])
        print(f"Noise={noise_bps} bps, every {stride} min: N={observed['count']}, latent SD={latent['volatility']:.5%}, observed SD={observed['volatility']:.5%}, expected noise RV={bias:.8f}")
close(2*390*.0003**2,.0000702)
close(2*78*.0003**2,.00001404)
print("Observed-minus-latent RV of one realization need not equal its expectation.")

Noise=0 bps, every 1 min: N=390, latent SD=1.00759%, observed SD=1.00759%, expected noise RV=0.00000000
Noise=0 bps, every 5 min: N=78, latent SD=0.99700%, observed SD=0.99700%, expected noise RV=0.00000000
Noise=0 bps, every 15 min: N=26, latent SD=1.08954%, observed SD=1.08954%, expected noise RV=0.00000000
Noise=0 bps, every 30 min: N=13, latent SD=1.11371%, observed SD=1.11371%, expected noise RV=0.00000000
Noise=0 bps, every 390 min: N=1, latent SD=1.51420%, observed SD=1.51420%, expected noise RV=0.00000000
Noise=3 bps, every 1 min: N=390, latent SD=1.00759%, observed SD=1.31379%, expected noise RV=0.00007020
Noise=3 bps, every 5 min: N=78, latent SD=0.99700%, observed SD=1.06703%, expected noise RV=0.00001404
Noise=3 bps, every 15 min: N=26, latent SD=1.08954%, observed SD=1.10030%, expected noise RV=0.00000468
Noise=3 bps, every 30 min: N=13, latent SD=1.11371%, observed SD=1.08931%, expected noise RV=0.00000234
Noise=3 bps, every 390 min: N=1, latent SD=1.51420%, observed SD=1

## หารด้วย volatility แล้วเหลืออะไร

เมื่อรวมช่วง volatility สูงกับต่ำ ผลตอบแทนดิบอาจมีหางหนามาก การหารด้วย volatility ของแต่ละช่วงช่วยลดความต่างของสเกล ในข้อมูลที่ Taylor ยกมา realized volatility มีการแจกแจงเบ้ขวา ส่วน standardized returns ใกล้ Normal มากขึ้น ผลที่ได้ยังขึ้นกับสินทรัพย์ วิธีประมาณ และช่วงข้อมูล

ถ้าใช้ \(\sqrt{\operatorname{RV}_t}\) ซึ่งคำนวณจากราคาตลอดวัน t ตัวหารจะรู้ได้เมื่อจบวัน การดู \(r_t/\sqrt{\operatorname{RV}_t}\) จึงเป็นการวิเคราะห์ย้อนหลัง หากต้องการตั้ง VaR ก่อนวันเริ่ม ต้องใช้ \(\widehat\sigma_{t\mid t-1}\) ที่ประมาณได้จากข้อมูลก่อนหน้านั้น

$$
z_t^{\rm forecast}=\frac{r_t-\widehat\mu_{t\mid t-1}}{\widehat\sigma_{t\mid t-1}}.
$$

จากนั้นตรวจ distribution และ ACF ของทั้ง \(z_t\) กับ \(z_t^2\) เพื่อดูว่าโมเดลอธิบายความสัมพันธ์ที่ต้องการได้แค่ไหน นอกจากนี้ผลตอบแทนและ RV ต้องครอบคลุมช่วงเวลาเดียวกัน การหาร close-to-close return ด้วย open-to-close realized volatility จะมี overnight component อยู่ในตัวเศษเพียงด้านเดียว

ACF ของ volatility ที่ลดลงช้าเป็นลักษณะ persistence ที่ควรตรวจต่อ แต่กราฟ ACF เส้นเดียวแยก true long memory ออกจาก structural breaks หรือการผสมหลายสภาวะได้ไม่เด็ดขาด และการเห็น log volatility ดูใกล้ Normal ก็ยังไม่พิสูจน์ว่า volatility เป็น Lognormal ทุกช่วงเวลา

In [14]:
# A separate simulation has KNOWN sigma. These are not standardized S&P 500 returns.
synthetic_returns = clustered_returns()
known_sigma = [.005 if (i//50)%2==0 else .025 for i in range(len(synthetic_returns))]
standardized = [r/s for r,s in zip(synthetic_returns,known_sigma)]
print("Known-sigma standardized simulation:",moments(standardized))
print(f"ACF of squared standardized shocks, lag 1={acf([z*z for z in standardized])[1]:.6f}")
print("In market data, same-day realized volatility is known only after observing that day.")

Known-sigma standardized simulation: {'mean': -0.040632624930982665, 'sd': 0.971290017192985, 'variance': 0.941831957002918, 'kurtosis': 2.8061421741970656}
ACF of squared standardized shocks, lag 1=-0.048319
In market data, same-day realized volatility is known only after observing that day.


## เหตุการณ์สั้น ๆ ที่ข้อมูลรายวันมองไม่เห็น

ใน Flash Crash วันที่ 6 พฤษภาคม 2010 ราคาสินทรัพย์สหรัฐฯ เคลื่อนลงและฟื้นกลับภายในวัน ราคาปิดจึงบอกได้เพียงส่วนหนึ่งของเหตุการณ์ รายงานร่วม [CFTC และ SEC (2010)](https://www.sec.gov/news/studies/2010/marketevents-report.pdf) ใช้ข้อมูลธุรกรรมและการทำงานของตลาดเพื่อศึกษาลำดับเหตุการณ์ระหว่างวัน

ข้อมูลความถี่สูงยังช่วยตรวจการเปลี่ยนแปลงรวดเร็วที่อาจเป็น price jump แต่จุดโดดในกราฟอาจมาจากข้อมูลผิด quote ที่ล้าสมัย หรือ spread ที่กว้างขึ้นด้วย การสรุปว่ามี jump ต้องอาศัยการตรวจข้อมูลและวิธีทดสอบที่คำนึงถึง noise กับ sampling frequency

ในแบบจำลองที่มี jumps RV จะนับกำลังสองของ jumps รวมอยู่ด้วย หากโจทย์ต้องการแยก continuous variation ออกจาก jump variation ต้องใช้ estimator เพิ่ม เช่น bipower variation ภายใต้เงื่อนไขของวิธีนั้น แทนการเรียก RV ทั้งก้อนว่า diffusion variance

เมื่อนำข้อมูลระหว่างวันไปพยากรณ์ความผันผวนวันถัดไป ให้ทดสอบกับข้อมูลนอกช่วงฝึก การเปรียบเทียบโมเดลต้องใช้ระยะเวลาพยากรณ์เดียวกัน และให้แต่ละโมเดลใช้ข้อมูลที่รู้ได้ ณ เวลาเดียวกัน

## ACF ของผลตอบแทนที่แปลงแล้ว

นอกจาก r เราอาจคำนวณ ACF ของ |r|, r² หรือ \(|r|^\delta\) เมื่อ δ>0 เพื่อดูความสัมพันธ์ของขนาดผลตอบแทน กำลังที่ต่างกันให้น้ำหนักกับเหตุการณ์รุนแรงต่างกัน r² ไวต่อค่าปลายหางมากกว่า |r| ส่วนการเลือก δ หลังลองหลายค่าแล้วต้องนับเป็นการค้นหาหลายสมมติฐานด้วย

Population ACF ของ r² ต้องมี Var(r²) จำกัด จึงต้องการ fourth moment ของ r ส่วน ACF ของ |r| ต้องมี second moment ของ r ในข้อมูล finite sample เราคำนวณได้แม้เงื่อนไขของประชากรอาจไม่ผ่าน จึงต้องระวังการตีความและการใช้ standard errors

หาก ACF ของ r ต่ำ แต่ของ |r| หรือ r² สูง แสดงว่าข้อมูลมีความสัมพันธ์ที่ ACF ของ r จับไม่ได้ ตัวทดลอง [สลับลำดับวัน](#clustering-lab) ให้เห็นผลนี้โดยเก็บค่าทุกตัวเหมือนเดิม แล้วเปลี่ยนเพียงลำดับเวลา

ภายใต้ stationary Gaussian process ที่มี mean ศูนย์ มีความสัมพันธ์ \(\operatorname{Corr}(r_t^2,r_{t-k}^2)=\rho_k^2\) เราจึงใช้กรณีนี้เป็น benchmark ได้ แต่หากช็อกไม่ Gaussian ต้องมีพจน์จาก fourth cumulant เพิ่มตาม [ภาคผนวก](#squared-linear-appendix)

In [15]:
for phi in [.6,-.6]:
    raw=arma11(phi,0)['acf']
    print('Gaussian AR(1), phi=',phi,'raw ACF lag1=',raw[1],'squared ACF lag1=',raw[1]**2)
close(.6**2,.36)

Gaussian AR(1), phi= 0.6 raw ACF lag1= 0.6 squared ACF lag1= 0.36
Gaussian AR(1), phi= -0.6 raw ACF lag1= -0.6 squared ACF lag1= 0.36


## Uncorrelated แต่ dependent: ตัวอย่างที่คำนวณได้

ให้ \(\varepsilon_t\overset{\rm iid}{\sim}N(0,1)\) และกำหนด \(X_t=\varepsilon_t\varepsilon_{t-1}\) จะได้ E[Xₜ]=0, E[Xₜ²]=1 และ E[Xₜ⁴]=9

ที่ lag 1 ผลคูณ \(X_tX_{t-1}=\varepsilon_t\varepsilon_{t-1}^2\varepsilon_{t-2}\) มีค่าคาดหมายศูนย์ เพราะ εₜ และ εₜ₋₂ เป็นอิสระและมี mean ศูนย์ lag ที่มากกว่านั้นก็มี covariance ศูนย์เช่นกัน

เมื่อยกกำลังสองกลับได้

$$
\mathbb E[X_t^2X_{t-1}^2]
=\mathbb E[\varepsilon_t^2]\mathbb E[\varepsilon_{t-1}^4]\mathbb E[\varepsilon_{t-2}^2]
=1\times3\times1=3.
$$
$$
\operatorname{Cov}(X_t^2,X_{t-1}^2)=3-1=2,\qquad
\operatorname{Corr}(X_t^2,X_{t-1}^2)=\frac{2}{9-1}=\frac14.
$$

นี่เป็นกระบวนการ strictly stationary และ white noise ตามนิยาม second moments แต่ไม่เป็นอิสระ เมื่อกำหนดข้อมูลอดีตเป็นช็อก ε ทั้งหมดจนถึง t−1 จะมี conditional mean ศูนย์ และ conditional variance εₜ₋₁² จึงมีความเสี่ยงที่เปลี่ยนตามข้อมูลเก่า

ตัวอย่างนี้ช่วยแยกการพยากรณ์ mean ออกจาก variance การพบ dependence ในข้อมูลตลาดยังต้องตรวจว่าเกิดจาก nonlinear dynamics, non-Gaussian shocks, ปฏิทิน หรือการเปลี่ยนสภาวะ การดู ACF เพียงชุดเดียวไม่สามารถเลือกคำอธิบายแทนการทดสอบเหล่านี้ได้

In [16]:
# X_t=e_t e_(t-1); iid standard Normal moments E[e^2]=1 and E[e^4]=3.
variance=1
fourth=3*3
cross_squared=1*3*1
squared_correlation=(cross_squared-variance**2)/(fourth-variance**2)
close(squared_correlation,.25)
print('Nonlinear product process: raw ACF lag1=0; squared ACF lag1=',squared_correlation)

Nonlinear product process: raw ACF lag1=0; squared ACF lag1= 0.25


## ภาคผนวก: ACF ที่เกิดจากวันในสัปดาห์

สมมติ \(r_t=m_{d(t)}+\varepsilon_t\) โดย m₁,…,m₅ เป็น mean ของห้าวัน และ εₜ เป็น iid mean ศูนย์ variance sε² ใช้ปฏิทินสมมติที่มีห้าวันสม่ำเสมอและไม่มีวันหยุด

ให้ \(\bar m=5^{-1}\sum_dm_d\) และ a_d=m_d−m̄ เมื่อเฉลี่ยจุดเริ่มต้นของสัปดาห์ทั้งห้าแบบเท่ากัน จะได้สำหรับ k≥1

$$
\gamma_0=\frac15\sum_{d=1}^5a_d^2+s_\varepsilon^2,\qquad
\gamma_k=\frac15\sum_{d=1}^5a_da_{d-k},\qquad
\rho_k=\frac{\gamma_k}{\gamma_0}.
$$

ดัชนีวันวนกลับทุกห้าวัน สูตรเป็น ACF ของแบบจำลองที่สุ่ม phase เริ่มต้นอย่างสม่ำเสมอ หรือเป็น pooled covariance ที่เฉลี่ยครบทุก phase สำหรับปฏิทินคงที่ mean เปลี่ยนตามวัน กระบวนการเดิมจึงไม่ weak stationary ตามนิยาม mean คงที่ ต้องระบุการเฉลี่ยนี้ก่อนเรียกผลว่า ACF ทฤษฎี

ตัวอย่างสมมติ m=(−0.4%, 0.1%, 0.1%, 0.1%, 0.1%) และ SD ของ noise 1% ให้ m̄=0, variance ของ mean ตามวันเท่ากับ 0.000004 และ variance รวม 0.000104 จึงได้ ρ₁≈−0.009615 และ ρ₅≈0.038462 แม้ ε แต่ละวันเป็นอิสระ

ถ้าหัก mean ของวันนั้นที่ทราบจริงออก จะเหลือ εₜ ซึ่งมี ACF ศูนย์ทุก lag บวก แต่ในข้อมูลจริงเราต้องประมาณ mean เหล่านี้ การปรับค่าและทดสอบในข้อมูลชุดเดียวกันจึงต้องคำนึงถึง estimation error ด้วย

ปฏิทินอาจเปลี่ยน variance แทน mean ได้เช่นกัน ให้ \(r_t=\sigma_{d(t)}z_t\) โดย z เป็น iid mean ศูนย์ variance 1 จะยังมี ACF ของ r เป็นศูนย์ แต่ mean ของ r² ตามวันเท่ากับ σ_d² ความเป็นคาบนี้จึงสร้าง ACF ใน squared returns แบบ pooled ได้ ควรปรับ mean ตามวันของตัวแปรที่กำลังวิเคราะห์ หรือปรับสเกลด้วย σ_d เมื่อโมเดลรองรับ

In [17]:
calendar=calendar_acf()
close(calendar['variance'],.000104)
close(calendar['acf'][1],-1/104);close(calendar['acf'][5],4/104)
print('Pooled calendar ACF lags 1..15:',calendar['acf'][1:])
assert all(v==0 for v in calendar_acf(0)['acf'][1:])
print('Known weekday means removed: independent residuals have theoretical ACF zero.')

Pooled calendar ACF lags 1..15: [-0.009615384615384614, -0.009615384615384614, -0.009615384615384614, -0.009615384615384614, 0.03846153846153846, -0.009615384615384614, -0.009615384615384614, -0.009615384615384614, -0.009615384615384614, 0.03846153846153846, -0.009615384615384614, -0.009615384615384614, -0.009615384615384614, -0.009615384615384614, 0.03846153846153846]
Known weekday means removed: independent residuals have theoretical ACF zero.


## ภาคผนวก: ACF ของ squared linear process

ให้กระบวนการ mean ศูนย์เป็น \(X_t=\sum_{j\geq0}\psi_j\varepsilon_{t-j}\) โดย innovations เป็น iid mean ศูนย์ variance σ² และมี fourth moment จำกัด กำหนด ψ_j=0 เมื่อ j<0 และสมมติผลรวมลู่เข้าพอให้คำนวณ fourth moments ได้ เช่น \(\sum|\psi_j|<\infty\)

ให้ \(c_4=\mathbb E[\varepsilon_t^4]-3\sigma^4\) เป็น fourth cumulant ของ innovation เมื่อขยายผลคูณ Xₜ²Xₜ₋ₖ² พจน์ที่จับคู่ช็อกคนละเวลาจะให้ส่วนของ covariance กำลังสอง ส่วนที่ช็อกทั้งสี่ตัวอยู่เวลาเดียวกันให้พจน์ c₄

$$
\operatorname{Cov}(X_t^2,X_{t-k}^2)
=2\gamma_k^2+c_4\sum_{j=0}^{\infty}\psi_j^2\psi_{j+k}^2,
$$
$$
\operatorname{Var}(X_t^2)=2\gamma_0^2+c_4\sum_{j=0}^{\infty}\psi_j^4,
\qquad
\gamma_k=\sigma^2\sum_{j=0}^{\infty}\psi_j\psi_{j+k}.
$$

ACF ของ X² คือบรรทัดแรกหารด้วย variance ในบรรทัดที่สอง เมื่อ variance นั้นเป็นบวก สำหรับ Gaussian innovations c₄=0 จึงลดรูปเป็น ρₖ² เช่น Gaussian AR(1) ที่ φ=0.6 มี ρ₁=0.6 แต่ squared-return correlation ที่ lag 1 เท่ากับ 0.36

สำหรับ MA(1) ที่ ψ₀=1, ψ₁=θ และ σ²=1 สูตรให้

$$
\operatorname{Corr}(X_t^2,X_{t-1}^2)
=\frac{(2+c_4)\theta^2}{2(1+\theta^2)^2+c_4(1+\theta^4)}.
$$

ถ้า θ=0.5 และช็อกเป็น Gaussian จะได้ 0.16 แต่ถ้า innovation มี variance 1 และ kurtosis 6 จะมี c₄=3 และได้ประมาณ 0.198020 การยกกำลังสอง ACF ของผลตอบแทนอย่างเดียวจึงใช้แทนสูตรทั่วไปไม่ได้

สูตรทั้งหมดนี้ใช้ X ที่มี mean ศูนย์ หากใช้ผลตอบแทนที่ mean ไม่เป็นศูนย์ ต้อง center ก่อนหรือรวมพจน์ที่เกิดจาก mean เพิ่ม ตัวอย่างและการตรวจด้วยการแจกแจง innovations แบบไม่ต่อเนื่องอยู่ใน Notebook

In [18]:
import itertools
# Innovations: +/-sqrt(6) with probability 1/12 each; zero with probability 5/6.
# Mean 0, variance 1, kurtosis 6, fourth cumulant 3.
points=[(-math.sqrt(6),1/12),(0,5/6),(math.sqrt(6),1/12)]
theta=.5
ex2=ex4=cross=0
for (a,pa),(b,pb),(c,pc) in itertools.product(points,repeat=3):
    weight=pa*pb*pc;x=a+theta*b;previous=b+theta*c
    ex2+=weight*x*x;ex4+=weight*x**4;cross+=weight*x*x*previous*previous
exact=(cross-ex2**2)/(ex4-ex2**2)
formula=squared_linear_correlation([1,theta],3)
close(exact,formula);close(formula,20/101)
close(squared_linear_correlation([1,theta],0),.16)
print('Squared MA(1) correlation: Gaussian .16; non-Gaussian exact=',exact,'formula=',formula)

Squared MA(1) correlation: Gaussian .16; non-Gaussian exact= 0.19801980198019792 formula= 0.19801980198019803


## เลือกสิ่งที่จะตรวจจากคำถามที่ต้องตอบ

| งานที่จะทำ | สิ่งที่ต้องตรวจให้ตรงกับงาน |
|---|---|
| ประมาณผลตอบแทนเฉลี่ยหรือ premium | นิยาม return, benchmark, ความคลาดเคลื่อนของ mean และช่วงข้อมูล |
| พยากรณ์ mean | ACF ของ r, ปฏิทิน, แบบจำลองเชิงเส้น/ไม่เชิงเส้น และผลนอกช่วงประมาณ |
| พยากรณ์ volatility | ACF ของ absolute returns กับ r², การเปลี่ยนสภาวะ และ standardized residuals |
| ประเมิน VaR/ES | การแจกแจงของ standardized shocks, หางทั้งสองด้าน และ backtesting |
| ใช้ข้อมูลระหว่างวัน | ช่วงเวลาเก็บราคา, microstructure noise, overnight และ jumps |

ก่อนรายงานผลจากโมเดล ให้ระบุว่าข้อมูลใดรู้ได้ ณ เวลาพยากรณ์ แล้วเก็บช่วงทดสอบที่ไม่ได้ใช้เลือกพารามิเตอร์ไว้ตรวจผล เปรียบเทียบกับแบบจำลองพื้นฐานบนข้อมูลและระยะเวลาพยากรณ์เดียวกัน

## ตารางเทียบหัวข้อกับสารบัญบท 2–4

ตารางนี้เทียบกับภาพสารบัญที่ให้มาและ [สารบัญฉบับผู้เขียน](https://www.lancaster.ac.uk/people/afasjt/apdvp_contents.pdf#page=3) หัวข้อ 4.14 ที่ถูกตัดขอบในภาพตรวจจากสารบัญฉบับเต็มแล้ว เนื้อหาเรียบเรียงใหม่พร้อมตัวอย่างคำนวณ ครอบคลุมหัวข้อทั้ง 31 ข้อ โดยไม่ได้อ้างว่าเป็นคำแปลทุกหน้าหรือใช้ชุดข้อมูลเดียวกับหนังสือ

| หัวข้อเดิม | อ่านในบทเรียน |
|---|---|
| 2.1 Introduction | [เปิดหัวข้อ](../prices-and-returns.html#introduction) |
| 2.2 Two Examples of Price Series | [เปิดหัวข้อ](../prices-and-returns.html#two-price-series) |
| 2.3 Data-Collection Issues | [เปิดหัวข้อ](../prices-and-returns.html#data-collection) |
| 2.4 Two Returns Series | [เปิดหัวข้อ](../prices-and-returns.html#two-return-series) |
| 2.5 Definitions of Returns | [เปิดหัวข้อ](../prices-and-returns.html#return-definitions) |
| 2.6 Further Examples of Time Series of Returns | [เปิดหัวข้อ](../prices-and-returns.html#other-return-series) |
| 3.1 Introduction | [เปิดหัวข้อ](../stochastic-processes.html#introduction) |
| 3.2 Random Variables | [เปิดหัวข้อ](../stochastic-processes.html#random-variables) |
| 3.3 Stationary Stochastic Processes | [เปิดหัวข้อ](../stochastic-processes.html#stationarity) |
| 3.4 Uncorrelated Processes | [เปิดหัวข้อ](../stochastic-processes.html#uncorrelated-processes) |
| 3.5 ARMA Processes | [เปิดหัวข้อ](../stochastic-processes.html#arma) |
| 3.6 Examples of ARMA(1, 1) Specifications | [เปิดหัวข้อ](../stochastic-processes.html#arma-examples) |
| 3.7 ARIMA Processes | [เปิดหัวข้อ](../stochastic-processes.html#arima) |
| 3.8 ARFIMA Processes | [เปิดหัวข้อ](../stochastic-processes.html#arfima) |
| 3.9 Linear Stochastic Processes | [เปิดหัวข้อ](../stochastic-processes.html#linear-processes) |
| 3.10 Continuous-Time Stochastic Processes | [เปิดหัวข้อ](../stochastic-processes.html#continuous-time) |
| 3.11 Notation for Random Variables and Observations | [เปิดหัวข้อ](../stochastic-processes.html#notation) |
| 4.1 Introduction | [เปิดหัวข้อ](../asset-returns-stylized-facts.html#daily-evidence) |
| 4.2 Summary Statistics | [เปิดหัวข้อ](../asset-returns-stylized-facts.html#summary-statistics) |
| 4.3 Average Returns and Risk Premia | [เปิดหัวข้อ](../asset-returns-stylized-facts.html#average-returns-risk-premia) |
| 4.4 Standard Deviations | [เปิดหัวข้อ](../asset-returns-stylized-facts.html#standard-deviations) |
| 4.5 Calendar Effects | [เปิดหัวข้อ](../asset-returns-stylized-facts.html#calendar-effects) |
| 4.6 Skewness and Kurtosis | [เปิดหัวข้อ](../asset-returns-stylized-facts.html#skewness) · [Kurtosis](#fat-tails) |
| 4.7 The Shape of the Returns Distribution | [เปิดหัวข้อ](../asset-returns-stylized-facts.html#distribution-shape) |
| 4.8 Probability Distributions for Returns | [เปิดหัวข้อ](../asset-returns-stylized-facts.html#return-distributions) |
| 4.9 Autocorrelations of Returns | [เปิดหัวข้อ](../asset-returns-stylized-facts.html#autocorrelation) |
| 4.10 Autocorrelations of Transformed Returns | [เปิดหัวข้อ](../asset-returns-stylized-facts.html#transformed-autocorrelations) |
| 4.11 Nonlinearity of the Returns Process | [เปิดหัวข้อ](../asset-returns-stylized-facts.html#nonlinearity) |
| 4.12 Concluding Remarks | [เปิดหัวข้อ](../asset-returns-stylized-facts.html#model-selection-checks) |
| 4.13 Appendix: Autocorrelation Caused by Day-of-the-Week Effects | [เปิดหัวข้อ](../asset-returns-stylized-facts.html#calendar-acf-appendix) |
| 4.14 Appendix: Autocorrelations of a Squared Linear Process | [เปิดหัวข้อ](../asset-returns-stylized-facts.html#squared-linear-appendix) |

## ทดลองและตรวจคำตอบ

1. ซื้อหุ้นที่ 100 ปลายช่วงราคา 99 และได้รับปันผล 2 คำนวณ simple return กับ log return แล้วอธิบายว่าทำไมใช้ price return อย่างเดียวจึงได้เครื่องหมายต่างกัน
2. ในตัวทดลอง clustering สับลำดับวันแล้วเทียบ mean, SD, kurtosis และ ACF ของ |r| สถิติใดเปลี่ยน และสถิติใดเก็บข้อมูลเกี่ยวกับลำดับเวลาไว้?
3. เลื่อนอัตราส่วน SD ของ mixture เป็น 1 แล้วคำนวณ kurtosis จากสูตร ลองอธิบายว่าถ้าสุ่มเลือกสภาวะใหม่ทุกวันอย่างอิสระ หางหนายังอยู่ได้โดยไม่มี clustering อย่างไร
4. คำนวณ RV และ √RV จาก log returns 1%, −1%, 1%, −1% แล้วเทียบกับการเก็บเฉพาะราคาต้น–ปลายวัน
5. เมื่อ noise ของ log price เป็น ±3 bps อย่างอิสระ จงหาส่วนเพิ่มของ E[RV] ถ้าใช้ 390 ผลตอบแทน เทียบกับ 78 ผลตอบแทน โดยไม่เปลี่ยนระยะเวลาของวัน

**เปิดแนวคำตอบ**

ข้อ 1 simple return เท่ากับ 1% และ log return เท่ากับ \(\log(1.01)\approx0.9950\%\) ส่วนราคาดิบลด 1% เพราะยังไม่ได้รวมปันผล

ข้อ 2 mean, SD, kurtosis และ histogram เท่าเดิมทุกค่า การสับลำดับเปลี่ยนคู่ข้อมูลที่ใช้หา ACF จึงเปลี่ยนสถิติที่วัดความสัมพันธ์ข้ามเวลา

ข้อ 3 เมื่อ SD เท่ากัน variance ของ \(\sigma^2\) เป็นศูนย์ จึงได้ κ=3 ส่วนการสุ่มสภาวะแบบอิสระยังสร้าง mixture marginal ได้ แต่ไม่ได้สร้าง persistence ของสภาวะ

ข้อ 4 RV=0.0004 และ √RV=2% ส่วนผลตอบแทนต้น–ปลายวันเป็นศูนย์

ข้อ 5 \(\eta=0.0003\) ให้ \(2(390)\eta^2=0.0000702\) กับ \(2(78)\eta^2=0.00001404\) ในหน่วยทศนิยม² ต่างกันห้าเท่า ตัวเลขนี้เป็นส่วนเพิ่มของค่าคาดหมายภายใต้โมเดล noise ไม่ใช่ความต่างที่ต้องเกิดพอดีทุกเส้นทาง

[ดาวน์โหลด Python Notebook](asset-returns-stylized-facts.ipynb) เพื่อคำนวณ returns, ACF, Box–Pierce/Ljung–Box, mixture kurtosis, intraday profile และ realized variance มีราคาปิด S&P 500 ชุดเดียวกับกราฟและภาพประกอบฝังไว้ในไฟล์ ตัวอย่างจำลองกับการสับลำดับใช้ seed และวิธีคำนวณเดียวกับตัวทดลอง ใช้ Python standard library ได้โดยไม่ต้องดาวน์โหลดข้อมูลเพิ่ม

## อ่านเพิ่มเติม

- [arch 8.0.0: ชุดข้อมูล S&P 500](https://github.com/bashtage/arch/tree/v8.0.0/arch/data/sp500) จาก Yahoo Finance ใช้ราคาปิดช่วง 4 มกราคม 1999 ถึง 31 ธันวาคม 2018 · [รายละเอียดข้อมูลที่ใช้ในบท](../data/sp500-daily.json)
- John P. Nolan, [*Stable Distributions*, บทนำ](https://edspace.american.edu/jpnolan/wp-content/uploads/sites/1720/2020/09/Chap1.pdf), เงื่อนไขของโมเมนต์ใน stable distributions
- NIST, [*Measures of Skewness and Kurtosis*](https://www.itl.nist.gov/div898/handbook/eda/section3/eda35b.htm) และ [*t Distribution*](https://www.itl.nist.gov/div898/handbook/eda/section3/eda3664.htm)
- Stephen J. Taylor, *Asset Price Dynamics, Volatility, and Prediction* (2005), บท 2, 4 และ 12 · [บทนำจาก Princeton University Press](https://assets.press.princeton.edu/chapters/i8055.pdf)
- Benoit Mandelbrot, [*The Variation of Certain Speculative Prices* (1963)](https://oftp.cyrax.hu/doc/mandelbrot.pdf), โดยเฉพาะข้อสังเกตเรื่องการเกิดกลุ่มของความผันผวนในหน้า 418
- NIST, [*Autocorrelation Plot*](https://www.itl.nist.gov/div898/handbook/eda/section3/eda331.htm), นิยาม sample ACF และกรอบอ้างอิง
- Andersen, Bollerslev, Diebold และ Labys, [*Modeling and Forecasting Realized Volatility* (2003)](https://econ.duke.edu/~boller/Published_Papers/ecta_03.pdf)
- Aït-Sahalia, Mykland และ Zhang, [*How Often to Sample a Continuous-Time Process in the Presence of Market Microstructure Noise*](https://www.nber.org/papers/w9611), working paper 2003, ตีพิมพ์ในปี 2005
- CFTC และ SEC, [*Findings Regarding the Market Events of May 6, 2010*](https://www.sec.gov/news/studies/2010/marketevents-report.pdf), รายงานวันที่ 30 กันยายน 2010